# 농구 영상 탐지랑 이벤트 스탯 정리



### Core 셀 나눈 이유

밑에서 쓰는 함수들이 너무 길어서 Core로 따로 빼둔 부분.
한셀 로 넣기에 너무 많아 분리.

- Core 1: 코트 밖에 잡힌 오탐 줄이기
- Core 2: 공 위치 비는 부분 이어붙이기
- Core 3: 탐지 결과 한 번에 후처리
- Core 4: 공 좌표가 튀는 부분 다시 정리
- Core 5: 슛/패스 같은 이벤트 대략 계산
- Core 6: YOLO 돌리고 CSV 만드는 흐름



## Core 1. 코트 밖 오탐 처리

관중석, 벤치, 광고판 쪽에서 잡히는 박스를 줄이려고 넣은 부분.
코트 polygon 안쪽인지 보고, 사람은 발 위치 기준 / 공은 중심 기준으로 판단함.


In [1]:
# - 먼저 코트 밖 오탐을 줄여야 뒤쪽 공/이벤트 계산이 덜 흔들림.
# - 사람은 발 위치, 공은 중심 위치를 기준으로 코트 안쪽인지 봄.
# - 공은 공중에 떠 있을 수 있어서 사람보다 margin을 넓게 둠.
# - polygon 좌표는 정규화 좌표라 영상 크기가 바뀌어도 어느 정도 맞음.
# - 영상 구도가 바뀌면 DEFAULT_BROADCAST_COURT_POLYGON_NORM부터 조정.

# 코트 밖에 잡힌 탐지 결과를 걸러내는 부분.
# 관중석, 벤치, 광고판 쪽 오탐이 생각보다 많이 섞여서 코트 안쪽 객체만 남기도록 해둠.


from __future__ import annotations

import json
import math
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple


Point = Tuple[float, float]
Detection = Dict[str, Any]


# 여기서는 검출된 객체가 코트 안쪽에 있는지만 먼저 본다.
# 코트 밖 객체가 CSV에 섞이면 선수나 공 위치 계산이 흔들려서,
# 정규화된 다각형으로 코트 영역을 잡고, 클래스별 기준점에 맞춰 필터링함.
# 코트 모양이 영상마다 다르면 DEFAULT_BROADCAST_COURT_POLYGON_NORM 값을 조정함.


# 기본 코트 polygon이다. 좌표는 (x / frame_width, y / frame_height) 형태의 정규화 좌표다.
# 실제 영상 카메라 구도에 따라 이 값을 직접 조정하면 필터 정확도가 높아진다.
DEFAULT_BROADCAST_COURT_POLYGON_NORM: List[Point] = [
    (0.05, 0.30),
    (0.95, 0.30),
    (1.00, 0.98),
    (0.00, 0.98),
]

DEFAULT_ANCHOR_BY_CLASS = {
    # 사람은 중심점보다 발 위치가 코트 안/밖 판단에 더 적합하다.
    "person": "bottom_center",
    "player": "bottom_center",
    "referee": "bottom_center",
    "sports ball": "center",
    "frisbee": "center",
    "basketball": "center",
    "ball": "center",
    # 골대는 코트 가장자리에 있으므로 중심점을 기준으로 남긴다.
    "hoop": "center",
    "rim": "center",
    "basketball hoop": "center",
}

DEFAULT_MARGIN_BY_CLASS = {
    # 공은 공중 이동 중 코트 바닥보다 훨씬 위에 떠 있을 수 있어 더 넓게 허용함.
    "ball": 260.0,
    "sports ball": 260.0,
    "basketball": 260.0,
    "frisbee": 260.0,
}


# 코트 안쪽에 있는 탐지만 남기려고 만든 필터 클래스.
# 코트 polygon 기준으로 row를 걸러내는 필터 클래스.
# 설정값이나 상태를 먼저 묶어두고, 아래 메서드들이 그 값을 같이 사용함.
class CourtObjectFilter:
    # 농구 코트 polygon을 기준으로 YOLO 탐지 row를 필터링함.

    # init__ 처리용 보조 함수.
    # court_polygon, frame_size, polygon_is_normalized 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def __init__(
        self,
        court_polygon: Sequence[Any],
        frame_size: Optional[Tuple[int, int]] = None,
        polygon_is_normalized: bool = True,
        margin_px: float = 8.0,
        margin_by_class: Optional[Mapping[str, float]] = None,
        anchor_by_class: Optional[Mapping[str, str]] = None,
        default_anchor: str = "center",
    ) -> None:
        # 클래스마다 코트 안/밖을 판단하는 기준점이 다름.
        self.frame_size = frame_size
        self.polygon_is_normalized = polygon_is_normalized
        self.margin_px = float(margin_px)
        self.margin_by_class = dict(DEFAULT_MARGIN_BY_CLASS)
        if margin_by_class:
            self.margin_by_class.update(
                {str(key): float(value) for key, value in margin_by_class.items()}
            )
        self.anchor_by_class = dict(DEFAULT_ANCHOR_BY_CLASS)
        if anchor_by_class:
            self.anchor_by_class.update(anchor_by_class)
        self.default_anchor = default_anchor
        self.court_polygon = normalize_polygon(
            court_polygon,
            frame_size=frame_size,
            polygon_is_normalized=polygon_is_normalized,
        )

    # keep_detection 처리용 보조 함수.
    # detection, class_key, x_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def keep_detection(
        self,
        detection: Mapping[str, Any],
        class_key: str = "class",
        x_key: str = "x_center",
        y_key: str = "y_center",
    ) -> bool:
        class_name = str(detection.get(class_key))
        anchor = self.anchor_by_class.get(class_name, self.default_anchor)
        point = detection_anchor_point(detection, anchor=anchor, x_key=x_key, y_key=y_key)
        if point is None:
            return False
        margin = self.margin_by_class.get(class_name, self.margin_px)
        return point_in_polygon(point, self.court_polygon, margin_px=margin)

    # 조건에 맞지 않는 row를 걸러내는 함수.
    # 기준값을 계산하고, 조건에 맞지 않는 row를 mask/거리 기준으로 제거함.
    def filter_records(
        self,
        detections: Iterable[Mapping[str, Any]],
        class_key: str = "class",
        x_key: str = "x_center",
        y_key: str = "y_center",
    ) -> List[Detection]:
        # 여러 탐지 row 중 코트 안에 있는 row만 남긴다.
        kept: List[Detection] = []
        for detection in detections:
            if self.keep_detection(detection, class_key=class_key, x_key=x_key, y_key=y_key):
                data_row = dict(detection)
                data_row["inside_court"] = True
                kept.append(data_row)
        return kept


# DataFrame에서 코트 밖 탐지를 빼고 다시 정렬.
def filter_detections_by_court(
    detection_data: Any,
    court_polygon: Optional[Sequence[Any]] = None,
    frame_size: Optional[Tuple[int, int]] = None,
    video_path: Optional[str] = None,
    polygon_is_normalized: bool = True,
    margin_px: float = 8.0,
    margin_by_class: Optional[Mapping[str, float]] = None,
    anchor_by_class: Optional[Mapping[str, str]] = None,
    default_anchor: str = "center",
    class_key: str = "class",
    x_key: str = "x_center",
    y_key: str = "y_center",
) -> Any:

    import pandas as pd

    # 다각형 값을 따로 넘기지 않으면 기본 중계 화면용 코트 영역을 쓴다.
    polygon = DEFAULT_BROADCAST_COURT_POLYGON_NORM if court_polygon is None else court_polygon
    size = resolve_frame_size(frame_size=frame_size, video_path=video_path)

    court_filter = CourtObjectFilter(
        polygon,
        frame_size=size,
        polygon_is_normalized=polygon_is_normalized,
        margin_px=margin_px,
        margin_by_class=margin_by_class,
        anchor_by_class=anchor_by_class,
        default_anchor=default_anchor,
    )

    kept_records = court_filter.filter_records(
        detection_data.to_dict("records"),
        class_key=class_key,
        x_key=x_key,
        y_key=y_key,
    )

    filtered_data = pd.DataFrame(kept_records)
    if filtered_data.empty:
        return detection_data.iloc[0:0].copy()

    sort_columns = [
        column
        for column in ["frame", class_key, "track_id"]
        if column in filtered_data.columns
    ]
    if sort_columns:
        filtered_data = filtered_data.sort_values(sort_columns).reset_index(drop=True)
    return filtered_data


# 이미 만든 탐지 CSV만 다시 코트 기준으로 정리할 때 사용.
# CSV 파일만 따로 읽어서 코트 필터를 적용할 때 사용.
# 입력 row를 정리한 뒤 필요한 컬럼 순서로 맞춰서 CSV/표 형태로 넘김.
def filter_detection_csv(
    input_csv: str = "video_detection.csv",
    output_csv: str = "video_detection_court_filtered.csv",
    court_polygon: Optional[Sequence[Any]] = None,
    court_config_path: Optional[str] = None,
    frame_size: Optional[Tuple[int, int]] = None,
    video_path: Optional[str] = "Video Project.mp4",
    polygon_is_normalized: bool = True,
    margin_px: float = 8.0,
    margin_by_class: Optional[Mapping[str, float]] = None,
    **kwargs: Any,
) -> Any:

    import pandas as pd

    polygon = court_polygon
    config_normalized = polygon_is_normalized
    config_frame_size = frame_size
    config_margin = margin_px

    if court_config_path:
        config = load_court_config(court_config_path)
        polygon = config["polygon"]
        config_normalized = bool(config.get("normalized", polygon_is_normalized))
        config_frame_size = _tuple_frame_size(config.get("frame_size")) or frame_size
        config_margin = float(config.get("margin_px", margin_px))

    detection_data = pd.read_csv(input_csv)
    filtered_data = filter_detections_by_court(
        detection_data,
        court_polygon=polygon,
        frame_size=config_frame_size,
        video_path=video_path,
        polygon_is_normalized=config_normalized,
        margin_px=config_margin,
        margin_by_class=margin_by_class,
        **kwargs,
    )
    filtered_data.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return filtered_data


# 코트 polygon 설정 JSON을 읽는 부분.
# path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def load_court_config(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as file:
        config = json.load(file)

    if isinstance(config, list):
        return {"polygon": config, "normalized": True}
    if not isinstance(config, dict) or "polygon" not in config:
        raise ValueError("Core1 polygon 설정")
    return config


# 영상 크기를 직접 입력했는지, 파일에서 읽을지 정하는 부분.
# 영상 크기를 직접 받은 값 또는 파일 정보에서 결정.
# frame_size, video_path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def resolve_frame_size(
    frame_size: Optional[Tuple[int, int]] = None,
    video_path: Optional[str] = None,
) -> Optional[Tuple[int, int]]:
    if frame_size is not None:
        return _tuple_frame_size(frame_size)
    if video_path:
        return read_video_frame_size(video_path)
    return None


# cv2로 영상 width, height만 가져옴.
# video_path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def read_video_frame_size(video_path: str) -> Optional[Tuple[int, int]]:
    try:
        import cv2
    except ImportError:
        return None

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        return None

    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()
    if width <= 0 or height <= 0:
        return None
    return width, height


# 정규화 좌표를 실제 픽셀 좌표로 바꿈.
# 값을 같은 형식으로 맞춘 뒤 뒤쪽 계산에서 비교하기 쉽게 넘김.
def normalize_polygon(
    polygon: Sequence[Any],
    frame_size: Optional[Tuple[int, int]],
    polygon_is_normalized: bool,
) -> List[Point]:
    points = [_parse_point(point) for point in polygon]
    if len(points) < 3:
        raise ValueError("Core1 코트 점 개수")

    if not polygon_is_normalized:
        return points

    if frame_size is None:
        raise ValueError("Core1 frame_size 확인")

    width, height = frame_size
    return [(x * width, y * height) for x, y in points]


# 클래스별 기준점(center/bottom)을 잡는 함수.
# 객체별로 중심점/발 위치 중 어떤 점을 쓸지 정함.
# detection, anchor, x_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def detection_anchor_point(
    detection: Mapping[str, Any],
    anchor: str,
    x_key: str = "x_center",
    y_key: str = "y_center",
) -> Optional[Point]:
    x_center = to_safe_float(detection.get(x_key))
    y_center = to_safe_float(detection.get(y_key))
    if x_center is None or y_center is None:
        return None

    x1 = to_safe_float(detection.get("x1"))
    y1 = to_safe_float(detection.get("y1"))
    x2 = to_safe_float(detection.get("x2"))
    y2 = to_safe_float(detection.get("y2"))

    if anchor == "center":
        return x_center, y_center
    if anchor == "bottom_center" and y2 is not None:
        return x_center, y2
    if anchor == "top_center" and y1 is not None:
        return x_center, y1
    if anchor == "bbox_center" and None not in (x1, y1, x2, y2):
        return (float(x1) + float(x2)) / 2.0, (float(y1) + float(y2)) / 2.0
    return x_center, y_center


# 점이 코트 polygon 안쪽인지 확인.
# 점이 polygon 안쪽인지 ray 방식으로 확인.
# point, polygon, margin_px 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def point_in_polygon(
    point: Point,
    polygon: Sequence[Point],
    margin_px: float = 0.0,
) -> bool:
    if margin_px > 0.0 and _distance_to_polygon(point, polygon) <= margin_px:
        return True

    x, y = point
    inside = False
    count = len(polygon)
    # index 항목을 돌면서 조건에 맞는 값만 남김.
    for index in range(count):
        x1, y1 = polygon[index]
        x2, y2 = polygon[(index + 1) % count]

        if _point_on_segment(point, (x1, y1), (x2, y2)):
            return True

        intersects = (y1 > y) != (y2 > y)
        if intersects:
            x_intersection = (x2 - x1) * (y - y1) / (y2 - y1) + x1
            if x < x_intersection:
                inside = not inside
    return inside


# _distance_to_polygon는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _distance_to_polygon(point: Point, polygon: Sequence[Point]) -> float:
    return min(
        _distance_to_segment(point, polygon[index], polygon[(index + 1) % len(polygon)])
        for index in range(len(polygon))
    )


# _distance_to_segment는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _distance_to_segment(
    point: Point,
    segment_start: Point,
    segment_end: Point,
) -> float:
    px, py = point
    x1, y1 = segment_start
    x2, y2 = segment_end
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0.0 and dy == 0.0:
        return math.hypot(px - x1, py - y1)

    t = ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)
    t = max(0.0, min(1.0, t))
    nearest_x = x1 + t * dx
    nearest_y = y1 + t * dy
    return math.hypot(px - nearest_x, py - nearest_y)


# _point_on_segment는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _point_on_segment(
    point: Point,
    segment_start: Point,
    segment_end: Point,
    eps: float = 1e-7,
) -> bool:
    return _distance_to_segment(point, segment_start, segment_end) <= eps


# _parse_point는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _parse_point(point: Any) -> Point:
    if isinstance(point, Mapping):
        return float(point["x"]), float(point["y"])
    if isinstance(point, Sequence) and len(point) >= 2:
        return float(point[0]), float(point[1])
    raise ValueError("Core1 polygon point 확인")


# _tuple_frame_size에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _tuple_frame_size(value: Any) -> Optional[Tuple[int, int]]:
    if value is None:
        return None
    if isinstance(value, Sequence) and len(value) >= 2:
        return int(value[0]), int(value[1])
    raise ValueError("Core1 frame_size 형식 확인")


# 값 형식이 흔들리지 않게 정리하는 함수.
def to_safe_float(value: Any) -> Optional[float]:
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(number) or math.isinf(number):
        return None
    return number


__all__ = [
    "CourtObjectFilter",
    "DEFAULT_BROADCAST_COURT_POLYGON_NORM",
    "DEFAULT_MARGIN_BY_CLASS",
    "filter_detections_by_court",
    "filter_detection_csv",
    "load_court_config",
    "point_in_polygon",
]




## Core 2. 공 위치 이어붙이기

공은 작아서 한 프레임에서 여러 개 잡히거나 아예 안 잡히는 경우가 많았다.
그래서 후보를 고르고, 짧게 비는 구간만 보간한 뒤 Kalman 필터로 조금 부드럽게 만듦.


In [2]:


# - 한 프레임에 공 후보가 여러 개면 confidence와 이전 이동 방향을 같이 봄.
# - 짧게 비는 프레임은 보간하고, 긴 공백은 억지로 이어붙이지 않음.
# - Kalman은 원본 탐지를 완전히 대체하는 게 아니라 좌표 흔들림을 줄이는 용도.
# - rim 근처는 공이 가려질 수 있어서 gap 판단을 조금 다르게 둠.
# - segment가 바뀌면 이전 속도를 버리고 새 공 위치에서 다시 시작.
# - occlusion 정보는 선수에게 가려졌을 가능성을 표시하려고 남김.

from __future__ import annotations

from dataclasses import dataclass, field
import math
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np


Point = Dict[str, Any]


# 공은 한 프레임에 여러 후보가 잡히는 경우가 많아서, 실제 공 궤적에 가까운 후보를 고른다.
# 1) 후보 선택, 2) 짧은 누락 구간 보간, 3) 칼만 필터 기반 평활화, 4) 선수 가림(occlusion) 상황 표시.
# 아래 실행 셀에서는 이 함수들을 거쳐 공 좌표를 한 줄짜리 timeline으로 정리.


@dataclass
# 공 좌표를 Kalman 방식으로 부드럽게 이어주는 클래스.
# 공 위치를 예측하고 관측값으로 다시 보정하는 Kalman tracker.
# 설정값이나 상태를 먼저 묶어두고, 아래 메서드들이 그 값을 같이 사용함.
class BallTracker:

    dt: float = 1.0
    process_noise: float = 90.0
    measurement_noise: float = 7.0
    gravity_px_per_frame2: float = 0.0
    max_missing: int = 12
    state: Optional[np.ndarray] = field(init=False, default=None)
    covariance: Optional[np.ndarray] = field(init=False, default=None)
    missing_frames: int = field(init=False, default=0)

    @property
    # initialized 처리용 보조 함수.
    # 입력값을 정리해서 바로 다음 계산에 넘기는 흐름.
    def initialized(self) -> bool:
        return self.state is not None and self.covariance is not None

    # reset 처리용 보조 함수.
    def reset(self) -> None:
        self.state = None
        self.covariance = None
        self.missing_frames = 0

    # _transition는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
    def _transition(self, dt: Optional[float] = None) -> np.ndarray:
        step = self.dt if dt is None else float(dt)
        return np.array(
            [
                [1.0, 0.0, step, 0.0],
                [0.0, 1.0, 0.0, step],
                [0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 1.0],
            ],
            dtype=float,
        )

    # _gravity_control는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
    def _gravity_control(self, dt: Optional[float] = None) -> np.ndarray:
        step = self.dt if dt is None else float(dt)
        return np.array(
            [
                0.0,
                0.5 * self.gravity_px_per_frame2 * step * step,
                0.0,
                self.gravity_px_per_frame2 * step,
            ],
            dtype=float,
        )

    # _process_noise는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
    def _process_noise(self, dt: Optional[float] = None) -> np.ndarray:
        step = self.dt if dt is None else float(dt)
        dt2 = step * step
        dt3 = dt2 * step
        dt4 = dt2 * dt2
        q = float(self.process_noise)
        return q * np.array(
            [
                [dt4 / 4.0, 0.0, dt3 / 2.0, 0.0],
                [0.0, dt4 / 4.0, 0.0, dt3 / 2.0],
                [dt3 / 2.0, 0.0, dt2, 0.0],
                [0.0, dt3 / 2.0, 0.0, dt2],
            ],
            dtype=float,
        )

    def _initialize(self, x: float, y: float) -> None:
        self.state = np.array([x, y, 0.0, 0.0], dtype=float)
        self.covariance = np.diag([400.0, 400.0, 250.0, 250.0]).astype(float)
        self.missing_frames = 0

    # predict 처리용 보조 함수.
    # dt 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def predict(self, dt: Optional[float] = None) -> Optional[Point]:
        if not self.initialized:
            return None

        assert self.state is not None
        assert self.covariance is not None

        transition = self._transition(dt)
        self.state = transition @ self.state + self._gravity_control(dt)
        self.covariance = transition @ self.covariance @ transition.T + self._process_noise(dt)
        self.missing_frames += 1

        if self.missing_frames > self.max_missing:
            self.reset()
            return None

        return self.as_dict(source="kalman_prediction")

    # update 처리용 보조 함수.
    # x, y, confidence 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def update(
        self,
        x: float,
        y: float,
        confidence: Optional[float] = None,
        measurement_noise: Optional[float] = None,
    ) -> Point:
        # YOLO/보간 측정값을 Kalman 상태에 반영함.
        if not self.initialized:
            self._initialize(float(x), float(y))
            return self.as_dict(source="kalman_update")

        assert self.state is not None
        assert self.covariance is not None

        measurement = np.array([float(x), float(y)], dtype=float)
        observation = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]], dtype=float)

        conf = 1.0 if confidence is None else max(0.05, min(1.0, float(confidence)))
        noise = self.measurement_noise if measurement_noise is None else float(measurement_noise)
        measurement_variance = (noise**2) / conf
        measurement_covariance = np.eye(2, dtype=float) * measurement_variance

        innovation = measurement - observation @ self.state
        innovation_covariance = (
            observation @ self.covariance @ observation.T + measurement_covariance
        )
        # 칼만 이득은 예측값과 측정값 중 어느 쪽을 더 믿을지 결정함.
        kalman_gain = self.covariance @ observation.T @ np.linalg.inv(innovation_covariance)

        self.state = self.state + kalman_gain @ innovation
        identity = np.eye(4, dtype=float)
        self.covariance = (identity - kalman_gain @ observation) @ self.covariance
        self.missing_frames = 0
        return self.as_dict(source="kalman_update")

    # step 처리용 보조 함수.
    # measurement, confidence, dt 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def step(
        self,
        measurement: Optional[Tuple[float, float]],
        confidence: Optional[float] = None,
        dt: Optional[float] = None,
        measurement_noise: Optional[float] = None,
    ) -> Optional[Point]:
        if measurement is None:
            return self.predict(dt)

        if self.initialized:
            prediction = self.predict(dt)
            if prediction is None:
                self._initialize(float(measurement[0]), float(measurement[1]))
                return self.as_dict(source="kalman_update")

        return self.update(
            float(measurement[0]),
            float(measurement[1]),
            confidence=confidence,
            measurement_noise=measurement_noise,
        )

    # as_dict 처리용 보조 함수.
    # source 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def as_dict(self, source: str) -> Point:
        if self.state is None:
            raise RuntimeError("BallTracker is not initialized.")
        return {
            "x_center": float(self.state[0]),
            "y_center": float(self.state[1]),
            "velocity_x": float(self.state[2]),
            "velocity_y": float(self.state[3]),
            "source": source,
            "missing_frames": self.missing_frames,
        }


# 짧게 비는 공 좌표를 앞뒤 프레임 기준으로 채움.
# 공 후보를 모은 뒤 confidence, 위치, 이전 이동 방향을 같이 보고 남길 후보를 고름.
def interpolate_ball_path(
    ball_points: Sequence[Mapping[str, Any]],
    max_gap: int = 18,
    method: str = "auto",
    gravity_px_per_frame2: Optional[float] = None,
    auto_gravity: bool = True,
    fallback_gravity_px_per_frame2: float = 0.8,
    min_gravity_px_per_frame2: float = 0.05,
    max_gravity_px_per_frame2: float = 3.0,
    support_window: int = 4,
    interpolated_confidence_floor: float = 0.75,
    total_frames: Optional[int] = None,
    frame_key: str = "frame",
    x_key: str = "x_center",
    y_key: str = "y_center",
    confidence_key: str = "confidence",
    rim_points_by_frame: Optional[Mapping[int, Sequence[Mapping[str, Any]]]] = None,
    rim_proximity_px: float = 180.0,
    rim_interpolation_extra_gap: int = 12,
) -> List[Point]:

    # 짧게 사라진 공 좌표 구간만 보간함.
    # 긴 구간까지 억지로 이어 붙이면 슛/패스 분석에서 잘못된 공 위치가 생길 수 있음.
    if method not in {"auto", "linear", "quadratic"}:
        raise ValueError("Core2 보간 방식 확인")

    # 한 프레임에 공 후보가 여러 개 있으면 우선 confidence가 가장 높은 후보만 남긴다.
    best_by_frame = _select_best_point_by_frame(ball_points, frame_key, confidence_key)
    if not best_by_frame:
        return []

    max_seen_frame = max(best_by_frame)
    frame_count = max_seen_frame + 1 if total_frames is None else int(total_frames)
    timeline = [
        _make_timeline_row(frame, best_by_frame.get(frame), x_key, y_key, confidence_key)
        for frame in range(frame_count)
    ]

    valid_frames = [data_row["frame"] for data_row in timeline if _has_xy(data_row, x_key, y_key)]
    if len(valid_frames) < 2:
        return timeline

    global_gravity = _resolve_gravity(
        [timeline[frame] for frame in valid_frames],
        requested_gravity=gravity_px_per_frame2,
        auto_gravity=auto_gravity,
        fallback_gravity=fallback_gravity_px_per_frame2,
        min_gravity=min_gravity_px_per_frame2,
        max_gravity=max_gravity_px_per_frame2,
        frame_key=frame_key,
        y_key=y_key,
    )

    # 프레임을 돌면서 탐지 결과와 누적값을 계속 갱신함.
    for left_frame, right_frame in zip(valid_frames, valid_frames[1:]):
        gap = right_frame - left_frame - 1
        if gap <= 0:
            continue

        left = timeline[left_frame]
        right = timeline[right_frame]
        # 공을 다시 잡은 지점이 이전 경로와 다른 segment이면, 두 점 사이를 억지로 잇지 않는다.
        # 이렇게 해야 엉뚱한 후보로 재시작했을 때 큰 직선/포물선 점프가 만들어지지 않는다.
        left_segment = left.get("ball_segment")
        right_segment = right.get("ball_segment")
        if left_segment is not None and right_segment is not None and left_segment != right_segment:
            continue
        near_rim_gap = _gap_near_rim(
            left,
            right,
            rim_points_by_frame,
            frame_key=frame_key,
            x_key=x_key,
            y_key=y_key,
            proximity_px=rim_proximity_px,
        )
        allowed_gap = (
            max_gap + max(0, int(rim_interpolation_extra_gap))
            if near_rim_gap
            else max_gap
        )
        # 림 근처는 공이 가려지는 경우가 많아서 조금 더 긴 gap까지 보간을 허용함.
        if gap > allowed_gap:
            continue
        previous_point = _nearest_valid_before(timeline, left_frame, x_key, y_key)
        next_point = _nearest_valid_after(timeline, right_frame, x_key, y_key)
        support_points = _support_points_around_gap(
            timeline,
            left_frame,
            right_frame,
            support_window,
            x_key,
            y_key,
        )
        gap_gravity = _resolve_gravity(
            support_points,
            requested_gravity=gravity_px_per_frame2,
            auto_gravity=auto_gravity,
            fallback_gravity=global_gravity,
            min_gravity=min_gravity_px_per_frame2,
            max_gravity=max_gravity_px_per_frame2,
            frame_key=frame_key,
            y_key=y_key,
        )

        # 프레임을 하나씩 보면서 탐지 결과와 시간 기준값을 같이 갱신.
        for frame in range(left_frame + 1, right_frame):
            interpolated_x, interpolated_y = _interpolate_between(
                frame,
                left,
                right,
                previous_point,
                next_point,
                method,
                gap_gravity,
                x_key,
                y_key,
            )
            # 보간점은 실제 탐지보다 약하지만, 궤적을 유지해야 하므로 최소 신뢰도를 준다.
            confidence = _interpolated_confidence(
                left,
                right,
                confidence_key,
                floor=interpolated_confidence_floor,
            )
            timeline[frame].update(
                {
                    x_key: interpolated_x,
                    y_key: interpolated_y,
                    confidence_key: confidence,
                    "detected": False,
                    "interpolated": True,
                    "measurement_source": "rim_interpolated" if near_rim_gap else "interpolated",
                    "interpolation_method": "linear" if method == "linear" else "quadratic",
                    "estimated_gravity": round(gap_gravity, 4),
                    "near_rim_gap": near_rim_gap,
                }
            )

    return timeline


# 여러 공 후보에서 하나의 공 궤적을 만드는 중심 함수.
def track_ball_detections(
    detections: Iterable[Mapping[str, Any]],
    total_frames: Optional[int] = None,
    max_gap: int = 18,
    interpolation: str = "auto",
    dt: float = 1.0,
    process_noise: float = 90.0,
    measurement_noise: float = 7.0,
    gravity_px_per_frame2: Optional[float] = None,
    auto_gravity: bool = True,
    fallback_gravity_px_per_frame2: float = 0.8,
    min_gravity_px_per_frame2: float = 0.05,
    max_gravity_px_per_frame2: float = 3.0,
    support_window: int = 4,
    trust_interpolated_arc: bool = True,
    interpolated_measurement_noise: float = 8.0,
    interpolated_confidence_floor: float = 0.45,
    max_prediction_frames: int = 12,
    enable_player_occlusion: bool = True,
    player_classes: Sequence[str] = ("person",),
    occlusion_margin_px: float = 35.0,
    player_center_radius_px: float = 90.0,
    enable_rim_proximity_boost: bool = False,
    rim_classes: Sequence[str] = ("hoop", "rim", "basketball hoop"),
    rim_proximity_px: float = 180.0,
    rim_interpolation_extra_gap: int = 0,
    use_motion_association: bool = True,
    association_max_distance_px: float = 280.0,
    association_gap_growth_px: float = 55.0,
    association_confidence_weight: float = 18.0,
    association_restart_after_frames: int = 7,
    association_restart_min_confidence: float = 0.004,
    ball_classes: Sequence[str] = ("sports ball", "frisbee", "basketball", "ball"),
    ball_label: str = "sports ball",
    stable_track_id: int = 0,
    class_key: str = "class",
    frame_key: str = "frame",
    x_key: str = "x_center",
    y_key: str = "y_center",
    confidence_key: str = "confidence",
) -> List[Point]:

    # 전체 탐지 결과에서 공 후보만 따로 추려 하나의 공 track으로 정리.
    # 선수 row는 occlusion 판단에만 사용하고, 최종 선수 track 자체는 변경하지 않습니다.
    detection_list = list(detections)
    ball_class_set = set(ball_classes)
    ball_points = [
        item
        for item in detection_list
        if item.get(class_key) in ball_class_set
        and to_safe_float(item.get(x_key)) is not None
        and to_safe_float(item.get(y_key)) is not None
    ]
    # 선수 occlusion 판정을 빠르게 하기 위해 선수 탐지를 프레임별로 묶는다.
    players_by_frame = _group_detections_by_frame(
        detection_list,
        frame_key=frame_key,
        class_key=class_key,
        classes=player_classes,
    )
    rims_by_frame = _group_detections_by_frame(
        detection_list,
        frame_key=frame_key,
        class_key=class_key,
        classes=rim_classes,
    )

    if total_frames is None:
        frames = [
            to_safe_int(item.get(frame_key))
            for item in detection_list
            if to_safe_int(item.get(frame_key)) is not None
        ]
        total_frames = max(frames) + 1 if frames else None

    if use_motion_association:
        ball_points = _select_ball_points_by_motion(
            ball_points,
            total_frames=total_frames,
            frame_key=frame_key,
            x_key=x_key,
            y_key=y_key,
            confidence_key=confidence_key,
            max_distance_px=association_max_distance_px,
            gap_growth_px=association_gap_growth_px,
            confidence_weight=association_confidence_weight,
            restart_after_frames=association_restart_after_frames,
            restart_min_confidence=association_restart_min_confidence,
        )

    tracker_gravity = _resolve_gravity(
        ball_points,
        requested_gravity=gravity_px_per_frame2,
        auto_gravity=auto_gravity,
        fallback_gravity=fallback_gravity_px_per_frame2,
        min_gravity=min_gravity_px_per_frame2,
        max_gravity=max_gravity_px_per_frame2,
        frame_key=frame_key,
        y_key=y_key,
    )

    timeline = interpolate_ball_path(
        ball_points,
        max_gap=max_gap,
        method=interpolation,
        gravity_px_per_frame2=gravity_px_per_frame2,
        auto_gravity=auto_gravity,
        fallback_gravity_px_per_frame2=fallback_gravity_px_per_frame2,
        min_gravity_px_per_frame2=min_gravity_px_per_frame2,
        max_gravity_px_per_frame2=max_gravity_px_per_frame2,
        support_window=support_window,
        interpolated_confidence_floor=interpolated_confidence_floor,
        total_frames=total_frames,
        frame_key=frame_key,
        x_key=x_key,
        y_key=y_key,
        confidence_key=confidence_key,
        rim_points_by_frame=rims_by_frame if enable_rim_proximity_boost else None,
        rim_proximity_px=rim_proximity_px,
        rim_interpolation_extra_gap=rim_interpolation_extra_gap,
    )

    tracker = BallTracker(
        dt=dt,
        process_noise=process_noise,
        measurement_noise=measurement_noise,
        gravity_px_per_frame2=tracker_gravity,
        max_missing=max_prediction_frames,
    )

    tracked: List[Point] = []
    active_ball_segment: Optional[int] = None
    # row를 하나씩 보면서 필요한 값만 모으거나 바꿈.
    for data_row in timeline:
        measurement = None
        if _has_xy(data_row, x_key, y_key):
            measurement = (float(data_row[x_key]), float(data_row[y_key]))

        row_segment = to_safe_int(data_row.get("ball_segment"))
        if measurement is not None and row_segment is not None:
            if active_ball_segment is not None and row_segment != active_ball_segment:
                tracker = BallTracker(
                    dt=dt,
                    process_noise=process_noise,
                    measurement_noise=measurement_noise,
                    gravity_px_per_frame2=tracker_gravity,
                    max_missing=max_prediction_frames,
                )
            active_ball_segment = row_segment

        measurement_source = data_row.get("measurement_source", "missing")
        detected = bool(data_row.get("detected", False))
        interpolated = bool(data_row.get("interpolated", False))
        predicted = measurement is None
        effective_confidence = to_safe_float(data_row.get(confidence_key))
        measurement_noise_override = None
        if trust_interpolated_arc and interpolated:
            effective_confidence = max(effective_confidence or 0.0, interpolated_confidence_floor)
            measurement_noise_override = interpolated_measurement_noise

        filtered = tracker.step(
            measurement,
            confidence=effective_confidence,
            measurement_noise=measurement_noise_override,
        )
        if filtered is None:
            continue
        estimated_gravity = to_safe_float(data_row.get("estimated_gravity"))
        if estimated_gravity is None:
            estimated_gravity = tracker_gravity
        occlusion = _find_player_occlusion(
            point=(filtered["x_center"], filtered["y_center"]),
            player_rows=players_by_frame.get(int(data_row["frame"]), []),
            enabled=enable_player_occlusion and predicted,
            margin_px=occlusion_margin_px,
            center_radius_px=player_center_radius_px,
            x_key=x_key,
            y_key=y_key,
        )
        occluded = occlusion is not None
        if occluded:
            measurement_source = "occluded_by_player"

        tracked.append(
            {
                frame_key: int(data_row["frame"]),
                "track_id": stable_track_id,
                "raw_track_id": data_row.get("raw_track_id", data_row.get("track_id", -1)),
                class_key: ball_label,
                confidence_key: round(float(data_row.get(confidence_key) or 0.0), 3),
                x_key: round(filtered["x_center"], 2),
                y_key: round(filtered["y_center"], 2),
                "measurement_x": _round_or_none(data_row.get(x_key)),
                "measurement_y": _round_or_none(data_row.get(y_key)),
                "velocity_x": round(filtered["velocity_x"], 2),
                "velocity_y": round(filtered["velocity_y"], 2),
                "dx": None,
                "dy": None,
                "detected": detected,
                "interpolated": interpolated,
                "predicted": predicted,
                "occluded": occluded,
                "occluded_by_track_id": occlusion.get("track_id") if occlusion else None,
                "occlusion_distance": occlusion.get("distance") if occlusion else None,
                "candidate_count": int(data_row.get("candidate_count", 0)),
                "association_distance": _round_or_none(data_row.get("association_distance")),
                "association_score": _round_or_none(data_row.get("association_score")),
                "measurement_source": measurement_source,
                "tracking_source": filtered["source"],
                "missing_frames": int(filtered["missing_frames"]),
                "estimated_gravity": round(estimated_gravity, 4),
            }
        )

    _append_motion_delta(tracked, x_key, y_key)
    return tracked


# 원래 탐지 결과에서 공 row만 보정된 궤적으로 교체.
# 기존 탐지 결과에서 공 row만 보정된 row로 바꿈.
def correct_detections_with_ball_tracking(
    detection_data: Any,
    total_frames: Optional[int] = None,
    **tracking_kwargs: Any,
) -> Any:

    import pandas as pd


    class_key = tracking_kwargs.get("class_key", "class")
    frame_key = tracking_kwargs.get("frame_key", "frame")
    ball_classes = tuple(
        tracking_kwargs.get(
            "ball_classes",
            ("sports ball", "frisbee", "basketball", "ball"),
        )
    )

    tracked_ball = track_ball_detections(
        detection_data.to_dict("records"),
        total_frames=total_frames,
        **tracking_kwargs,
    )

    if not tracked_ball:
        return detection_data.copy()

    non_ball_data = detection_data[~detection_data[class_key].isin(ball_classes)].copy()
    ball_table = pd.DataFrame(tracked_ball)
    combined = pd.concat([non_ball_data, ball_table], ignore_index=True, sort=False)

    sort_columns = [
        column
        for column in [frame_key, class_key, "track_id"]
        if column in combined.columns
    ]
    if sort_columns:
        combined = combined.sort_values(sort_columns).reset_index(drop=True)

    return combined


# CSV 파일 기준으로 공 추적 보정을 다시 돌리는 함수.
# CSV 입력 기준으로 공 궤적 보정을 다시 돌릴 때 사용.
# 입력 row를 정리한 뒤 필요한 컬럼 순서로 맞춰서 CSV/표 형태로 넘김.
def correct_detection_csv(
    input_csv: str = "video_detection.csv",
    output_csv: str = "video_detection_ball_tracked.csv",
    total_frames: Optional[int] = None,
    **tracking_kwargs: Any,
) -> Any:

    import pandas as pd

    detection_data = pd.read_csv(input_csv)
    corrected_df = correct_detections_with_ball_tracking(
        detection_data,
        total_frames=total_frames,
        **tracking_kwargs,
    )
    corrected_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return corrected_df


# 공 후보나 공 좌표를 정리하는 보조 함수.
def _predict_ball_candidate_position(
    previous: Optional[Mapping[str, Any]],
    latest: Mapping[str, Any],
    target_frame: int,
    frame_key: str,
    x_key: str,
    y_key: str,
) -> Tuple[float, float]:
    latest_x = to_safe_float(latest.get(x_key)) or 0.0
    latest_y = to_safe_float(latest.get(y_key)) or 0.0
    latest_frame = to_safe_int(latest.get(frame_key)) or target_frame
    if previous is None:
        return latest_x, latest_y

    previous_x = to_safe_float(previous.get(x_key))
    previous_y = to_safe_float(previous.get(y_key))
    previous_frame = to_safe_int(previous.get(frame_key))
    if previous_x is None or previous_y is None or previous_frame is None:
        return latest_x, latest_y

    frame_delta = float(latest_frame - previous_frame)
    if frame_delta <= 0.0:
        return latest_x, latest_y

    step = float(target_frame - latest_frame)
    velocity_x = (latest_x - previous_x) / frame_delta
    velocity_y = (latest_y - previous_y) / frame_delta
    return latest_x + velocity_x * step, latest_y + velocity_y * step


# estimate_candidate_speed 처리용 보조 함수.
# previous, latest, frame_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _estimate_candidate_speed(
    previous: Optional[Mapping[str, Any]],
    latest: Mapping[str, Any],
    frame_key: str,
    x_key: str,
    y_key: str,
) -> Optional[float]:
    if previous is None:
        return None

    previous_x = to_safe_float(previous.get(x_key))
    previous_y = to_safe_float(previous.get(y_key))
    previous_frame = to_safe_int(previous.get(frame_key))
    latest_x = to_safe_float(latest.get(x_key))
    latest_y = to_safe_float(latest.get(y_key))
    latest_frame = to_safe_int(latest.get(frame_key))
    if None in (previous_x, previous_y, previous_frame, latest_x, latest_y, latest_frame):
        return None

    frame_delta = float(int(latest_frame) - int(previous_frame))
    if frame_delta <= 0.0:
        return None
    return (
        math.hypot(
            float(latest_x) - float(previous_x),
            float(latest_y) - float(previous_y),
        )
        / frame_delta
    )


# _select_best_point_by_frame에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _select_best_point_by_frame(
    points: Sequence[Mapping[str, Any]],
    frame_key: str,
    confidence_key: str,
) -> Dict[int, Point]:
    best_by_frame: Dict[int, Point] = {}
    # point 항목을 돌면서 조건에 맞는 값만 남김.
    for point in points:
        frame = to_safe_int(point.get(frame_key))
        if frame is None:
            continue

        confidence = to_safe_float(point.get(confidence_key)) or 0.0
        current = best_by_frame.get(frame)
        current_confidence = to_safe_float(current.get(confidence_key)) if current is not None else None
        if current is None or confidence >= (current_confidence or 0.0):
            best_by_frame[frame] = dict(point)

    return best_by_frame


# make_timeline_row 처리용 보조 함수.
# frame, point, x_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _make_timeline_row(
    frame: int,
    point: Optional[Mapping[str, Any]],
    x_key: str,
    y_key: str,
    confidence_key: str,
) -> Point:
    if point is None:
        return {
            "frame": frame,
            x_key: None,
            y_key: None,
            confidence_key: 0.0,
            "detected": False,
            "interpolated": False,
            "measurement_source": "missing",
        }

    data_row = dict(point)
    data_row["frame"] = frame
    data_row[x_key] = to_safe_float(data_row.get(x_key))
    data_row[y_key] = to_safe_float(data_row.get(y_key))
    data_row["raw_track_id"] = data_row.get("track_id", -1)
    data_row["detected"] = _has_xy(data_row, x_key, y_key)
    data_row["interpolated"] = False
    data_row["measurement_source"] = "detected" if data_row["detected"] else "missing"
    return data_row


# interpolate_between 처리용 보조 함수.
# frame, left, right 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _interpolate_between(
    frame: int,
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    previous_point: Optional[Mapping[str, Any]],
    next_point: Optional[Mapping[str, Any]],
    method: str,
    gravity_px_per_frame2: Optional[float],
    x_key: str,
    y_key: str,
) -> Tuple[float, float]:
    left_frame = int(left["frame"])
    right_frame = int(right["frame"])
    total_time = float(right_frame - left_frame)
    elapsed = float(frame - left_frame)
    ratio = elapsed / total_time

    x0 = float(left[x_key])
    y0 = float(left[y_key])
    x1 = float(right[x_key])
    y1 = float(right[y_key])

    x = x0 + (x1 - x0) * ratio
    if method == "linear":
        y = y0 + (y1 - y0) * ratio
        return float(x), float(y)

    left_slope = _slope(previous_point, left, y_key)
    right_slope = _slope(right, next_point, y_key)
    y = _parabolic_y(
        y0,
        y1,
        total_time,
        elapsed,
        left_slope=left_slope,
        right_slope=right_slope,
        gravity_px_per_frame2=gravity_px_per_frame2,
    )
    return float(x), float(y)


# parabolic_y 처리용 보조 함수.
# y0, y1, total_time 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _parabolic_y(
    y0: float,
    y1: float,
    total_time: float,
    elapsed: float,
    left_slope: Optional[float],
    right_slope: Optional[float],
    gravity_px_per_frame2: Optional[float],
) -> float:
    if total_time <= 0:
        return y0

    gravity = 0.0 if gravity_px_per_frame2 is None else float(gravity_px_per_frame2)
    if abs(gravity) > 1e-9:
        initial_velocity = (y1 - y0 - 0.5 * gravity * total_time * total_time) / total_time
        return y0 + initial_velocity * elapsed + 0.5 * gravity * elapsed * elapsed

    if left_slope is not None:
        acceleration = 2.0 * (y1 - y0 - left_slope * total_time) / (total_time * total_time)
        return y0 + left_slope * elapsed + 0.5 * acceleration * elapsed * elapsed

    if right_slope is not None:
        acceleration = 2.0 * (y0 - y1 + right_slope * total_time) / (total_time * total_time)
        initial_velocity = right_slope - acceleration * total_time
        return y0 + initial_velocity * elapsed + 0.5 * acceleration * elapsed * elapsed

    ratio = elapsed / total_time
    return y0 + (y1 - y0) * ratio


# _nearest_valid_before는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _nearest_valid_before(
    timeline: Sequence[Mapping[str, Any]],
    start_frame: int,
    x_key: str,
    y_key: str,
) -> Optional[Mapping[str, Any]]:
    for index in range(start_frame - 1, -1, -1):
        if _has_xy(timeline[index], x_key, y_key):
            return timeline[index]
    return None


# _nearest_valid_after는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _nearest_valid_after(
    timeline: Sequence[Mapping[str, Any]],
    start_frame: int,
    x_key: str,
    y_key: str,
) -> Optional[Mapping[str, Any]]:
    for index in range(start_frame + 1, len(timeline)):
        if _has_xy(timeline[index], x_key, y_key):
            return timeline[index]
    return None


# _support_points_around_gap는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _support_points_around_gap(
    timeline: Sequence[Mapping[str, Any]],
    left_frame: int,
    right_frame: int,
    support_window: int,
    x_key: str,
    y_key: str,
) -> List[Mapping[str, Any]]:
    start = max(0, left_frame - max(0, support_window))
    end = min(len(timeline), right_frame + max(0, support_window) + 1)
    return [data_row for data_row in timeline[start:end] if _has_xy(data_row, x_key, y_key)]


# 림/골대 후보를 정리할 때 쓰는 함수.
# 림/골대 후보를 다룰 때 쓰는 보조 함수.
# 림 후보를 고른 뒤 짧은 누락을 보정해서 슛 판단 기준점으로 사용함.
def _gap_near_rim(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    rim_points_by_frame: Optional[Mapping[int, Sequence[Mapping[str, Any]]]],
    frame_key: str,
    x_key: str,
    y_key: str,
    proximity_px: float,
) -> bool:
    
    
    if not rim_points_by_frame:
        return False

    left_frame = to_safe_int(left.get(frame_key))
    right_frame = to_safe_int(right.get(frame_key))
    left_x = to_safe_float(left.get(x_key))
    left_y = to_safe_float(left.get(y_key))
    right_x = to_safe_float(right.get(x_key))
    right_y = to_safe_float(right.get(y_key))
    if None in (left_frame, right_frame, left_x, left_y, right_x, right_y):
        return False

    middle_frame = int(round((int(left_frame) + int(right_frame)) / 2.0))
    middle_point = (
        (float(left_x) + float(right_x)) / 2.0,
        (float(left_y) + float(right_y)) / 2.0,
    )
    checks = [
        (int(left_frame), (float(left_x), float(left_y))),
        (middle_frame, middle_point),
        (int(right_frame), (float(right_x), float(right_y))),
    ]

    for frame, point in checks:
        rims = _rim_points_around_frame(rim_points_by_frame, frame, window=8)
        if _point_near_rims(point, rims, x_key, y_key, proximity_px):
            return True
    return False


# _rim_points_around_frame에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _rim_points_around_frame(
    rim_points_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    frame: int,
    window: int,
) -> List[Mapping[str, Any]]:
    rims: List[Mapping[str, Any]] = []
    for current_frame in range(frame - window, frame + window + 1):
        rims.extend(rim_points_by_frame.get(current_frame, []))
    return rims


# _point_near_rims는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _point_near_rims(
    point: Tuple[float, float],
    rims: Sequence[Mapping[str, Any]],
    x_key: str,
    y_key: str,
    proximity_px: float,
) -> bool:
    if not rims:
        return False

    # rim 항목을 돌면서 조건에 맞는 값만 남김.
    for rim in rims:
        bbox = _bbox_from_detection(rim, x_key=x_key, y_key=y_key)
        if bbox is not None and _distance_to_bbox(point, bbox) <= proximity_px:
            return True
        center = _center_from_detection(rim, x_key=x_key, y_key=y_key)
        if center is not None and math.hypot(
            point[0] - center[0],
            point[1] - center[1],
        ) <= proximity_px:
            return True
    return False


# _slope는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _slope(
    left: Optional[Mapping[str, Any]],
    right: Optional[Mapping[str, Any]],
    y_key: str,
) -> Optional[float]:
    if left is None or right is None:
        return None
    left_y = to_safe_float(left.get(y_key))
    right_y = to_safe_float(right.get(y_key))
    if left_y is None or right_y is None:
        return None

    frame_delta = float(int(right["frame"]) - int(left["frame"]))
    if frame_delta == 0.0:
        return None
    return (right_y - left_y) / frame_delta


# resolve_gravity 처리용 보조 함수.
# points, requested_gravity, auto_gravity 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _resolve_gravity(
    points: Sequence[Mapping[str, Any]],
    requested_gravity: Optional[float],
    auto_gravity: bool,
    fallback_gravity: float,
    min_gravity: float,
    max_gravity: float,
    frame_key: str,
    y_key: str,
) -> float:
    requested = to_safe_float(requested_gravity)
    if requested is not None and abs(requested) > 1e-9:
        return _clamp_acceleration(requested, min_gravity, max_gravity)

    if auto_gravity:
        estimated = _estimate_gravity_from_points(points, frame_key, y_key)
        if estimated is not None and abs(estimated) > 1e-9:
            return _clamp_acceleration(estimated, min_gravity, max_gravity)

    fallback = to_safe_float(fallback_gravity)
    if fallback is None:
        return 0.0
    return _clamp_acceleration(fallback, min_gravity, max_gravity)


# estimate_gravity_from_points 처리용 보조 함수.
# points, frame_key, y_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _estimate_gravity_from_points(
    points: Sequence[Mapping[str, Any]],
    frame_key: str,
    y_key: str,
) -> Optional[float]:
    valid_points: List[Tuple[int, float]] = []
    for point in points:
        frame = to_safe_int(point.get(frame_key))
        if frame is None:
            frame = to_safe_int(point.get("frame"))
        y = to_safe_float(point.get(y_key))
        if frame is not None and y is not None:
            valid_points.append((frame, y))

    unique_by_frame = {frame: y for frame, y in valid_points}
    ordered = sorted(unique_by_frame.items())
    if len(ordered) < 3:
        return None

    accelerations: List[float] = []
    for (frame0, y0), (frame1, y1), (frame2, y2) in zip(ordered, ordered[1:], ordered[2:]):
        dt0 = float(frame1 - frame0)
        dt1 = float(frame2 - frame1)
        if dt0 <= 0.0 or dt1 <= 0.0:
            continue

        velocity0 = (y1 - y0) / dt0
        velocity1 = (y2 - y1) / dt1
        acceleration = 2.0 * (velocity1 - velocity0) / (dt0 + dt1)
        if math.isfinite(acceleration):
            accelerations.append(acceleration)

    if not accelerations:
        return None

    positive_accelerations = [value for value in accelerations if value > 0.0]
    candidates = positive_accelerations if positive_accelerations else accelerations
    return float(np.median(candidates))


# _clamp_acceleration는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _clamp_acceleration(value: float, min_abs: float, max_abs: float) -> float:
    if abs(value) <= 1e-9:
        return 0.0

    lower = max(0.0, float(min_abs))
    upper = max(lower, float(max_abs))
    sign = -1.0 if value < 0.0 else 1.0
    return sign * min(max(abs(float(value)), lower), upper)


# _group_detections_by_frame에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _group_detections_by_frame(
    detections: Sequence[Mapping[str, Any]],
    frame_key: str,
    class_key: str,
    classes: Sequence[str],
) -> Dict[int, List[Mapping[str, Any]]]:
    class_set = set(classes)
    grouped: Dict[int, List[Mapping[str, Any]]] = {}
    for detection in detections:
        if detection.get(class_key) not in class_set:
            continue
        frame = to_safe_int(detection.get(frame_key))
        if frame is None:
            continue
        grouped.setdefault(frame, []).append(detection)
    return grouped


# 선수 후보나 선수 위치를 정리하는 보조 함수.
# point, player_rows, enabled 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _find_player_occlusion(
    point: Tuple[float, float],
    player_rows: Sequence[Mapping[str, Any]],
    enabled: bool,
    margin_px: float,
    center_radius_px: float,
    x_key: str,
    y_key: str,
) -> Optional[Point]:
    if not enabled or not player_rows:
        return None

    best: Optional[Point] = None
    best_distance: Optional[float] = None
    # player에서 필요한 좌표나 ID만 꺼내 다음 계산에 사용.
    for player in player_rows:
        bbox = _bbox_from_detection(player, x_key=x_key, y_key=y_key)
        if bbox is not None:
            distance = _distance_to_bbox(point, bbox)
            if distance <= margin_px:
                best, best_distance = _better_occlusion_candidate(
                    player,
                    distance,
                    best,
                    best_distance,
                )
            continue

        center = _center_from_detection(player, x_key=x_key, y_key=y_key)
        if center is None:
            continue
        distance = math.hypot(point[0] - center[0], point[1] - center[1])
        if distance <= center_radius_px:
            best, best_distance = _better_occlusion_candidate(
                player,
                distance,
                best,
                best_distance,
            )

    return best


# _better_occlusion_candidate에서는 여러 후보 중 쓸 만한 값만 남김.
def _better_occlusion_candidate(
    player: Mapping[str, Any],
    distance: float,
    best: Optional[Point],
    best_distance: Optional[float],
) -> Tuple[Optional[Point], Optional[float]]:
    if best_distance is not None and distance >= best_distance:
        return best, best_distance

    track_id = to_safe_int(player.get("track_id"))
    return {
        "track_id": track_id if track_id is not None else -1,
        "distance": round(float(distance), 2),
    }, distance


# 후보를 찾아내는 탐지 보조 함수.
# bbox나 중심 좌표를 계산할 때 쓰는 함수.
# bbox 좌표에서 중심, 크기, 거리 값을 계산해서 다음 판단에 넘김.
def _bbox_from_detection(
    detection: Mapping[str, Any],
    x_key: str,
    y_key: str,
) -> Optional[Tuple[float, float, float, float]]:
    if isinstance(detection.get("bbox"), Sequence) and len(detection["bbox"]) >= 4:
        x1, y1, x2, y2 = detection["bbox"][:4]
        return float(x1), float(y1), float(x2), float(y2)

    keys = [
        ("x1", "y1", "x2", "y2"),
        ("x_min", "y_min", "x_max", "y_max"),
        ("left", "top", "right", "bottom"),
    ]
    # (x1_key, y1_key, x2_key, y2_key) 항목을 돌면서 조건에 맞는 값만 남김.
    for x1_key, y1_key, x2_key, y2_key in keys:
        values = [
            to_safe_float(detection.get(x1_key)),
            to_safe_float(detection.get(y1_key)),
            to_safe_float(detection.get(x2_key)),
            to_safe_float(detection.get(y2_key)),
        ]
        if all(value is not None for value in values):
            x1, y1, x2, y2 = [float(value) for value in values]
            return min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)

    width = to_safe_float(detection.get("width"))
    height = to_safe_float(detection.get("height"))
    center = _center_from_detection(detection, x_key=x_key, y_key=y_key)
    if width is not None and height is not None and center is not None:
        half_width = width / 2.0
        half_height = height / 2.0
        return (
            center[0] - half_width,
            center[1] - half_height,
            center[0] + half_width,
            center[1] + half_height,
        )

    return None


# _center_from_detection는 bbox에서 중심 좌표를 뽑아 다음 계산에 넘김.
def _center_from_detection(
    detection: Mapping[str, Any],
    x_key: str,
    y_key: str,
) -> Optional[Tuple[float, float]]:
    x = to_safe_float(detection.get(x_key))
    y = to_safe_float(detection.get(y_key))
    if x is None or y is None:
        return None
    return x, y


# _distance_to_bbox는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _distance_to_bbox(point: Tuple[float, float], bbox: Tuple[float, float, float, float]) -> float:
    x, y = point
    x1, y1, x2, y2 = bbox
    dx = max(x1 - x, 0.0, x - x2)
    dy = max(y1 - y, 0.0, y - y2)
    return math.hypot(dx, dy)


# _interpolated_confidence는 짧게 비는 구간을 앞뒤 값으로 채울 때 사용.
def _interpolated_confidence(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    confidence_key: str,
    floor: float = 0.75,
) -> float:
    left_confidence = to_safe_float(left.get(confidence_key))
    right_confidence = to_safe_float(right.get(confidence_key))
    confidences = [value for value in [left_confidence, right_confidence] if value is not None]
    if not confidences:
        return floor
    return max(0.05, min(1.0, max(floor, min(confidences) * 0.8)))


# _append_motion_delta는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _append_motion_delta(detection_records: List[Point], x_key: str, y_key: str) -> None:
    previous_x: Optional[float] = None
    previous_y: Optional[float] = None
    # detection_record에서 필요한 좌표나 ID만 꺼내 다음 계산에 사용.
    for detection_record in detection_records:
        current_x = to_safe_float(detection_record.get(x_key))
        current_y = to_safe_float(detection_record.get(y_key))
        if previous_x is None or previous_y is None or current_x is None or current_y is None:
            detection_record["dx"] = None
            detection_record["dy"] = None
        else:
            detection_record["dx"] = round(current_x - previous_x, 2)
            detection_record["dy"] = round(current_y - previous_y, 2)
        previous_x = current_x
        previous_y = current_y


# _has_xy는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _has_xy(data_row: Mapping[str, Any], x_key: str, y_key: str) -> bool:
    return to_safe_float(data_row.get(x_key)) is not None and to_safe_float(data_row.get(y_key)) is not None


# 값 형식이 흔들리지 않게 정리하는 함수.
# 값을 같은 형식으로 맞춘 뒤 뒤쪽 계산에서 비교하기 쉽게 넘김.
def to_safe_float(value: Any) -> Optional[float]:
    if value is None:
        return None
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(result) or math.isinf(result):
        return None
    return result


# to_safe_int는 값 형식을 뒤쪽 계산에서 쓰기 좋게 맞춤.
def to_safe_int(value: Any) -> Optional[int]:
    number = to_safe_float(value)
    if number is None:
        return None
    return int(number)


# _round_or_none는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _round_or_none(value: Any, ndigits: int = 2) -> Optional[float]:
    number = to_safe_float(value)
    if number is None:
        return None
    return round(number, ndigits)


# _select_ball_points_by_motion는 공 좌표나 공 후보를 따로 정리할 때 사용.
def _select_ball_points_by_motion(
    points: Sequence[Mapping[str, Any]],
    total_frames: Optional[int],
    frame_key: str,
    x_key: str,
    y_key: str,
    confidence_key: str,
    max_distance_px: float,
    gap_growth_px: float,
    confidence_weight: float,
    restart_after_frames: int,
    restart_min_confidence: float,
) -> List[Point]:
    
    
    if not points:
        return []

    split_gap = max(1, restart_after_frames)
    chunks = _split_motion_points_by_gap(points, frame_key=frame_key, max_gap=split_gap)
    selected: List[Point] = []
    segment_offset = 0

    # chunk 항목을 돌면서 조건에 맞는 값만 남김.
    for chunk in chunks:
        chunk_rows = _select_ball_points_by_motion_chunk(
            chunk,
            total_frames=total_frames,
            frame_key=frame_key,
            x_key=x_key,
            y_key=y_key,
            confidence_key=confidence_key,
            max_distance_px=max_distance_px,
            gap_growth_px=gap_growth_px,
            confidence_weight=confidence_weight,
            restart_after_frames=restart_after_frames,
            restart_min_confidence=restart_min_confidence,
        )
        if not chunk_rows:
            continue

        max_segment = 0
        # data_row에서 필요한 좌표나 ID만 꺼내 다음 계산에 사용.
        for data_row in chunk_rows:
            output_row = dict(data_row)
            segment_id = to_safe_int(output_row.get("ball_segment"))
            if segment_id is None:
                segment_id = 0
            output_row["ball_segment"] = int(segment_id + segment_offset)
            max_segment = max(max_segment, int(segment_id))
            selected.append(output_row)
        segment_offset += max_segment + 1

    return sorted(
        selected,
        key=lambda data_row: (
            to_safe_int(data_row.get(frame_key)) or 0,
            to_safe_float(data_row.get(confidence_key)) or 0.0,
        ),
    )


# _select_ball_points_by_motion_chunk는 공 좌표나 공 후보를 따로 정리할 때 사용.
def _select_ball_points_by_motion_chunk(
    points: Sequence[Mapping[str, Any]],
    total_frames: Optional[int],
    frame_key: str,
    x_key: str,
    y_key: str,
    confidence_key: str,
    max_distance_px: float,
    gap_growth_px: float,
    confidence_weight: float,
    restart_after_frames: int,
    restart_min_confidence: float,
) -> List[Point]:
    if not points:
        return []

    grouped: Dict[int, List[Mapping[str, Any]]] = {}
    for point in points:
        frame = to_safe_int(point.get(frame_key))
        if frame is None:
            continue
        grouped.setdefault(frame, []).append(point)

    if not grouped:
        return []

    frame_candidate_counts = {frame: len(candidates) for frame, candidates in grouped.items()}
    grouped = {
        frame: _top_motion_candidates(candidates, confidence_key, x_key, y_key, limit=12)
        for frame, candidates in grouped.items()
    }
    frame_order = sorted(grouped)

    beam_width = 18
    max_link_gap = max(1, restart_after_frames)
    max_keep_gap = max(max_link_gap, 90)
    restart_gap = max(3, min(restart_after_frames, 8))
    restart_confidence_floor = max(restart_min_confidence, 0.08)
    beam: List[Dict[str, Any]] = []

    for frame in frame_order:
        candidates = grouped.get(frame, [])
        if not candidates:
            continue

        active_beam = _active_motion_hypotheses(
            beam,
            frame=frame,
            frame_key=frame_key,
            max_link_gap=max_keep_gap,
        )
        expanded: List[Dict[str, Any]] = []
        candidate_count = frame_candidate_counts.get(frame, len(candidates))

        # candidate 후보를 비교하면서 너무 약한 값은 제외.
        for candidate in candidates:
            confidence = to_safe_float(candidate.get(confidence_key)) or 0.0
            if confidence < restart_min_confidence:
                continue

            start_row = _make_motion_candidate_row(
                candidate,
                candidate_count=candidate_count,
                segment_id=0,
                association_distance=0.0,
                association_score=0.0,
                association_mode="start",
            )
            expanded.append(
                {
                    "score": (
                        _motion_candidate_quality(candidate, confidence_key, x_key, y_key)
                        + 0.8
                    ),
                    "hits": 1,
                    "restarts": 0,
                    "segment_id": 0,
                    "previous": None,
                    "latest": start_row,
                    "path": [start_row],
                }
            )

            # hypothesis 항목을 돌면서 조건에 맞는 값만 남김.
            for hypothesis in active_beam:
                latest = hypothesis["latest"]
                previous = hypothesis.get("previous")
                transition = _score_motion_transition(
                    previous=previous,
                    latest=latest,
                    candidate=candidate,
                    target_frame=frame,
                    frame_key=frame_key,
                    x_key=x_key,
                    y_key=y_key,
                    confidence_key=confidence_key,
                    max_distance_px=max_distance_px,
                    gap_growth_px=gap_growth_px,
                    confidence_weight=confidence_weight,
                )

                if transition is not None:
                    data_row = _make_motion_candidate_row(
                        candidate,
                        candidate_count=candidate_count,
                        segment_id=int(hypothesis.get("segment_id", 0)),
                        association_distance=transition["distance"],
                        association_score=transition["score"],
                        association_mode="linked",
                    )
                    expanded.append(
                        {
                            "score": float(hypothesis["score"]) + transition["score"],
                            "hits": int(hypothesis["hits"]) + 1,
                            "restarts": int(hypothesis.get("restarts", 0)),
                            "segment_id": int(hypothesis.get("segment_id", 0)),
                            "previous": latest,
                            "latest": data_row,
                            "path": hypothesis["path"] + [data_row],
                        }
                    )
                    continue

                latest_frame = to_safe_int(latest.get(frame_key))
                frame_gap = frame - latest_frame if latest_frame is not None else 0
                if frame_gap < restart_gap or confidence < restart_confidence_floor:
                    continue

                next_segment_id = int(hypothesis.get("segment_id", 0)) + 1
                restart_row = _make_motion_candidate_row(
                    candidate,
                    candidate_count=candidate_count,
                    segment_id=next_segment_id,
                    association_distance=0.0,
                    association_score=0.0,
                    association_mode="restart",
                )
                restart_score = (
                    _motion_candidate_quality(candidate, confidence_key, x_key, y_key)
                    + 0.45
                    - 2.0
                    - min(2.0, max(0, frame_gap - restart_gap) * 0.12)
                )
                expanded.append(
                    {
                        "score": float(hypothesis["score"]) + restart_score,
                        "hits": int(hypothesis["hits"]) + 1,
                        "restarts": int(hypothesis.get("restarts", 0)) + 1,
                        "segment_id": next_segment_id,
                        "previous": None,
                        "latest": restart_row,
                        "path": hypothesis["path"] + [restart_row],
                    }
                )

        if expanded:
            beam = sorted(
                expanded + active_beam,
                key=_rank_motion_hypothesis,
                reverse=True,
            )[:beam_width]
        else:
            beam = active_beam[:beam_width]

    if not beam:
        return []

    best = max(beam, key=_rank_motion_hypothesis)
    return [dict(data_row) for data_row in best["path"]]


# split_motion_points_by_gap 처리용 보조 함수.
# points, frame_key, max_gap 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _split_motion_points_by_gap(
    points: Sequence[Mapping[str, Any]],
    frame_key: str,
    max_gap: int,
) -> List[List[Mapping[str, Any]]]:

    grouped: Dict[int, List[Mapping[str, Any]]] = {}
    for point in points:
        frame = to_safe_int(point.get(frame_key))
        if frame is None:
            continue
        grouped.setdefault(frame, []).append(point)

    chunks: List[List[Mapping[str, Any]]] = []
    current: List[Mapping[str, Any]] = []
    previous_frame: Optional[int] = None
    for frame in sorted(grouped):
        if previous_frame is not None and frame - previous_frame > max_gap:
            if current:
                chunks.append(current)
            current = []
        current.extend(grouped[frame])
        previous_frame = frame

    if current:
        chunks.append(current)
    return chunks


# _top_motion_candidates에서는 여러 후보 중 쓸 만한 값만 남김.
def _top_motion_candidates(
    candidates: Sequence[Mapping[str, Any]],
    confidence_key: str,
    x_key: str,
    y_key: str,
    limit: int,
) -> List[Mapping[str, Any]]:
    valid_candidates = [
        candidate
        for candidate in candidates
        if to_safe_float(candidate.get(x_key)) is not None
        and to_safe_float(candidate.get(y_key)) is not None
    ]
    return sorted(
        valid_candidates,
        key=lambda item: _motion_candidate_quality(item, confidence_key, x_key, y_key),
        reverse=True,
    )[:limit]


# motion_candidate_quality 처리용 보조 함수.
# candidate, confidence_key, x_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _motion_candidate_quality(
    candidate: Mapping[str, Any],
    confidence_key: str,
    x_key: str,
    y_key: str,
) -> float:
    confidence = to_safe_float(candidate.get(confidence_key)) or 0.0
    quality = confidence * 4.0

    bbox = _bbox_from_detection(candidate, x_key=x_key, y_key=y_key)
    if bbox is None:
        return quality - 0.25

    x1, y1, x2, y2 = bbox
    width = max(0.0, x2 - x1)
    height = max(0.0, y2 - y1)
    if width <= 0.0 or height <= 0.0:
        return quality - 0.75

    area = width * height
    aspect = max(width / height, height / width)

    quality -= max(0.0, aspect - 1.35) * 0.65
    quality -= max(0.0, area - 2800.0) / 2200.0
    quality -= max(0.0, 18.0 - area) / 60.0
    return quality


# active_motion_hypotheses 처리용 보조 함수.
# beam, frame, frame_key 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _active_motion_hypotheses(
    beam: Sequence[Mapping[str, Any]],
    frame: int,
    frame_key: str,
    max_link_gap: int,
) -> List[Dict[str, Any]]:
    active: List[Dict[str, Any]] = []
    for hypothesis in beam:
        latest = hypothesis.get("latest")
        if not isinstance(latest, Mapping):
            continue
        latest_frame = to_safe_int(latest.get(frame_key))
        if latest_frame is None:
            continue
        frame_gap = frame - latest_frame
        if frame_gap <= 0 or frame_gap > max_link_gap:
            continue

        clone = dict(hypothesis)
        clone["score"] = float(clone.get("score", 0.0)) - min(
            1.5,
            max(0, frame_gap - 1) * 0.08,
        )
        active.append(clone)
    return active


# score_motion_transition 처리용 보조 함수.
# previous, latest, candidate 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _score_motion_transition(
    previous: Optional[Mapping[str, Any]],
    latest: Mapping[str, Any],
    candidate: Mapping[str, Any],
    target_frame: int,
    frame_key: str,
    x_key: str,
    y_key: str,
    confidence_key: str,
    max_distance_px: float,
    gap_growth_px: float,
    confidence_weight: float,
) -> Optional[Dict[str, float]]:
    latest_frame = to_safe_int(latest.get(frame_key))
    candidate_x = to_safe_float(candidate.get(x_key))
    candidate_y = to_safe_float(candidate.get(y_key))
    if latest_frame is None or candidate_x is None or candidate_y is None:
        return None

    frame_gap = target_frame - latest_frame
    if frame_gap <= 0:
        return None
    if frame_gap > 12:
        return None

    predicted_x, predicted_y = _predict_ball_candidate_position(
        previous,
        latest,
        target_frame,
        frame_key,
        x_key,
        y_key,
    )
    distance = math.hypot(candidate_x - predicted_x, candidate_y - predicted_y)

    dynamic_gate = max_distance_px + gap_growth_px * min(max(0, frame_gap - 1), 4)
    speed = _estimate_candidate_speed(previous, latest, frame_key, x_key, y_key)
    if speed is not None:
        dynamic_gate += min(max_distance_px * 0.45, speed * frame_gap * 0.22)
    dynamic_gate = min(
        dynamic_gate,
        max_distance_px + gap_growth_px * 4.0 + max_distance_px * 0.45,
    )
    if distance > dynamic_gate:
        return None

    quality = _motion_candidate_quality(candidate, confidence_key, x_key, y_key)
    confidence = to_safe_float(candidate.get(confidence_key)) or 0.0
    distance_penalty = distance / max(45.0, max_distance_px * 0.45)
    gap_penalty = max(0, frame_gap - 1) * 0.28
    acceleration_penalty = _candidate_acceleration_penalty(
        previous,
        latest,
        candidate,
        target_frame,
        frame_key,
        x_key,
        y_key,
    )

    score = (
        1.35
        + quality
        + confidence * max(0.0, confidence_weight) / 45.0
        - distance_penalty
        - gap_penalty
        - acceleration_penalty
    )
    return {"score": float(score), "distance": float(distance)}


# candidate_acceleration_penalty 처리용 보조 함수.
def _candidate_acceleration_penalty(
    previous: Optional[Mapping[str, Any]],
    latest: Mapping[str, Any],
    candidate: Mapping[str, Any],
    target_frame: int,
    frame_key: str,
    x_key: str,
    y_key: str,
) -> float:
    if previous is None:
        return 0.0

    previous_x = to_safe_float(previous.get(x_key))
    previous_y = to_safe_float(previous.get(y_key))
    previous_frame = to_safe_int(previous.get(frame_key))
    latest_x = to_safe_float(latest.get(x_key))
    latest_y = to_safe_float(latest.get(y_key))
    latest_frame = to_safe_int(latest.get(frame_key))
    candidate_x = to_safe_float(candidate.get(x_key))
    candidate_y = to_safe_float(candidate.get(y_key))
    if None in (
        previous_x,
        previous_y,
        previous_frame,
        latest_x,
        latest_y,
        latest_frame,
        candidate_x,
        candidate_y,
    ):
        return 0.0

    previous_delta = float(int(latest_frame) - int(previous_frame))
    next_delta = float(target_frame - int(latest_frame))
    if previous_delta <= 0.0 or next_delta <= 0.0:
        return 0.0

    previous_vx = (float(latest_x) - float(previous_x)) / previous_delta
    previous_vy = (float(latest_y) - float(previous_y)) / previous_delta
    next_vx = (float(candidate_x) - float(latest_x)) / next_delta
    next_vy = (float(candidate_y) - float(latest_y)) / next_delta
    acceleration = math.hypot(next_vx - previous_vx, next_vy - previous_vy)
    return min(3.0, acceleration / 75.0)


# _make_motion_candidate_row에서는 여러 후보 중 쓸 만한 값만 남김.
def _make_motion_candidate_row(
    candidate: Mapping[str, Any],
    candidate_count: int,
    segment_id: int,
    association_distance: float,
    association_score: float,
    association_mode: str,
) -> Point:
    data_row = dict(candidate)
    data_row["candidate_count"] = int(candidate_count)
    data_row["association_distance"] = round(float(association_distance), 2)
    data_row["association_score"] = round(float(association_score), 2)
    data_row["association_mode"] = association_mode
    data_row["ball_segment"] = int(segment_id)
    return data_row


# _rank_motion_hypothesis는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _rank_motion_hypothesis(hypothesis: Mapping[str, Any]) -> float:
    return (
        float(hypothesis.get("score", 0.0))
        + int(hypothesis.get("hits", 0)) * 1.15
        - int(hypothesis.get("restarts", 0)) * 0.55
    )


__all__ = [
    "BallTracker",
    "interpolate_ball_path",
    "track_ball_detections",
    "correct_detections_with_ball_tracking",
    "correct_detection_csv",
]




## Core 3. 탐지 결과 후처리

Core 1 코트 필터랑 Core 2 공 보정을 이어서 쓰는 부분.
여기서 정리된 결과를 뒤쪽 이벤트 계산에서 다시 사용함.


In [3]:
# - Core 1에서 코트 밖 row를 줄이고, Core 2에서 공 row를 한 궤적으로 정리.
# - 여기서는 두 처리를 순서대로 묶어서 뒤쪽 이벤트 계산용 입력을 만듦.
# - tracking_kwargs를 바꾸면 공 보간/추적 성향이 달라질 수 있음.

# Core 3. 탐지 결과 후처리

# 코트 필터랑 공 보정을 한 번에 돌리는 부분.

# 순서 헷갈려서 세 단계로만 나눠둠.
# 1. 코트 바깥 객체 제거
# 2. 코트 안 객체만 이용해 공 후보 선택
# 3. 공 누락/가림/포물선 궤적 보정


from __future__ import annotations

from typing import Any, Mapping, Optional, Sequence, Tuple

import pandas as pd


# 코트 필터와 공 보정을 한 번에 묶어서 적용.
# 공 후보를 모은 뒤 confidence, 위치, 이전 이동 방향을 같이 보고 남길 후보를 고름.
def postprocess_basketball_detections(
    detection_data: pd.DataFrame,
    total_frames: Optional[int] = None,
    court_polygon: Optional[Sequence[Any]] = None,
    frame_size: Optional[Tuple[int, int]] = None,
    video_path: Optional[str] = "Video Project.mp4",
    polygon_is_normalized: bool = True,
    court_margin_px: float = 12.0,
    court_margin_by_class: Optional[Mapping[str, float]] = None,
    tracking_kwargs: Optional[Mapping[str, Any]] = None,
) -> pd.DataFrame:
    # 코트 필터와 공 추적 보정을 순서대로 적용함.

    # 먼저 코트 polygon 밖에 있는 선수/공/기타 객체를 제거.
    # 이 단계를 먼저 해야 뒤쪽 공 추적이 관중석 오탐에 덜 흔들린다.
    
    court_filtered_data = filter_detections_by_court(
        detection_data,
        court_polygon=court_polygon,
        frame_size=frame_size,
        video_path=video_path,
        polygon_is_normalized=polygon_is_normalized,
        margin_px=court_margin_px,
        margin_by_class=court_margin_by_class,
    )

    
    tracker_options = {
        "max_gap": 12,
        "auto_gravity": True,
        "enable_player_occlusion": True,
        "use_motion_association": True,
        "association_max_distance_px": 280.0,
        "association_gap_growth_px": 55.0,
        "association_confidence_weight": 18.0,
        "association_restart_after_frames": 7,
        "association_restart_min_confidence": 0.004,
        "player_center_radius_px": 90.0,
        "enable_rim_proximity_boost": False,
        "rim_interpolation_extra_gap": 0,
        "max_prediction_frames": 12,
        "process_noise": 90.0,
        "measurement_noise": 7.0,
        "interpolated_measurement_noise": 8.0,
        "interpolated_confidence_floor": 0.45,
        "ball_classes": ("ball", "sports ball", "basketball", "frisbee"),
        "ball_label": "ball",
        "player_classes": ("player", "person"),
    }
    if tracking_kwargs:
        tracker_options.update(dict(tracking_kwargs))

    # 코트 안 객체만 남은 DataFrame에서 공 row를 보정된 단일 궤적으로 교체함.
    return correct_detections_with_ball_tracking(
        court_filtered_data,
        total_frames=total_frames,
        **tracker_options,
    )


# 탐지 CSV를 읽고 후처리된 CSV로 저장.
# 입력 row를 정리한 뒤 필요한 컬럼 순서로 맞춰서 CSV/표 형태로 넘김.
def postprocess_detection_csv(
    input_csv: str = "video_detection.csv",
    output_csv: str = "video_detection_postprocessed.csv",
    total_frames: Optional[int] = None,
    court_polygon: Optional[Sequence[Any]] = None,
    frame_size: Optional[Tuple[int, int]] = None,
    video_path: Optional[str] = "Video Project.mp4",
    polygon_is_normalized: bool = True,
    court_margin_px: float = 12.0,
    court_margin_by_class: Optional[Mapping[str, float]] = None,
    tracking_kwargs: Optional[Mapping[str, Any]] = None,
) -> pd.DataFrame:
    # 만든 CSV를 다시 읽어서 후처리 CSV로 저장.
    # 앞에서 만든 탐지 CSV 그대로 사용.
    detection_data = pd.read_csv(input_csv)
    result_df = postprocess_basketball_detections(
        detection_data,
        total_frames=total_frames,
        court_polygon=court_polygon,
        frame_size=frame_size,
        video_path=video_path,
        polygon_is_normalized=polygon_is_normalized,
        court_margin_px=court_margin_px,
        court_margin_by_class=court_margin_by_class,
        tracking_kwargs=tracking_kwargs,
    )
    # 뒤쪽 분석은 이 CSV 기준으로 봄.
    result_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return result_df


__all__ = [
    "postprocess_basketball_detections",
    "postprocess_detection_csv",
]




## Core 4. 공 좌표 다시 정리

공 좌표가 갑자기 튀거나 선수 몸 안쪽에 붙어서 잡히는 경우를 한 번 더 줄인다.
너무 긴 구간은 억지로 보간하지 않고, 짧게 비는 부분만 채우는 쪽으로 둠.


In [4]:
# - Core 2가 만든 공 궤적에서 한 번 더 이상치와 흔들림을 줄임.
# - 선수 몸 안쪽, 화면 가장자리, 너무 큰 이동 같은 경우를 오탐 후보로 봄.
# - confidence가 낮아도 실제 슛/패스일 수 있어서 너무 세게 지우지는 않음.
# - 짧은 누락은 채우고, 긴 누락은 새 segment로 보는 쪽으로 둠.
# - 마지막 CSV용 bbox가 비면 중심 좌표 기준으로 작은 bbox를 다시 만듦.

# Core 4. 공 좌표 한번 더 정리

# 공 좌표가 자주 튀어서, 누락과 선수 몸통 오탐을 한 번 더 정리.

# 이 부분은 결과 CSV 품질에 바로 영향을 줘서 따로 빼 두었다.

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd


BALL_CLASS_NAMES = {"ball", "sports ball", "basketball", "frisbee"}
BASE_TRACKING_COLUMNS = [
    "frame",
    "track_id",
    "class",
    "confidence",
    "x_center",
    "y_center",
    "x1",
    "y1",
    "x2",
    "y2",
]


@dataclass
# 공 좌표 정리에 필요한 기준값들을 모아둔 설정.
# 설정값이나 상태를 먼저 묶어두고, 아래 메서드들이 그 값을 같이 사용함.
class BallRefinerConfig:

    max_step_px_per_frame: float = 160.0
    strict_max_step_px_per_frame: float = 135.0
    # 탐지를 놓친 구간이 너무 길면 보간이 오히려 거짓 좌표를 만들 수 있으므로 5프레임까지만 제한.
    max_interpolation_gap: int = 5
    # 선수 가림 등으로 짧게 놓친 직후, 이전 속도 벡터로 예측할 최대 프레임 수.
    max_prediction_gap: int = 3
    # 1~2px 수준의 흔들림을 줄이기 위한 이동 평균 창.
    smoothing_window: int = 3
    # 새 segment를 시작할 때 너무 낮은 confidence 후보를 무조건 믿지 않기 위한 기준.
    restart_confidence_threshold: float = 0.25
    # 긴 공백 뒤 완전히 다른 위치에서 다시 시작하는 후보는 confidence가 충분히 높아야 인정.
    high_confidence_restart_threshold: float = 0.65
    # 화면 가장자리 후보는 관중석/광고판/조명 오탐이 많아 별도 감점.
    edge_margin_px: float = 24.0
    edge_confidence_threshold: float = 0.55
    # 선수/심판/골대 같은 농구 맥락에서 너무 멀리 떨어진 낮은 confidence 후보는 제거.
    context_distance_px: float = 420.0
    context_confidence_threshold: float = 0.18
    player_body_confidence_threshold: float = 0.56
    player_body_lock_min_frames: int = 6
    tail_length: int = 10
    default_ball_box_size: float = 18.0


# 영상 fps/해상도에 맞게 공 보정 기준을 조금 조절.
# frame_size, fps 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def build_adaptive_refiner_config(
    frame_size: Tuple[int, int],
    fps: float,
) -> BallRefinerConfig:

    width, height = frame_size
    max_dim = max(float(width), float(height), 1.0)
    fps_value = float(fps) if fps and fps > 0 else 30.0

    # fps가 다르면 한 프레임 이동량도 달라져서 너무 세게 지우지 않게 제한.
    fps_factor = max(0.75, min(1.25, 30.0 / fps_value))

    resolution_factor = max(0.85, min(1.35, max_dim / 1280.0))

    return BallRefinerConfig(
        max_step_px_per_frame=165.0 * fps_factor,
        strict_max_step_px_per_frame=138.0 * fps_factor,
        max_interpolation_gap=5,
        max_prediction_gap=5 if fps_value >= 45.0 else 3,
        smoothing_window=3,
        restart_confidence_threshold=0.24,
        high_confidence_restart_threshold=0.62,
        edge_margin_px=max(20.0, min(42.0, max_dim * 0.018)),
        edge_confidence_threshold=0.55,
        context_distance_px=420.0 * resolution_factor,
        context_confidence_threshold=0.18,
        player_body_confidence_threshold=0.56,
        player_body_lock_min_frames=max(5, int(round(fps_value * 0.20))),
        tail_length=10,
        default_ball_box_size=max(12.0, min(26.0, max_dim * 0.012)),
    )


# 공 좌표가 튀는 구간을 줄이고 CSV용 결과로 정리하는 클래스.
# 공 좌표 이상치, 보간, 평활화를 묶어 처리하는 클래스.
class BallCoordinateRefiner:
    # 원본 공 후보랑 정리한 공 좌표를 같이 들고 있음.

    def __init__(self, config: Optional[BallRefinerConfig] = None) -> None:
        self.config = config or BallRefinerConfig()

    # export_raw_detection_verify 처리용 보조 함수.
    # raw_df, output_path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def export_raw_detection_verify(self, raw_df: pd.DataFrame, output_path: str) -> pd.DataFrame:

        raw_ball_df = self._ball_rows(raw_df).copy()
        if raw_ball_df.empty:
            verify_df = pd.DataFrame(
                columns=[
                    "frame",
                    "raw_candidate_index",
                    "track_id",
                    "confidence",
                    "x_center",
                    "y_center",
                    "x1",
                    "y1",
                    "x2",
                    "y2",
                    "bbox_width",
                    "bbox_height",
                    "bbox_area",
                ]
            )
        else:
            raw_ball_df = self._ensure_numeric(raw_ball_df, ["frame", "confidence", "x_center", "y_center", "x1", "y1", "x2", "y2"])
            raw_ball_df = raw_ball_df.sort_values(["frame", "confidence"], ascending=[True, False]).copy()
            raw_ball_df["raw_candidate_index"] = raw_ball_df.groupby("frame").cumcount()
            raw_ball_df["bbox_width"] = (raw_ball_df["x2"] - raw_ball_df["x1"]).abs()
            raw_ball_df["bbox_height"] = (raw_ball_df["y2"] - raw_ball_df["y1"]).abs()
            raw_ball_df["bbox_area"] = raw_ball_df["bbox_width"] * raw_ball_df["bbox_height"]
            verify_df = raw_ball_df[
                [
                    "frame",
                    "raw_candidate_index",
                    "track_id",
                    "confidence",
                    "x_center",
                    "y_center",
                    "x1",
                    "y1",
                    "x2",
                    "y2",
                    "bbox_width",
                    "bbox_height",
                    "bbox_area",
                ]
            ].copy()

        output = Path(output_path)
        output.parent.mkdir(parents=True, exist_ok=True)
        verify_df.to_csv(output, index=False, encoding="utf-8-sig")
        return verify_df

    # 값 형식이 흔들리지 않게 정리하는 함수.
    # 값을 같은 형식으로 맞춘 뒤 뒤쪽 계산에서 비교하기 쉽게 넘김.
    def create_cleaned_tracking_results(
        self,
        tracking_data: pd.DataFrame,
        frame_size: Tuple[int, int],
        total_frames: Optional[int],
        output_path: str,
    ) -> pd.DataFrame:
        # 공 좌표 정리하고 tracking CSV로 넘김.

        result_data = tracking_data.copy()
        if result_data.empty:
            self._write_csv(result_data, output_path)
            return result_data

        non_ball_data = result_data[~result_data.get("class", pd.Series(dtype=str)).isin(BALL_CLASS_NAMES)].copy()
        ball_table = self._select_frame_ball_observations(result_data)
        context_by_frame = self._context_points_by_frame(non_ball_data)
        player_by_frame = self._player_records_by_frame(non_ball_data)

        if ball_table.empty:
            cleaned_ball_data = non_ball_data
        else:
            # 공 후보만 따로 정리한 뒤, 주변 선수/골대 맥락과 이동량을 같이 보고 이상치를 줄인다.
            accepted_points = self._remove_motion_outliers(
                ball_table,
                frame_size,
                context_by_frame,
                player_by_frame,
            )
            timeline = self._build_clean_ball_timeline(accepted_points, total_frames=total_frames)
            # 선수 몸통에 고정된 것처럼 보이는 공 후보는 실제 공보다 오탐일 가능성이 높아서 한 번 더 제거.
            timeline = self._remove_player_body_locked_timeline(timeline)
            cleaned_ball_df = self._smooth_ball_timeline(timeline, frame_size)
            cleaned_ball_data = pd.concat([non_ball_data, cleaned_ball_df], ignore_index=True, sort=False)

        cleaned_ball_data = self._normalize_output_columns(cleaned_ball_data)
        self._write_csv(cleaned_ball_data, output_path)
        return cleaned_ball_data


    # _select_frame_ball_observations에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
    def _select_frame_ball_observations(self, tracking_data: pd.DataFrame) -> pd.DataFrame:
        # 프레임별 공 row가 여러 개일 때 confidence가 가장 높은 후보를 선택.

        ball_table = self._ball_rows(tracking_data).copy()
        if ball_table.empty:
            return ball_table

        ball_table = self._ensure_numeric(ball_table, ["frame", "confidence", "x_center", "y_center", "x1", "y1", "x2", "y2"])
        ball_table = ball_table.dropna(subset=["frame", "x_center", "y_center"]).copy()
        if ball_table.empty:
            return ball_table

        ball_table["bbox_size"] = self._bbox_size_series(ball_table)
        ball_table = ball_table.sort_values(["frame", "confidence"], ascending=[True, False])
        return ball_table.drop_duplicates("frame", keep="first").sort_values("frame").reset_index(drop=True)

    # remove_motion_outliers 처리용 보조 함수.
    # ball_table, frame_size, context_by_frame 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _remove_motion_outliers(
        self,
        ball_table: pd.DataFrame,
        frame_size: Tuple[int, int],
        context_by_frame: Mapping[int, Sequence[Tuple[float, float]]],
        player_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    ) -> List[Dict[str, Any]]:
        

        max_step = self._scaled_max_step(frame_size)
        accepted: List[Dict[str, Any]] = []
        segment_id = 0

        # row를 하나씩 보면서 필요한 값만 모으거나 바꿈.
        for data_row in ball_table.to_dict("records"):
            point = dict(data_row)
            point["ball_status"] = "Detected"
            point["cleaning_note"] = "accepted"
            point["raw_x_center"] = point.get("x_center")
            point["raw_y_center"] = point.get("y_center")
            point["is_reliable"] = True
            point["reliability_reason"] = "accepted"

            confidence = float(point.get("confidence", 0.0) or 0.0)
            near_edge = self._is_near_frame_edge(point, frame_size)
            context_distance = self._nearest_context_distance(point, context_by_frame)
            point["context_distance"] = round(context_distance, 2) if np.isfinite(context_distance) else None
            self._apply_player_body_metadata(point, self._player_body_overlap(point, player_by_frame))

            # 화면 가장자리 후보는 광고판/관중석/조명 오탐이 많아 첫 좌표부터 가장자리 저신뢰 후보이면 추적 시작점으로 사용하지 않는다.
            if not accepted and near_edge and confidence < self.config.edge_confidence_threshold:
                point["cleaning_note"] = "removed_edge_low_confidence"
                continue

            if (
                np.isfinite(context_distance)
                and context_distance > self.config.context_distance_px
                and confidence < self.config.context_confidence_threshold
            ):
                point["cleaning_note"] = "removed_far_from_context"
                continue

            if self._is_low_confidence_player_body_point(point):
                point["cleaning_note"] = "removed_player_body_false_positive"
                continue

            if not accepted:
                point["ball_segment"] = segment_id
                accepted.append(point)
                continue

            previous = accepted[-1]
            frame_gap = int(point["frame"]) - int(previous["frame"])
            if frame_gap <= 0:
                continue

            distance = float(np.hypot(
                float(point["x_center"]) - float(previous["x_center"]),
                float(point["y_center"]) - float(previous["y_center"]),
            ))
            allowed_distance = max_step * max(1, min(frame_gap, 3))

            if frame_gap == 1 and distance > max_step:
                point["cleaning_note"] = "removed_impossible_speed"
                continue

            if frame_gap <= self.config.max_interpolation_gap + 1 and distance > allowed_distance:
                point["cleaning_note"] = "removed_short_gap_jump"
                continue

            if distance > allowed_distance and confidence < self.config.restart_confidence_threshold:
                point["cleaning_note"] = "removed_far_low_confidence"
                continue

            if distance > allowed_distance and near_edge:
                point["cleaning_note"] = "removed_edge_restart"
                continue

            if (
                distance > allowed_distance
                and np.isfinite(context_distance)
                and context_distance > self.config.context_distance_px
                and confidence < self.config.high_confidence_restart_threshold
            ):
                point["cleaning_note"] = "removed_far_restart_without_context"
                continue

            if distance > allowed_distance:
                if confidence < self.config.high_confidence_restart_threshold:
                    point["cleaning_note"] = "removed_weak_new_segment"
                    continue
                segment_id += 1
                point["cleaning_note"] = "accepted_new_segment"

            point["ball_segment"] = segment_id
            point["motion_distance"] = round(distance, 3)
            point["frame_gap_from_previous_ball"] = frame_gap
            accepted.append(point)

        accepted = self._remove_player_body_locked_runs(accepted)
        return self._prune_weak_segments(accepted)

    # 공 후보나 공 좌표를 정리하는 보조 함수.
    # 공 후보를 모은 뒤 confidence, 위치, 이전 이동 방향을 같이 보고 남길 후보를 고름.
    def _build_clean_ball_timeline(
        self,
        accepted_points: Sequence[Mapping[str, Any]],
        total_frames: Optional[int],
    ) -> pd.DataFrame:

        if not accepted_points:
            return pd.DataFrame(columns=BASE_TRACKING_COLUMNS + ["ball_status", "cleaning_note"])

        sorted_points = sorted(accepted_points, key=lambda item: int(item["frame"]))
        data_rows: List[Dict[str, Any]] = []

        # (index, point) 항목을 돌면서 조건에 맞는 값만 남김.
        for index, point in enumerate(sorted_points):
            data_rows.append(self._make_ball_row(point, status="Detected", note=str(point.get("cleaning_note", "accepted"))))

            if index >= len(sorted_points) - 1:
                continue

            next_point = sorted_points[index + 1]
            frame_gap = int(next_point["frame"]) - int(point["frame"])
            if frame_gap <= 1:
                continue

            same_segment = int(point.get("ball_segment", -1)) == int(next_point.get("ball_segment", -2))
            missing_count = frame_gap - 1

            if same_segment and missing_count <= self.config.max_interpolation_gap:
                data_rows.extend(self._interpolate_between(point, next_point, missing_count))
            elif same_segment:
                data_rows.extend(self._predict_after(point, sorted_points, index, missing_count))

        if total_frames is not None and sorted_points:
            last_point = sorted_points[-1]
            remaining = int(total_frames) - int(last_point["frame"]) - 1
            if remaining > 0:
                data_rows.extend(self._predict_after(last_point, sorted_points, len(sorted_points) - 1, remaining))

        return pd.DataFrame(data_rows).sort_values("frame").reset_index(drop=True)

    # prune_weak_segments 처리용 보조 함수.
    # points 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _prune_weak_segments(self, points: Sequence[Mapping[str, Any]]) -> List[Dict[str, Any]]:
        

        if not points:
            return []

        grouped: Dict[int, List[Dict[str, Any]]] = {}
        for point in points:
            grouped.setdefault(int(point.get("ball_segment", 0)), []).append(dict(point))

        kept: List[Dict[str, Any]] = []
        # 끊긴 구간을 하나씩 보면서 이어도 되는지 판단.
        for segment_points in grouped.values():
            confidences = [float(point.get("confidence", 0.0) or 0.0) for point in segment_points]
            context_distances = [
                float(point["context_distance"])
                for point in segment_points
                if point.get("context_distance") is not None
            ]
            max_confidence = max(confidences) if confidences else 0.0
            min_context_distance = min(context_distances) if context_distances else float("inf")

            if (
                len(segment_points) >= 3
                or max_confidence >= self.config.high_confidence_restart_threshold
                or min_context_distance <= self.config.context_distance_px
            ):
                kept.extend(segment_points)

        return sorted(kept, key=lambda item: int(item["frame"]))

    # interpolate_between 처리용 보조 함수.
    # left, right, missing_count 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _interpolate_between(
        self,
        left: Mapping[str, Any],
        right: Mapping[str, Any],
        missing_count: int,
    ) -> List[Dict[str, Any]]:

        data_rows: List[Dict[str, Any]] = []
        left_frame = int(left["frame"])
        right_frame = int(right["frame"])
        total_gap = float(right_frame - left_frame)
        # 프레임을 하나씩 보면서 탐지 결과와 시간 기준값을 같이 갱신.
        for frame in range(left_frame + 1, right_frame):
            ratio = float(frame - left_frame) / total_gap
            x = float(left["x_center"]) + (float(right["x_center"]) - float(left["x_center"])) * ratio
            y = float(left["y_center"]) + (float(right["y_center"]) - float(left["y_center"])) * ratio
            confidence = max(0.05, min(float(left.get("confidence", 0.0) or 0.0), float(right.get("confidence", 0.0) or 0.0)) * 0.8)
            data_rows.append(
                self._make_ball_row(
                    left,
                    frame=frame,
                    x_center=x,
                    y_center=y,
                    confidence=confidence,
                    status="Interpolated",
                    note=f"linear_interpolation_gap_{missing_count}",
                )
            )
        return data_rows

    # predict_after 처리용 보조 함수.
    # point, sorted_points, point_index 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _predict_after(
        self,
        point: Mapping[str, Any],
        sorted_points: Sequence[Mapping[str, Any]],
        point_index: int,
        missing_count: int,
    ) -> List[Dict[str, Any]]:

        velocity = self._estimate_velocity(sorted_points, point_index)
        if velocity is None:
            return []

        vx, vy = velocity
        data_rows: List[Dict[str, Any]] = []
        predict_count = min(self.config.max_prediction_gap, missing_count)
        # step 항목을 돌면서 조건에 맞는 값만 남김.
        for step in range(1, predict_count + 1):
            frame = int(point["frame"]) + step
            x = float(point["x_center"]) + vx * step
            y = float(point["y_center"]) + vy * step
            confidence = max(0.01, float(point.get("confidence", 0.0) or 0.0) * (0.75**step))
            data_rows.append(
                self._make_ball_row(
                    point,
                    frame=frame,
                    x_center=x,
                    y_center=y,
                    confidence=confidence,
                    status="Predicted",
                    note="linear_prediction_occlusion",
                )
            )
        return data_rows

    # _smooth_ball_timeline는 공 좌표나 공 후보를 따로 정리할 때 사용.
    def _smooth_ball_timeline(self, timeline: pd.DataFrame, frame_size: Tuple[int, int]) -> pd.DataFrame:

        if timeline.empty:
            return timeline

        smoothed_parts: List[pd.DataFrame] = []
        window = max(1, int(self.config.smoothing_window))
        if window % 2 == 0:
            window += 1

        for _, segment_df in timeline.groupby("ball_segment", sort=True):
            segment_df = segment_df.sort_values("frame").copy()
            segment_df["x_center"] = segment_df["x_center"].rolling(window=window, center=True, min_periods=1).mean()
            segment_df["y_center"] = segment_df["y_center"].rolling(window=window, center=True, min_periods=1).mean()
            segment_df = self._rebuild_ball_bbox(segment_df)
            smoothed_parts.append(segment_df)

        smoothed_df = pd.concat(smoothed_parts, ignore_index=True).sort_values("frame").reset_index(drop=True)
        smoothed_df = self._remove_implausible_clean_jumps(smoothed_df, frame_size)
        return self._append_reliability_columns(smoothed_df, frame_size)

    # _remove_implausible_clean_jumps는 값 형식을 뒤쪽 계산에서 쓰기 좋게 맞춤.
    def _remove_implausible_clean_jumps(self, ball_table: pd.DataFrame, frame_size: Tuple[int, int]) -> pd.DataFrame:

        if ball_table.empty:
            return ball_table

        strict_step = self._scaled_strict_step(frame_size)
        kept_rows: List[Mapping[str, Any]] = []

        for data_row in ball_table.sort_values("frame").to_dict("records"):
            if not kept_rows:
                kept_rows.append(data_row)
                continue

            previous = kept_rows[-1]
            frame_gap = int(data_row["frame"]) - int(previous["frame"])
            if frame_gap <= 0:
                continue

            distance = float(np.hypot(
                float(data_row["x_center"]) - float(previous["x_center"]),
                float(data_row["y_center"]) - float(previous["y_center"]),
            ))
            allowed = strict_step * max(1, min(frame_gap, 2))

            if frame_gap <= self.config.max_interpolation_gap + 1 and distance > allowed:
                data_row["is_reliable"] = False
                data_row["reliability_reason"] = "removed_final_jump_gate"
                continue

            kept_rows.append(data_row)

        if not kept_rows:
            return ball_table.iloc[0:0].copy()
        return pd.DataFrame(kept_rows).sort_values("frame").reset_index(drop=True)

    # append_reliability_columns 처리용 보조 함수.
    # ball_table, frame_size 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _append_reliability_columns(self, ball_table: pd.DataFrame, frame_size: Tuple[int, int]) -> pd.DataFrame:
    

        if ball_table.empty:
            return ball_table

        result_data = ball_table.sort_values("frame").copy()
        strict_step = self._scaled_strict_step(frame_size)
        previous_row: Optional[pd.Series] = None
        jump_distances: List[Optional[float]] = []
        reliable_values: List[bool] = []
        reasons: List[str] = []

        # (_, data_row)에서 필요한 좌표나 ID만 꺼내 다음 계산에 사용.
        for _, data_row in result_data.iterrows():
            confidence = float(data_row.get("confidence", 0.0) or 0.0)
            status = str(data_row.get("ball_status", "Detected"))
            reason = str(data_row.get("cleaning_note", "accepted"))
            is_reliable = True
            jump_distance: Optional[float] = None

            if previous_row is not None:
                frame_gap = int(data_row["frame"]) - int(previous_row["frame"])
                if frame_gap > 0:
                    same_segment = int(data_row.get("ball_segment", -1)) == int(previous_row.get("ball_segment", -2))
                    continuous_gap = frame_gap <= self.config.max_interpolation_gap + 1
                    if same_segment and continuous_gap:
                        jump_distance = float(np.hypot(
                            float(data_row["x_center"]) - float(previous_row["x_center"]),
                            float(data_row["y_center"]) - float(previous_row["y_center"]),
                        ))
                        allowed = strict_step * max(1, min(frame_gap, 2))
                        if jump_distance > allowed:
                            is_reliable = False
                            reason = "warning_large_clean_jump"

            if status == "Predicted" and confidence < 0.02:
                is_reliable = False
                reason = "warning_low_confidence_prediction"

            if status == "Detected" and confidence < 0.01:
                is_reliable = False
                reason = "warning_very_low_confidence_detection"

            jump_distances.append(round(jump_distance, 3) if jump_distance is not None else None)
            reliable_values.append(is_reliable)
            reasons.append(reason)
            previous_row = data_row

        result_data["jump_distance"] = jump_distances
        result_data["is_reliable"] = reliable_values
        result_data["reliability_reason"] = reasons
        return result_data

    # _make_ball_row는 공 좌표나 공 후보를 따로 정리할 때 사용.
    def _make_ball_row(
        self,
        source: Mapping[str, Any],
        frame: Optional[int] = None,
        x_center: Optional[float] = None,
        y_center: Optional[float] = None,
        confidence: Optional[float] = None,
        status: str = "Detected",
        note: str = "accepted",
    ) -> Dict[str, Any]:

        data_row = dict(source)
        data_row["frame"] = int(source["frame"] if frame is None else frame)
        data_row["track_id"] = 0
        data_row["class"] = "ball"
        data_row["confidence"] = round(float(source.get("confidence", 0.0) if confidence is None else confidence), 3)
        data_row["x_center"] = round(float(source["x_center"] if x_center is None else x_center), 2)
        data_row["y_center"] = round(float(source["y_center"] if y_center is None else y_center), 2)
        data_row["ball_status"] = status
        data_row["cleaning_note"] = note
        data_row["raw_x_center"] = source.get("raw_x_center", source.get("x_center"))
        data_row["raw_y_center"] = source.get("raw_y_center", source.get("y_center"))
        data_row["ball_segment"] = int(source.get("ball_segment", 0))
        data_row["bbox_size"] = float(source.get("bbox_size", self.config.default_ball_box_size) or self.config.default_ball_box_size)
        return data_row

    # estimate_velocity 처리용 보조 함수.
    # sorted_points, point_index 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _estimate_velocity(
        self,
        sorted_points: Sequence[Mapping[str, Any]],
        point_index: int,
    ) -> Optional[Tuple[float, float]]:

        if point_index <= 0:
            return None
        current = sorted_points[point_index]
        current_segment = int(current.get("ball_segment", -1))
        # prev_index 항목을 돌면서 조건에 맞는 값만 남김.
        for prev_index in range(point_index - 1, -1, -1):
            previous = sorted_points[prev_index]
            if int(previous.get("ball_segment", -2)) != current_segment:
                continue
            frame_delta = int(current["frame"]) - int(previous["frame"])
            if frame_delta <= 0:
                continue
            vx = (float(current["x_center"]) - float(previous["x_center"])) / frame_delta
            vy = (float(current["y_center"]) - float(previous["y_center"])) / frame_delta
            return vx, vy
        return None

    # _rebuild_ball_bbox에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
    def _rebuild_ball_bbox(self, ball_table: pd.DataFrame) -> pd.DataFrame:

        result_data = ball_table.copy()
        sizes = pd.to_numeric(result_data.get("bbox_size", self.config.default_ball_box_size), errors="coerce").fillna(self.config.default_ball_box_size)
        half = sizes / 2.0
        result_data["x1"] = (result_data["x_center"] - half).round(2)
        result_data["y1"] = (result_data["y_center"] - half).round(2)
        result_data["x2"] = (result_data["x_center"] + half).round(2)
        result_data["y2"] = (result_data["y_center"] + half).round(2)
        result_data["x_center"] = result_data["x_center"].round(2)
        result_data["y_center"] = result_data["y_center"].round(2)
        return result_data

    def _scaled_max_step(self, frame_size: Tuple[int, int]) -> float:

        width, height = frame_size
        scale = max(width, height) / 1920.0
        return float(self.config.max_step_px_per_frame) * max(0.75, scale)

    def _scaled_strict_step(self, frame_size: Tuple[int, int]) -> float:

        width, height = frame_size
        scale = max(width, height) / 1920.0
        return float(self.config.strict_max_step_px_per_frame) * max(0.75, scale)

    # _is_near_frame_edge에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
    def _is_near_frame_edge(self, point: Mapping[str, Any], frame_size: Tuple[int, int]) -> bool:

        width, height = frame_size
        try:
            x = float(point["x_center"])
            y = float(point["y_center"])
        except (KeyError, TypeError, ValueError):
            return True
        margin = float(self.config.edge_margin_px)
        return x <= margin or y <= margin or x >= width - margin or y >= height - margin

    # _context_points_by_frame에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
    def _context_points_by_frame(self, non_ball_data: pd.DataFrame) -> Dict[int, List[Tuple[float, float]]]:
        

        if non_ball_data.empty:
            return {}

        context_df = non_ball_data[non_ball_data.get("class", pd.Series(dtype=str)).isin({"player", "referee", "hoop"})].copy()
        if context_df.empty:
            return {}

        context_df = self._ensure_numeric(context_df, ["frame", "x_center", "y_center"])
        context_df = context_df.dropna(subset=["frame", "x_center", "y_center"])
        grouped: Dict[int, List[Tuple[float, float]]] = {}
        for data_row in context_df.to_dict("records"):
            grouped.setdefault(int(data_row["frame"]), []).append((float(data_row["x_center"]), float(data_row["y_center"])))
        return grouped

    # _player_records_by_frame에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
    def _player_records_by_frame(self, non_ball_data: pd.DataFrame) -> Dict[int, List[Dict[str, Any]]]:

        if non_ball_data.empty or "class" not in non_ball_data.columns:
            return {}

        player_table = non_ball_data[non_ball_data["class"].isin({"player", "person"})].copy()
        if player_table.empty:
            return {}

        numeric_columns = ["frame", "track_id", "x_center", "y_center", "x1", "y1", "x2", "y2"]
        player_table = self._ensure_numeric(player_table, numeric_columns)
        player_table = player_table.dropna(subset=["frame", "x1", "y1", "x2", "y2"])

        grouped: Dict[int, List[Dict[str, Any]]] = {}
        for data_row in player_table.to_dict("records"):
            grouped.setdefault(int(data_row["frame"]), []).append(data_row)
        return grouped

    # _nearest_context_distance는 두 위치가 얼마나 가까운지 판단할 때 사용.
    def _nearest_context_distance(
        self,
        point: Mapping[str, Any],
        context_by_frame: Mapping[int, Sequence[Tuple[float, float]]],
    ) -> float:

        try:
            frame = int(point["frame"])
            x = float(point["x_center"])
            y = float(point["y_center"])
        except (KeyError, TypeError, ValueError):
            return float("inf")

        candidates: List[Tuple[float, float]] = []
        for offset in (-1, 0, 1):
            candidates.extend(context_by_frame.get(frame + offset, []))
        if not candidates:
            return float("inf")
        return min(float(np.hypot(x - cx, y - cy)) for cx, cy in candidates)

    # 선수 후보나 선수 위치를 정리하는 보조 함수.
    # point, player_by_frame 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def _player_body_overlap(
        self,
        point: Mapping[str, Any],
        player_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    ) -> Optional[Dict[str, Any]]:

        try:
            frame = int(point["frame"])
            x = float(point["x_center"])
            y = float(point["y_center"])
        except (KeyError, TypeError, ValueError):
            return None

        best: Optional[Dict[str, Any]] = None
        best_center_distance = float("inf")
        # 프레임을 돌면서 탐지 결과와 누적값을 계속 갱신함.
        for player in player_by_frame.get(frame, []):
            try:
                x1 = float(player["x1"])
                y1 = float(player["y1"])
                x2 = float(player["x2"])
                y2 = float(player["y2"])
            except (KeyError, TypeError, ValueError):
                continue

            left, right = min(x1, x2), max(x1, x2)
            top, bottom = min(y1, y2), max(y1, y2)
            width = max(1.0, right - left)
            height = max(1.0, bottom - top)
            if not (left <= x <= right and top <= y <= bottom):
                continue

            rel_x = (x - left) / width
            rel_y = (y - top) / height
            torso = 0.08 <= rel_y <= 0.72
            core = 0.12 <= rel_x <= 0.88 and 0.12 <= rel_y <= 0.68
            region = "core" if core else ("lower" if rel_y > 0.68 else "edge")
            player_center_x = float(player.get("x_center", (left + right) / 2.0))
            player_center_y = float(player.get("y_center", (top + bottom) / 2.0))
            if not np.isfinite(player_center_x):
                player_center_x = (left + right) / 2.0
            if not np.isfinite(player_center_y):
                player_center_y = (top + bottom) / 2.0
            center_distance = float(np.hypot(x - player_center_x, y - player_center_y))
            if center_distance >= best_center_distance:
                continue

            track_id = player.get("track_id")
            best_center_distance = center_distance
            best = {
                "track_id": int(track_id) if pd.notna(track_id) else None,
                "rel_x": float(rel_x),
                "rel_y": float(rel_y),
                "core": bool(core),
                "torso": bool(torso),
                "region": region,
            }

        return best

    # _apply_player_body_metadata는 선수 row나 선수 위치를 따로 정리하는 부분.
    def _apply_player_body_metadata(self, point: Dict[str, Any], overlap: Optional[Mapping[str, Any]]) -> None:

        if overlap is None:
            point["player_body_overlap"] = False
            point["player_body_track_id"] = None
            point["player_body_region"] = None
            point["player_body_rel_x"] = None
            point["player_body_rel_y"] = None
            point["player_body_core"] = False
            point["player_body_torso"] = False
            return

        point["player_body_overlap"] = True
        point["player_body_track_id"] = overlap.get("track_id")
        point["player_body_region"] = overlap.get("region")
        point["player_body_rel_x"] = round(float(overlap.get("rel_x", 0.0)), 3)
        point["player_body_rel_y"] = round(float(overlap.get("rel_y", 0.0)), 3)
        point["player_body_core"] = bool(overlap.get("core"))
        point["player_body_torso"] = bool(overlap.get("torso"))

    # _is_low_confidence_player_body_point는 선수 row나 선수 위치를 따로 정리하는 부분.
    def _is_low_confidence_player_body_point(self, point: Mapping[str, Any]) -> bool:

        if not bool(point.get("player_body_core")) and not bool(point.get("player_body_torso")):
            return False
        confidence = float(point.get("confidence", 0.0) or 0.0)
        if bool(point.get("player_body_core")):
            return confidence < float(self.config.player_body_confidence_threshold)
        return confidence < 0.20

    # _remove_player_body_locked_runs는 선수 row나 선수 위치를 따로 정리하는 부분.
    def _remove_player_body_locked_runs(
        self,
        points: Sequence[Mapping[str, Any]],
    ) -> List[Dict[str, Any]]:

        if not points:
            return []

        min_run = max(3, int(self.config.player_body_lock_min_frames))
        kept: List[Dict[str, Any]] = []
        run: List[Dict[str, Any]] = []

        # flush_run 처리용 보조 함수.
        # 입력값을 정리해서 바로 다음 계산에 넘기는 흐름.
        def flush_run() -> None:
            nonlocal run
            if not run:
                return
            confidences = [float(point.get("confidence", 0.0) or 0.0) for point in run]
            median_confidence = float(np.median(confidences)) if confidences else 0.0
            max_confidence = max(confidences) if confidences else 0.0
            drop_whole_run = (
                len(run) >= min_run
                and (
                    median_confidence < float(self.config.player_body_confidence_threshold)
                    or max_confidence < 0.82
                )
            )
            if not drop_whole_run:
                for point in run:
                    if not self._is_low_confidence_player_body_point(point):
                        kept.append(dict(point))
            run = []

        previous_frame: Optional[int] = None
        previous_segment: Optional[int] = None
        for point in sorted(points, key=lambda item: int(item["frame"])):
            frame = int(point["frame"])
            segment = int(point.get("ball_segment", 0))
            is_torso = bool(point.get("player_body_torso"))
            contiguous = (
                is_torso
                and previous_frame is not None
                and previous_segment == segment
                and 0 < frame - previous_frame <= self.config.max_interpolation_gap + 1
            )

            if not is_torso:
                flush_run()
                kept.append(dict(point))
            elif run and contiguous:
                run.append(dict(point))
            else:
                flush_run()
                run.append(dict(point))

            previous_frame = frame
            previous_segment = segment

        flush_run()
        return sorted(kept, key=lambda item: int(item["frame"]))

    # _remove_player_body_locked_timeline는 선수 row나 선수 위치를 따로 정리하는 부분.
    def _remove_player_body_locked_timeline(self, timeline: pd.DataFrame) -> pd.DataFrame:

        if timeline.empty:
            return timeline

        data_rows = self._remove_player_body_locked_runs(timeline.to_dict("records"))
        if not data_rows:
            return timeline.iloc[0:0].copy()
        return pd.DataFrame(data_rows).sort_values("frame").reset_index(drop=True)

    # _bbox_size_series에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
    def _bbox_size_series(self, data_frame: pd.DataFrame) -> pd.Series:

        if {"x1", "y1", "x2", "y2"}.issubset(data_frame.columns):
            width = (pd.to_numeric(data_frame["x2"], errors="coerce") - pd.to_numeric(data_frame["x1"], errors="coerce")).abs()
            height = (pd.to_numeric(data_frame["y2"], errors="coerce") - pd.to_numeric(data_frame["y1"], errors="coerce")).abs()
            size = pd.concat([width, height], axis=1).max(axis=1)
            return size.fillna(self.config.default_ball_box_size).clip(lower=8.0, upper=40.0)
        return pd.Series(self.config.default_ball_box_size, index=data_frame.index)

    # 입력 row를 정리한 뒤 필요한 컬럼 순서로 맞춰서 CSV/표 형태로 넘김.
    def _normalize_output_columns(self, data_frame: pd.DataFrame) -> pd.DataFrame:

        if data_frame.empty:
            return data_frame
        for column in BASE_TRACKING_COLUMNS:
            if column not in data_frame.columns:
                data_frame[column] = None
        preferred = BASE_TRACKING_COLUMNS + [
            "ball_status",
            "cleaning_note",
            "raw_x_center",
            "raw_y_center",
            "ball_segment",
            "is_reliable",
            "reliability_reason",
            "jump_distance",
            "context_distance",
            "player_body_overlap",
            "player_body_track_id",
            "player_body_region",
            "player_body_rel_x",
            "player_body_rel_y",
            "player_body_torso",
            "motion_distance",
            "frame_gap_from_previous_ball",
        ]
        ordered = [column for column in preferred if column in data_frame.columns]
        remaining = [column for column in data_frame.columns if column not in ordered]
        result_data = data_frame[ordered + remaining].copy()
        return result_data.sort_values(["frame", "class", "track_id"]).reset_index(drop=True)


    def _ball_rows(self, data_frame: pd.DataFrame) -> pd.DataFrame:

        if data_frame.empty or "class" not in data_frame.columns:
            return data_frame.iloc[0:0].copy()
        return data_frame[data_frame["class"].isin(BALL_CLASS_NAMES)].copy()

    def _ensure_numeric(self, data_frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:

        result_data = data_frame.copy()
        for column in columns:
            if column in result_data.columns:
                result_data[column] = pd.to_numeric(result_data[column], errors="coerce")
        return result_data

    def _write_csv(self, data_frame: pd.DataFrame, output_path: str) -> None:

        output = Path(output_path)
        output.parent.mkdir(parents=True, exist_ok=True)
        data_frame.to_csv(output, index=False, encoding="utf-8-sig")


# 공 좌표가 얼마나 잘 이어졌는지 간단 리포트 저장.
# 공 좌표 품질을 한 줄 요약 CSV로 저장.
def write_ball_quality_report(
    cleaned_ball_data: pd.DataFrame,
    total_frames: Optional[int],
    output_path: str,
) -> pd.DataFrame:

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)

    ball_table = cleaned_ball_data[cleaned_ball_data.get("class", pd.Series(dtype=str)).eq("ball")].copy()
    frame_total = int(total_frames or 0)
    if ball_table.empty:
        report_df = pd.DataFrame(
            [
                {
                    "total_frames": frame_total,
                    "ball_rows": 0,
                    "ball_detected_frames": 0,
                    "ball_frame_coverage_pct": 0.0,
                    "reliable_ball_rows": 0,
                    "unreliable_ball_rows": 0,
                    "reliable_pct": 0.0,
                    "max_jump_distance": None,
                    "jump_gt_120_count": 0,
                    "jump_gt_180_count": 0,
                }
            ]
        )
        report_df.to_csv(output, index=False, encoding="utf-8-sig")
        return report_df

    frames = pd.to_numeric(ball_table.get("frame"), errors="coerce").dropna()
    jumps = pd.to_numeric(ball_table.get("jump_distance"), errors="coerce").dropna()
    reliable = ball_table.get("is_reliable", pd.Series([True] * len(ball_table), index=ball_table.index)).fillna(True).astype(bool)
    statuses = ball_table.get("ball_status", pd.Series(dtype=str)).fillna("Unknown").astype(str)

    coverage = (frames.nunique() / frame_total * 100.0) if frame_total > 0 else 0.0
    report: Dict[str, Any] = {
        "total_frames": frame_total,
        "ball_rows": int(len(ball_table)),
        "ball_detected_frames": int(frames.nunique()),
        "ball_frame_coverage_pct": round(coverage, 2),
        "reliable_ball_rows": int(reliable.sum()),
        "unreliable_ball_rows": int((~reliable).sum()),
        "reliable_pct": round(float(reliable.mean() * 100.0), 2) if len(reliable) > 0 else 0.0,
        "max_jump_distance": round(float(jumps.max()), 3) if not jumps.empty else None,
        "jump_gt_120_count": int(jumps.gt(120.0).sum()) if not jumps.empty else 0,
        "jump_gt_180_count": int(jumps.gt(180.0).sum()) if not jumps.empty else 0,
    }
    for status, count in statuses.value_counts().items():
        report[f"status_{status}"] = int(count)

    report_df = pd.DataFrame([report])
    report_df.to_csv(output, index=False, encoding="utf-8-sig")
    return report_df


# _safe_output_prefix는 값 형식을 뒤쪽 계산에서 쓰기 좋게 맞춤.
def _safe_output_prefix(prefix: str) -> str:

    if not prefix:
        return ""
    safe = "".join(char if char.isalnum() or char in {"_", "-", "."} else "_" for char in str(prefix))
    return safe if safe.endswith("_") else f"{safe}_"




## Core 5. 이벤트 스탯 계산

팀 구분, 공 소유, 슛/패스/리바운드 같은 이벤트를 대략 계산하는 부분.
정확한 경기 기록이라기보다는 탐지 결과에서 뽑은 추정값으로 보면 됨.


In [5]:

# - 모델이 팀을 직접 구분하지 않아서 유니폼 색과 위치로 team_id를 추정.
# - 공 소유자는 공 중심과 가장 가까운 선수 bbox로 대략 잡음.
# - 슛은 rim 근처 공 궤적을 먼저 보고, 패스/스틸/리바운드는 소유권 변화로 봄.
# - 결과는 공식 기록이 아니라 탐지 기반 추정값이라 오차가 있을 수 있음.
# - event row와 30초 누적 row를 같이 만들어서 뒤쪽 표에서 나눠 사용.

# Core 5. 이벤트 스탯 대략 계산

# 정제된 tracking 결과를 가지고 팀 정보와 이벤트 추정값을 붙임.
# 현재 모델은 선수를 팀별 클래스가 아니라 전부 player로 잡는다.
# 그래서 유니폼 색과 위치를 이용해 team_1, team_2를 추정하고, 정리된 객체
# 좌표를 바탕으로 슛, 패스, 스틸/블락, 파울, 리바운드, 볼점유율을 추정함.


from __future__ import annotations

from dataclasses import dataclass
import math
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd


PLAYER_CLASSES = {"player", "person"}
BALL_CLASSES = {"ball", "sports ball", "basketball", "frisbee"}
HOOP_CLASSES = {"hoop", "rim", "basket", "basketball hoop"}

TEAM_COLUMNS = ["team_id", "team_name", "team_confidence", "team_color_hex"]

EVENT_ORDER = [
    "shot_attempt",
    "shot_made",
    "shot_missed",
    "pass",
    "steal_or_block",
    "foul",
    "rebound",
]

EVENT_COUNT_COLUMNS = [
    f"{event_type}_count" for event_type in EVENT_ORDER
] + [
    "shot_success_rate_pct",
    "possession_frame_count",
    "possession_time_sec",
    "possession_pct",
]

CUMULATIVE_EVENT_INTERVAL_SECONDS = 30.0

EVENT_LABELS_KO = {
    "shot_attempt": "슛시도",
    "shot_made": "슛성공",
    "shot_missed": "슛실패",
    "pass": "패스",
    "steal_or_block": "스틸/블락",
    "foul": "파울",
    "rebound": "리바운드",
}

EVENT_COLUMNS = [
    "row_type",
    "event_id",
    "event_type",
    "event_label_ko",
    "count",
    "frame",
    "time_sec",
    "team_id",
    "team_name",
    *EVENT_COUNT_COLUMNS,
    "player_track_id",
    "secondary_track_id",
    "confidence",
    "details",
]


@dataclass
# 한 선수 track이 어느 팀으로 잡혔는지 담아두는 값.
# 한 선수 track의 팀 정보를 저장하는 작은 구조.
# 설정값이나 상태를 먼저 묶어두고, 아래 메서드들이 그 값을 같이 사용함.
class TeamAssignment:
    team_id: int
    team_name: str
    team_confidence: float
    team_color_hex: str


@dataclass
# 공 소유가 이어진 프레임 구간을 저장.
class PossessionSegment:
    start_frame: int
    end_frame: int
    player_track_id: int
    team_id: str
    team_name: str
    raw_frame_count: int


@dataclass
# 슛으로 볼 만한 프레임 구간 정보를 저장.
class ShotWindow:
    start_frame: int
    end_frame: int
    event_frame: int
    made: bool
    confidence: float
    team_id: str
    team_name: str
    player_track_id: Optional[int]
    min_rim_distance: float


# 선수 위치와 유니폼 색을 보고 team_id를 붙이는 함수.
# 유니폼 색/위치로 선수 team_id를 붙이는 함수.
# 선수/공 위치를 프레임별로 묶고, 가장 가까운 선수나 팀 기준으로 값을 붙임.
def assign_player_teams_from_video(
    tracking_data: pd.DataFrame,
    video_path: str,
    frame_size: Tuple[int, int],
    start_frame: int = 0,
    sample_stride_frames: int = 12,
    max_samples_per_track: int = 48,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    # 선수 위치와 유니폼 색을 샘플링해서 팀 번호를 추정함.

    result_data = _ensure_team_columns(tracking_data.copy())
    summary: Dict[str, Any] = {
        "method": "uniform_color_clustering",
        "team_class_source": "color_clustering",
        "tracks_sampled": 0,
        "teams_found": 0,
        "note": "",
    }

    if result_data.empty or "class" not in result_data.columns:
        summary["note"] = "No rows available for team assignment."
        return result_data, summary

    number_data = _coerce_tracking_numbers(result_data)
    player_table = number_data[
        number_data["class"].astype(str).str.lower().isin(PLAYER_CLASSES)
        & number_data["track_id"].notna()
        & number_data["frame"].notna()
    ].copy()
    player_table = player_table[player_table["track_id"] >= 0]
    if player_table.empty:
        summary["note"] = "No tracked player rows were available."
        return result_data, summary

    track_features = _sample_player_track_colors(
        player_table=player_table,
        video_path=video_path,
        frame_size=frame_size,
        start_frame=start_frame,
        sample_stride_frames=sample_stride_frames,
        max_samples_per_track=max_samples_per_track,
    )
    if len(track_features) < 2 and sample_stride_frames > 1:
        track_features = _sample_player_track_colors(
            player_table=player_table,
            video_path=video_path,
            frame_size=frame_size,
            start_frame=start_frame,
            sample_stride_frames=1,
            max_samples_per_track=max_samples_per_track,
        )
    summary["tracks_sampled"] = len(track_features)
    if len(track_features) < 2:
        summary["note"] = "Not enough sampled player tracks to split two teams."
        summary.update(_assign_player_teams_by_position(result_data, player_table))
        return result_data, summary

    assignments, team_summary = _cluster_track_features(track_features)
    summary.update(team_summary)
    if not assignments:
        summary["note"] = "Player colors were too similar to split teams reliably."
        summary.update(_assign_player_teams_by_position(result_data, player_table))
        return result_data, summary

    _apply_team_assignments(result_data, assignments)
    fallback_summary = _assign_missing_player_teams_by_position(result_data, player_table)
    if fallback_summary.get("fallback_assigned_tracks", 0):
        summary.update(fallback_summary)
    return result_data, summary


# track_id가 비어 있는 선수 row에 임시 id를 채움.
# 비어 있는 track_id를 임시 id로 채워 집계가 끊기지 않게 함.
# tracking_data, frame_size, max_gap_frames 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def assign_fallback_player_track_ids(
    tracking_data: pd.DataFrame,
    frame_size: Tuple[int, int],
    max_gap_frames: int = 12,
    max_match_distance_px: Optional[float] = None,
) -> pd.DataFrame:

    result_data = _coerce_tracking_numbers(tracking_data.copy())
    if result_data.empty or "class" not in result_data.columns:
        return tracking_data

    player_mask = result_data["class"].astype(str).str.lower().isin(PLAYER_CLASSES)
    missing_mask = player_mask & (
        result_data["track_id"].isna() | (result_data["track_id"] < 0)
    )
    if not bool(missing_mask.any()):
        return tracking_data

    existing_ids = result_data.loc[player_mask & result_data["track_id"].notna() & (result_data["track_id"] >= 0), "track_id"]
    next_track_id = int(existing_ids.max()) + 1 if not existing_ids.empty else 1
    scale = max(0.75, min(1.6, max(frame_size) / 1280.0))
    max_distance = float(max_match_distance_px) if max_match_distance_px is not None else 175.0 * scale
    max_gap = max(1, int(max_gap_frames))

    active_tracks: Dict[int, Dict[str, Any]] = {}
    # 프레임을 돌면서 탐지 결과와 누적값을 계속 갱신함.
    for frame in sorted(result_data.loc[player_mask, "frame"].dropna().astype(int).unique()):
        frame_indices = result_data.index[player_mask & result_data["frame"].eq(frame)].tolist()
        missing_indices = [
            index
            for index in frame_indices
            if pd.isna(result_data.at[index, "track_id"]) or float(result_data.at[index, "track_id"]) < 0
        ]

        stale_ids = [
            track_id
            for track_id, state in active_tracks.items()
            if frame - int(state["frame"]) > max_gap
        ]
        for track_id in stale_ids:
            active_tracks.pop(track_id, None)

        candidates: List[Tuple[float, int, int]] = []
        for index in missing_indices:
            center = _center(result_data.loc[index].to_dict())
            if center is None:
                continue
            for track_id, state in active_tracks.items():
                distance = math.hypot(center[0] - state["center"][0], center[1] - state["center"][1])
                if distance <= max_distance:
                    candidates.append((distance, index, track_id))

        assigned_indices: set[int] = set()
        assigned_tracks: set[int] = set()
        for _, index, track_id in sorted(candidates, key=lambda item: item[0]):
            if index in assigned_indices or track_id in assigned_tracks:
                continue
            result_data.at[index, "track_id"] = track_id
            center = _center(result_data.loc[index].to_dict())
            if center is not None:
                active_tracks[track_id] = {"center": center, "frame": frame}
            assigned_indices.add(index)
            assigned_tracks.add(track_id)

        for index in missing_indices:
            if index in assigned_indices:
                continue
            center = _center(result_data.loc[index].to_dict())
            if center is None:
                continue
            track_id = next_track_id
            next_track_id += 1
            result_data.at[index, "track_id"] = track_id
            active_tracks[track_id] = {"center": center, "frame": frame}

        for index in frame_indices:
            if index in missing_indices:
                continue
            try:
                track_id = int(float(result_data.at[index, "track_id"]))
            except (TypeError, ValueError):
                continue
            if track_id < 0:
                continue
            center = _center(result_data.loc[index].to_dict())
            if center is not None:
                active_tracks[track_id] = {"center": center, "frame": frame}

    return result_data


# 프레임별 이벤트 결과를 CSV로 저장.
# 프레임별 이벤트 결과를 event.csv로 저장.
# 먼저 후보 프레임 구간을 만들고, 조건을 통과한 경우에만 이벤트 row로 추가함.
def write_event_measurements(
    tracking_data: pd.DataFrame,
    output_csv_path: str,
    fps: float,
    frame_size: Tuple[int, int],
    video_path: str = "",
    cumulative_interval_seconds: float = CUMULATIVE_EVENT_INTERVAL_SECONDS,
) -> pd.DataFrame:
    # 프레임별 이벤트 누적 결과를 CSV로 저장.

    event_data_table = measure_basketball_events(
        tracking_data=tracking_data,
        fps=fps,
        frame_size=frame_size,
        video_path=video_path,
        cumulative_interval_seconds=cumulative_interval_seconds,
    )
    output = Path(output_csv_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    event_data_table.to_csv(output, index=False, encoding="utf-8-sig")
    return event_data_table


# 팀별/전체 이벤트 요약 CSV를 저장.
# 팀별/전체 요약을 event_summary.csv로 저장.
def write_event_summary(
    tracking_data: pd.DataFrame,
    output_csv_path: str,
    fps: float,
    frame_size: Tuple[int, int],
    video_path: str = "",
) -> pd.DataFrame:
    # 전체 이벤트 요약표를 CSV로 저장.

    summary_data_table = measure_basketball_event_summary(
        tracking_data=tracking_data,
        fps=fps,
        frame_size=frame_size,
        video_path=video_path,
    )
    output = Path(output_csv_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    summary_data_table.to_csv(output, index=False, encoding="utf-8-sig")
    return summary_data_table


# tracking 결과에서 이벤트 row를 만드는 시작점.
# 공 후보를 모은 뒤 confidence, 위치, 이전 이동 방향을 같이 보고 남길 후보를 고름.
def measure_basketball_events(
    tracking_data: pd.DataFrame,
    fps: float,
    frame_size: Tuple[int, int],
    video_path: str = "",
    cumulative_interval_seconds: float = CUMULATIVE_EVENT_INTERVAL_SECONDS,
) -> pd.DataFrame:
    # tracking 결과에서 30초 단위 누적 이벤트 표를 만듦.

    event_rows, fps_value, max_frame, known_teams, possession_by_frame = _measure_basketball_event_context(
        tracking_data=tracking_data,
        fps=fps,
        frame_size=frame_size,
        video_path=video_path,
    )
    return _cumulative_event_output_dataframe(
        event_rows,
        fps_value,
        max_frame=max_frame,
        known_teams=known_teams,
        possession_by_frame=possession_by_frame,
        interval_seconds=cumulative_interval_seconds,
    )


# 이벤트 row를 팀별 요약 형태로 다시 묶음.
# event.csv 내용을 요약표 형태로 다시 묶음.
def measure_basketball_event_summary(
    tracking_data: pd.DataFrame,
    fps: float,
    frame_size: Tuple[int, int],
    video_path: str = "",
) -> pd.DataFrame:

    event_rows, fps_value, _max_frame, known_teams, possession_by_frame = _measure_basketball_event_context(
        tracking_data=tracking_data,
        fps=fps,
        frame_size=frame_size,
        video_path=video_path,
    )
    possession_stats = _build_possession_stats(possession_by_frame, fps_value)
    return _event_output_dataframe(
        event_rows,
        fps_value,
        known_teams=known_teams,
        possession_stats=possession_stats,
    )


# 이벤트 계산에 필요한 팀, 소유권, 슛 구간을 미리 준비.
def _measure_basketball_event_context(
    tracking_data: pd.DataFrame,
    fps: float,
    frame_size: Tuple[int, int],
    video_path: str = "",
) -> Tuple[List[Dict[str, Any]], float, int, List[Dict[str, str]], Dict[int, Mapping[str, Any]]]:
    del video_path
    fps_value = float(fps) if fps and fps > 0 else 30.0
    data_frame = _coerce_tracking_numbers(_ensure_team_columns(tracking_data.copy()))
    data_frame = _filter_event_safe_tracking_rows(data_frame)
    max_frame = _max_tracking_frame(data_frame)
    if data_frame.empty:
        return [], fps_value, max_frame, [], {}

    # 이벤트 계산에 필요한 팀 정보, 프레임별 상태, 소유권 구간을 먼저 한 번에 만듦.
    known_teams = _known_teams_from_tracking(data_frame)
    frame_state = _build_frame_state(data_frame)
    possession_by_frame = _infer_frame_possessions(frame_state, frame_size)
    possession_segments = _build_possession_segments(possession_by_frame, fps_value)

    event_rows: List[Dict[str, Any]] = []
    # 슛은 림 근처 공 궤적을 먼저 찾고, 이후 패스/스틸/리바운드 판단에서 참고함.
    shot_windows = _detect_shot_windows(
        frame_state=frame_state,
        possession_segments=possession_segments,
        fps=fps_value,
        frame_size=frame_size,
    )
    # shot 항목을 돌면서 조건에 맞는 값만 남김.
    for shot in shot_windows:
        actor = "" if shot.player_track_id is None else str(shot.player_track_id)
        event_rows.append(
            _make_event_row(
                event_type="shot_attempt",
                frame=shot.event_frame,
                fps=fps_value,
                team_id=shot.team_id,
                team_name=shot.team_name,
                player_track_id=actor,
                confidence=shot.confidence,
                details=f"near_rim_distance_px={shot.min_rim_distance:.1f}",
            )
        )
        event_rows.append(
            _make_event_row(
                event_type="shot_made" if shot.made else "shot_missed",
                frame=shot.event_frame,
                fps=fps_value,
                team_id=shot.team_id,
                team_name=shot.team_name,
                player_track_id=actor,
                confidence=max(0.25, shot.confidence - 0.12),
                details="ball_crossed_rim_zone" if shot.made else "near_rim_no_make_detected",
            )
        )

    # 소유권이 바뀌는 지점을 보면서 같은 팀이면 패스, 다른 팀이면 스틸/블락 후보로 본다.
    event_rows.extend(
        _detect_passes_and_steals(
            possession_segments=possession_segments,
            fps=fps_value,
            shot_windows=shot_windows,
        )
    )
    event_rows.extend(
        _detect_rebounds(
            possession_segments=possession_segments,
            shot_windows=shot_windows,
            fps=fps_value,
        )
    )
    event_rows.extend(
        _detect_contact_fouls(
            frame_state=frame_state,
            possession_segments=possession_segments,
            fps=fps_value,
            frame_size=frame_size,
            shot_windows=shot_windows,
        )
    )

    return event_rows, fps_value, max_frame, known_teams, possession_by_frame


# _ensure_team_columns에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _ensure_team_columns(data_frame: pd.DataFrame) -> pd.DataFrame:
    for column in TEAM_COLUMNS:
        if column not in data_frame.columns:
            data_frame[column] = ""
    return data_frame


# _coerce_tracking_numbers는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _coerce_tracking_numbers(data_frame: pd.DataFrame) -> pd.DataFrame:
    numeric_columns = [
        "frame",
        "track_id",
        "confidence",
        "x_center",
        "y_center",
        "x1",
        "y1",
        "x2",
        "y2",
        "team_confidence",
    ]
    for column in numeric_columns:
        if column in data_frame.columns:
            data_frame[column] = pd.to_numeric(data_frame[column], errors="coerce")
    return data_frame


# 이벤트 결과를 만들거나 표 형태로 바꾸는 부분.
# 이벤트 row나 요약표를 만드는 함수.
def _filter_event_safe_tracking_rows(data_frame: pd.DataFrame) -> pd.DataFrame:

    if data_frame.empty or "class" not in data_frame.columns:
        return data_frame

    result_data = data_frame.copy()
    class_values = result_data["class"].astype(str).str.lower()
    ball_mask = class_values.isin(BALL_CLASSES)
    if not bool(ball_mask.any()):
        return result_data

    reliable = pd.Series(True, index=result_data.index)
    if "is_reliable" in result_data.columns:
        reliability_text = result_data["is_reliable"].astype(str).str.strip().str.lower()
        reliable &= ~reliability_text.isin({"false", "0", "no", "nan", "none"})

    if "reliability_reason" in result_data.columns:
        reasons = result_data["reliability_reason"].astype(str).str.strip().str.lower()
        unreliable_reason = reasons.str.contains(
            "removed|warning_large|warning_low|rim|edge|contextless",
            regex=True,
            na=False,
        )
        reliable &= ~unreliable_reason

    if "ball_status" in result_data.columns:
        statuses = result_data["ball_status"].astype(str).str.strip().str.lower()
        if "confidence" in result_data.columns:
            confidence_values = result_data["confidence"]
        else:
            confidence_values = pd.Series(0.0, index=result_data.index)
        confidences = pd.to_numeric(confidence_values, errors="coerce").fillna(0.0)
        low_confidence_prediction = statuses.eq("predicted") & (confidences < 0.10)
        reliable &= ~low_confidence_prediction

    return result_data[~ball_mask | reliable].copy()


# _max_tracking_frame에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _max_tracking_frame(data_frame: pd.DataFrame) -> int:
    if data_frame.empty or "frame" not in data_frame.columns:
        return 0
    frames = pd.to_numeric(data_frame["frame"], errors="coerce").dropna()
    if frames.empty:
        return 0
    return max(0, int(frames.max()))


# 선수 후보나 선수 위치를 정리하는 보조 함수.
# player_table, video_path, frame_size 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _sample_player_track_colors(
    player_table: pd.DataFrame,
    video_path: str,
    frame_size: Tuple[int, int],
    start_frame: int,
    sample_stride_frames: int,
    max_samples_per_track: int,
) -> Dict[int, np.ndarray]:
    stride = max(1, int(sample_stride_frames))
    sample_df = player_table[player_table["frame"].astype(int) % stride == 0].copy()
    if sample_df.empty:
        sample_df = player_table.copy()

    rows_by_frame: Dict[int, List[Mapping[str, Any]]] = {}
    for data_row in sample_df.sort_values(["frame", "track_id"]).to_dict("records"):
        frame = int(data_row["frame"])
        rows_by_frame.setdefault(frame, []).append(data_row)

    if not rows_by_frame:
        return {}

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        return {}
    if start_frame > 0:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))

    max_relative_frame = max(rows_by_frame)
    samples: Dict[int, List[np.ndarray]] = {}
    frame_number = 0
    while capture.isOpened() and frame_number <= max_relative_frame:
        has_frame, frame = capture.read()
        if not has_frame:
            break

        for data_row in rows_by_frame.get(frame_number, []):
            track_id = int(data_row["track_id"])
            track_samples = samples.setdefault(track_id, [])
            if len(track_samples) >= max(1, int(max_samples_per_track)):
                continue

            feature = _player_uniform_feature(frame, data_row, frame_size)
            if feature is not None:
                track_samples.append(feature)

        frame_number += 1

    capture.release()

    track_features: Dict[int, np.ndarray] = {}
    for track_id, track_samples in samples.items():
        if not track_samples:
            continue
        stacked = np.vstack(track_samples)
        track_features[track_id] = np.median(stacked, axis=0)
    return track_features


# frame, data_row, frame_size 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _player_uniform_feature(
    frame: Any,
    data_row: Mapping[str, Any],
    frame_size: Tuple[int, int],
) -> Optional[np.ndarray]:
    bbox = _safe_bbox(data_row, frame_size)
    if bbox is None:
        return None
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    if width < 12 or height < 24:
        return None

    torso_x1 = x1 + int(width * 0.18)
    torso_x2 = x2 - int(width * 0.18)
    torso_y1 = y1 + int(height * 0.18)
    torso_y2 = y1 + int(height * 0.68)
    if torso_x2 <= torso_x1 or torso_y2 <= torso_y1:
        return None

    crop = frame[torso_y1:torso_y2, torso_x1:torso_x2]
    if crop.size == 0:
        return None

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)
    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]
    mask = (value > 28) & (value < 248) & (saturation > 18)
    if int(mask.sum()) < 24:
        mask = value > 28
    if int(mask.sum()) < 24:
        return None

    pixels = lab[mask]
    median_lab = np.median(pixels.astype(float), axis=0)
    return median_lab


# cluster_track_features 처리용 보조 함수.
# track_features 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _cluster_track_features(
    track_features: Mapping[int, np.ndarray],
) -> Tuple[Dict[int, TeamAssignment], Dict[str, Any]]:
    track_ids = sorted(track_features)
    features = np.vstack([track_features[track_id] for track_id in track_ids]).astype(float)
    if len(track_ids) < 2:
        return {}, {"teams_found": 0}

    distance_matrix = np.linalg.norm(features[:, None, :] - features[None, :, :], axis=2)
    first, second = np.unravel_index(int(np.argmax(distance_matrix)), distance_matrix.shape)
    max_distance = float(distance_matrix[first, second])
    if max_distance < 10.0:
        return {}, {"teams_found": 0, "team_color_distance": round(max_distance, 3)}

    centers = np.vstack([features[first], features[second]]).astype(float)
    labels = np.zeros(len(features), dtype=int)
    # _ 항목을 돌면서 조건에 맞는 값만 남김.
    for _ in range(30):
        distances = np.linalg.norm(features[:, None, :] - centers[None, :, :], axis=2)
        new_labels = np.argmin(distances, axis=1)
        if np.array_equal(new_labels, labels):
            break
        labels = new_labels
        for label in (0, 1):
            members = features[labels == label]
            if len(members) > 0:
                centers[label] = np.mean(members, axis=0)

    # 팀 번호를 일정하게 유지하기 위해 더 밝은 유니폼 그룹을 team 1로 둔다
    light_order = sorted((0, 1), key=lambda label: centers[label][0], reverse=True)
    label_to_team_id = {light_order[0]: 1, light_order[1]: 2}
    team_names = {1: "team_1", 2: "team_2"}
    team_colors = {
        team_id: _lab_to_hex(centers[label])
        for label, team_id in label_to_team_id.items()
    }

    assignments: Dict[int, TeamAssignment] = {}
    # (index, track_id) 항목을 돌면서 조건에 맞는 값만 남김.
    for index, track_id in enumerate(track_ids):
        label = int(labels[index])
        team_id = label_to_team_id[label]
        distances = np.linalg.norm(features[index][None, :] - centers, axis=1)
        best_distance = float(distances[label])
        other_distance = float(distances[1 - label])
        if other_distance <= 1e-6:
            confidence = 0.5
        else:
            margin = max(0.0, other_distance - best_distance) / other_distance
            confidence = 0.5 + 0.49 * min(1.0, margin)
        assignments[track_id] = TeamAssignment(
            team_id=team_id,
            team_name=team_names[team_id],
            team_confidence=round(confidence, 3),
            team_color_hex=team_colors[team_id],
        )

    return assignments, {
        "teams_found": 2,
        "team_color_distance": round(max_distance, 3),
        "team_1_color_hex": team_colors[1],
        "team_2_color_hex": team_colors[2],
    }


# _lab_to_hex는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _lab_to_hex(lab_color: Sequence[float]) -> str:
    lab_pixel = np.array([[np.clip(lab_color, 0, 255)]], dtype=np.uint8)
    bgr = cv2.cvtColor(lab_pixel, cv2.COLOR_LAB2BGR)[0, 0]
    return f"#{int(bgr[2]):02x}{int(bgr[1]):02x}{int(bgr[0]):02x}"


# _apply_team_assignments에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _apply_team_assignments(
    data_frame: pd.DataFrame,
    assignments: Mapping[int, TeamAssignment],
) -> None:
    player_mask = data_frame["class"].astype(str).str.lower().isin(PLAYER_CLASSES)
    # 프레임을 하나씩 보면서 탐지 결과와 시간 기준값을 같이 갱신.
    for index, data_row in data_frame[player_mask].iterrows():
        try:
            track_id = int(float(data_row.get("track_id")))
        except (TypeError, ValueError):
            continue
        assignment = assignments.get(track_id)
        if assignment is None:
            data_frame.at[index, "team_name"] = "unknown"
            continue
        data_frame.at[index, "team_id"] = str(assignment.team_id)
        data_frame.at[index, "team_name"] = assignment.team_name
        data_frame.at[index, "team_confidence"] = assignment.team_confidence
        data_frame.at[index, "team_color_hex"] = assignment.team_color_hex


# 팀 구분이나 팀별 집계에 쓰는 부분.
# 팀 구분이나 팀별 결과를 맞추는 함수.
def _assign_player_teams_by_position(
    data_frame: pd.DataFrame,
    player_table: pd.DataFrame,
) -> Dict[str, Any]:

    track_positions = _player_track_x_positions(player_table)
    if len(track_positions) < 2:
        return {
            "teams_found": 0,
            "fallback_assigned_tracks": 0,
            "fallback_method": "position_split_unavailable",
        }

    assignments = _position_split_assignments(track_positions, set(track_positions))
    _apply_team_assignments(data_frame, assignments)
    return {
        "teams_found": 2,
        "fallback_assigned_tracks": len(assignments),
        "fallback_method": "median_x_position_split",
        "note": "Uniform color split unavailable; used player track x-position fallback.",
    }


# _assign_missing_player_teams_by_position는 선수 row나 선수 위치를 따로 정리하는 부분.
def _assign_missing_player_teams_by_position(
    data_frame: pd.DataFrame,
    player_table: pd.DataFrame,
) -> Dict[str, Any]:

    track_positions = _player_track_x_positions(player_table)
    if len(track_positions) < 2:
        return {"fallback_assigned_tracks": 0}

    assigned = _assigned_team_by_track(data_frame)
    missing_tracks = set(track_positions) - set(assigned)
    if not missing_tracks:
        return {"fallback_assigned_tracks": 0}

    if {"1", "2"}.issubset(set(assigned.values())):
        assignments = _nearest_known_team_assignments(track_positions, assigned, missing_tracks)
        method = "nearest_color_assigned_track"
    else:
        assignments = _position_split_assignments(track_positions, missing_tracks)
        method = "median_x_position_split"

    _apply_team_assignments(data_frame, assignments)
    return {
        "fallback_assigned_tracks": len(assignments),
        "fallback_method": method,
    }


# _player_track_x_positions는 선수 row나 선수 위치를 따로 정리하는 부분.
def _player_track_x_positions(player_table: pd.DataFrame) -> Dict[int, float]:
    if player_table.empty or "track_id" not in player_table.columns or "x_center" not in player_table.columns:
        return {}

    output: Dict[int, float] = {}
    number_data = _coerce_tracking_numbers(player_table.copy())
    # (track_id, track_df) 항목을 돌면서 조건에 맞는 값만 남김.
    for track_id, track_df in number_data.dropna(subset=["track_id", "x_center"]).groupby("track_id"):
        try:
            clean_track_id = int(float(track_id))
        except (TypeError, ValueError):
            continue
        if clean_track_id < 0:
            continue
        x_values = pd.to_numeric(track_df["x_center"], errors="coerce").dropna()
        if x_values.empty:
            continue
        output[clean_track_id] = float(x_values.median())
    return output


# _assigned_team_by_track에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _assigned_team_by_track(data_frame: pd.DataFrame) -> Dict[int, str]:
    if data_frame.empty or "team_id" not in data_frame.columns:
        return {}

    player_mask = data_frame["class"].astype(str).str.lower().isin(PLAYER_CLASSES)
    assigned: Dict[int, str] = {}
    for data_row in data_frame[player_mask].to_dict("records"):
        team_id = _clean_team_id(data_row.get("team_id"))
        if team_id not in {"1", "2"}:
            continue
        try:
            track_id = int(float(data_row.get("track_id")))
        except (TypeError, ValueError):
            continue
        assigned[track_id] = team_id
    return assigned


# position_split_assignments 처리용 보조 함수.
# track_positions, target_tracks 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _position_split_assignments(
    track_positions: Mapping[int, float],
    target_tracks: set[int],
) -> Dict[int, TeamAssignment]:
    sorted_tracks = sorted(track_positions.items(), key=lambda item: (item[1], item[0]))
    split_index = max(1, len(sorted_tracks) // 2)
    rank_by_track = {track_id: rank for rank, (track_id, _) in enumerate(sorted_tracks)}

    assignments: Dict[int, TeamAssignment] = {}
    # track_id 항목을 돌면서 조건에 맞는 값만 남김.
    for track_id in sorted(target_tracks):
        if track_id not in rank_by_track:
            continue
        team_id = 1 if rank_by_track[track_id] < split_index else 2
        assignments[track_id] = TeamAssignment(
            team_id=team_id,
            team_name=f"team_{team_id}",
            team_confidence=0.35,
            team_color_hex="",
        )
    return assignments


# _nearest_known_team_assignments는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _nearest_known_team_assignments(
    track_positions: Mapping[int, float],
    assigned: Mapping[int, str],
    target_tracks: set[int],
) -> Dict[int, TeamAssignment]:
    known_positions = [
        (track_positions[track_id], team_id)
        for track_id, team_id in assigned.items()
        if track_id in track_positions and team_id in {"1", "2"}
    ]
    assignments: Dict[int, TeamAssignment] = {}
    for track_id in sorted(target_tracks):
        x_position = track_positions.get(track_id)
        if x_position is None or not known_positions:
            continue
        _, team_id = min(known_positions, key=lambda item: abs(item[0] - x_position))
        assignments[track_id] = TeamAssignment(
            team_id=int(team_id),
            team_name=f"team_{team_id}",
            team_confidence=0.4,
            team_color_hex="",
        )
    return assignments


# build_frame_state 처리용 보조 함수.
# data_frame 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _build_frame_state(data_frame: pd.DataFrame) -> Dict[str, Dict[int, Any]]:
    balls: Dict[int, Mapping[str, Any]] = {}
    hoops: Dict[int, Mapping[str, Any]] = {}
    players: Dict[int, List[Mapping[str, Any]]] = {}

    if data_frame.empty:
        return {"balls": balls, "hoops": hoops, "players": players}

    for frame, frame_df in data_frame.dropna(subset=["frame"]).groupby("frame", sort=True):
        frame_number = int(frame)
        data_rows = frame_df.to_dict("records")
        ball_rows = [data_row for data_row in data_rows if str(data_row.get("class", "")).lower() in BALL_CLASSES]
        hoop_rows = [data_row for data_row in data_rows if str(data_row.get("class", "")).lower() in HOOP_CLASSES]
        player_rows = [data_row for data_row in data_rows if str(data_row.get("class", "")).lower() in PLAYER_CLASSES]
        if ball_rows:
            balls[frame_number] = max(ball_rows, key=lambda item: float(item.get("confidence", 0.0) or 0.0))
        if hoop_rows:
            hoops[frame_number] = max(hoop_rows, key=lambda item: float(item.get("confidence", 0.0) or 0.0))
        if player_rows:
            players[frame_number] = player_rows

    return {"balls": balls, "hoops": hoops, "players": players}


# 공 소유권 구간을 만들거나 집계하는 부분.
def _infer_frame_possessions(
    frame_state: Mapping[str, Dict[int, Any]],
    frame_size: Tuple[int, int],
) -> Dict[int, Mapping[str, Any]]:
    scale = max(0.75, min(1.6, max(frame_size) / 1280.0))
    possessions: Dict[int, Mapping[str, Any]] = {}
    balls = frame_state["balls"]
    players_by_frame = frame_state["players"]

    for frame, ball in balls.items():
        players = players_by_frame.get(frame, [])
        possessor = _nearest_possessor(ball, players, scale)
        if possessor is None:
            continue
        try:
            player_track_id = int(float(possessor.get("track_id")))
        except (TypeError, ValueError):
            continue
        possessions[frame] = {
            "frame": frame,
            "player_track_id": player_track_id,
            "team_id": _clean_team_id(possessor.get("team_id")),
            "team_name": _clean_text(possessor.get("team_name")),
        }

    return possessions


# nearest_possessor 처리용 보조 함수.
# ball, players, scale 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _nearest_possessor(
    ball: Mapping[str, Any],
    players: Sequence[Mapping[str, Any]],
    scale: float,
) -> Optional[Mapping[str, Any]]:
    ball_point = _center(ball)
    if ball_point is None:
        return None

    best_player: Optional[Mapping[str, Any]] = None
    best_distance = float("inf")
    # player 항목을 돌면서 조건에 맞는 값만 남김.
    for player in players:
        try:
            x1, y1, x2, y2 = float(player["x1"]), float(player["y1"]), float(player["x2"]), float(player["y2"])
        except (KeyError, TypeError, ValueError):
            continue
        distance = _point_rect_distance(ball_point[0], ball_point[1], x1, y1, x2, y2)
        width = abs(x2 - x1)
        height = abs(y2 - y1)
        threshold = max(55.0 * scale, min(150.0 * scale, height * 0.32 + width * 0.22))
        if distance <= threshold and distance < best_distance:
            best_distance = distance
            best_player = player

    return best_player


# _build_possession_segments는 공 소유권이 이어지는 구간을 만들 때 사용.
def _build_possession_segments(
    possession_by_frame: Mapping[int, Mapping[str, Any]],
    fps: float,
) -> List[PossessionSegment]:
    if not possession_by_frame:
        return []

    max_gap = max(2, int(round(fps * 0.35)))
    min_raw_frames = max(1, int(round(fps * 0.06)))
    segments: List[PossessionSegment] = []
    current: Optional[PossessionSegment] = None

    for frame in sorted(possession_by_frame):
        item = possession_by_frame[frame]
        player_track_id = int(item["player_track_id"])
        team_id = str(item.get("team_id", ""))
        team_name = str(item.get("team_name", ""))

        if (
            current is not None
            and current.player_track_id == player_track_id
            and frame - current.end_frame <= max_gap
        ):
            current.end_frame = frame
            current.raw_frame_count += 1
            if not current.team_id and team_id:
                current.team_id = team_id
                current.team_name = team_name
            continue

        if current is not None and current.raw_frame_count >= min_raw_frames:
            segments.append(current)
        current = PossessionSegment(
            start_frame=frame,
            end_frame=frame,
            player_track_id=player_track_id,
            team_id=team_id,
            team_name=team_name,
            raw_frame_count=1,
        )

    if current is not None and current.raw_frame_count >= min_raw_frames:
        segments.append(current)
    return segments


# _build_possession_stats는 공 소유권이 이어지는 구간을 만들 때 사용.
def _build_possession_stats(
    possession_by_frame: Mapping[int, Mapping[str, Any]],
    fps: float,
) -> Dict[str, Any]:

    fps_value = float(fps) if fps and fps > 0 else 30.0
    team_frames: Dict[str, int] = {}
    team_names: Dict[str, str] = {}
    for item in possession_by_frame.values():
        team_id = _clean_team_id(item.get("team_id"))
        if not team_id:
            continue
        team_frames[team_id] = team_frames.get(team_id, 0) + 1
        team_names.setdefault(team_id, _clean_text(item.get("team_name")) or f"team_{team_id}")

    total_frames = sum(team_frames.values())
    teams: Dict[str, Dict[str, Any]] = {}
    for team_id, frame_count in team_frames.items():
        teams[team_id] = {
            "team_id": team_id,
            "team_name": team_names.get(team_id, f"team_{team_id}"),
            "possession_frame_count": int(frame_count),
            "possession_time_sec": round(frame_count / fps_value, 3),
            "possession_pct": round((frame_count / total_frames) * 100.0, 2)
            if total_frames > 0
            else 0.0,
        }

    return {
        "total_possession_frames": int(total_frames),
        "total_possession_time_sec": round(total_frames / fps_value, 3),
        "teams": teams,
    }


# 슛으로 볼 만한 구간이나 성공 여부를 판단하는 부분.
# 슛 후보 구간이나 성공/실패 판단에 쓰는 함수.
def _detect_shot_windows(
    frame_state: Mapping[str, Dict[int, Any]],
    possession_segments: Sequence[PossessionSegment],
    fps: float,
    frame_size: Tuple[int, int],
) -> List[ShotWindow]:
    balls = frame_state["balls"]
    hoops = frame_state["hoops"]
    if not balls or not hoops:
        return []

    scale = max(0.75, min(1.6, max(frame_size) / 1280.0))
    near_frames: List[Tuple[int, float]] = []
    for frame, ball in balls.items():
        hoop = hoops.get(frame)
        if hoop is None:
            continue
        ball_center = _center(ball)
        hoop_center = _center(hoop)
        if ball_center is None or hoop_center is None:
            continue
        try:
            hoop_w = abs(float(hoop["x2"]) - float(hoop["x1"]))
            hoop_h = abs(float(hoop["y2"]) - float(hoop["y1"]))
        except (KeyError, TypeError, ValueError):
            hoop_w, hoop_h = 60.0 * scale, 40.0 * scale
        rim_radius = max(75.0 * scale, min(230.0 * scale, max(hoop_w * 1.8, hoop_h * 2.4)))
        distance = math.hypot(ball_center[0] - hoop_center[0], ball_center[1] - hoop_center[1])
        if distance <= rim_radius:
            near_frames.append((frame, distance))

    if not near_frames:
        return []

    clusters = _cluster_frame_runs(near_frames, max_gap=max(3, int(round(fps * 0.55))))
    min_separation = max(10, int(round(fps * 1.2)))
    shot_windows: List[ShotWindow] = []
    last_event_frame = -min_separation
    # 이어진 후보 묶음을 하나씩 보면서 실제 이벤트인지 확인.
    for cluster in clusters:
        if not cluster:
            continue
        event_frame, min_distance = min(cluster, key=lambda item: item[1])
        if event_frame - last_event_frame < min_separation:
            continue

        start_frame = cluster[0][0]
        end_frame = cluster[-1][0]
        possessor = _find_last_possession_before(
            possession_segments,
            frame=event_frame,
            max_lookback_frames=max(12, int(round(fps * 4.0))),
        )
        made = _shot_cluster_made(frame_state, cluster, fps)
        confidence = _shot_confidence(min_distance, cluster)
        shot_windows.append(
            ShotWindow(
                start_frame=start_frame,
                end_frame=end_frame,
                event_frame=event_frame,
                made=made,
                confidence=confidence,
                team_id=possessor.team_id if possessor else "",
                team_name=possessor.team_name if possessor else "",
                player_track_id=possessor.player_track_id if possessor else None,
                min_rim_distance=float(min_distance),
            )
        )
        last_event_frame = event_frame

    return shot_windows


# _cluster_frame_runs에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _cluster_frame_runs(items: Sequence[Tuple[int, float]], max_gap: int) -> List[List[Tuple[int, float]]]:
    clusters: List[List[Tuple[int, float]]] = []
    current: List[Tuple[int, float]] = []
    for frame, value in sorted(items):
        if current and frame - current[-1][0] > max_gap:
            clusters.append(current)
            current = []
        current.append((frame, value))
    if current:
        clusters.append(current)
    return clusters


# _shot_cluster_made에서는 슛으로 볼 만한 프레임 구간을 판단.
def _shot_cluster_made(
    frame_state: Mapping[str, Dict[int, Any]],
    cluster: Sequence[Tuple[int, float]],
    fps: float,
) -> bool:
    balls = frame_state["balls"]
    hoops = frame_state["hoops"]
    if not cluster:
        return False

    start = cluster[0][0]
    end = cluster[-1][0] + max(3, int(round(fps * 0.75)))
    candidate_frames = [frame for frame in sorted(balls) if start <= frame <= end and frame in hoops]
    previous_y: Optional[float] = None
    previous_frame_inside_x = False
    for frame in candidate_frames:
        ball = balls[frame]
        hoop = hoops[frame]
        ball_center = _center(ball)
        if ball_center is None:
            continue
        try:
            x1, y1, x2, y2 = float(hoop["x1"]), float(hoop["y1"]), float(hoop["x2"]), float(hoop["y2"])
        except (KeyError, TypeError, ValueError):
            continue
        hoop_w = max(8.0, abs(x2 - x1))
        hoop_h = max(8.0, abs(y2 - y1))
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        inside_x = (x1 - hoop_w * 0.45) <= ball_center[0] <= (x2 + hoop_w * 0.45)
        in_score_zone = inside_x and (y1 - hoop_h * 0.35) <= ball_center[1] <= (y2 + hoop_h * 1.45)
        crossed_down = (
            previous_y is not None
            and previous_frame_inside_x
            and previous_y < cy <= ball_center[1]
        )
        if in_score_zone and (ball_center[1] >= cy or crossed_down):
            return True
        previous_y = ball_center[1]
        previous_frame_inside_x = inside_x
    return False


# _shot_confidence에서는 슛으로 볼 만한 프레임 구간을 판단.
def _shot_confidence(min_distance: float, cluster: Sequence[Tuple[int, float]]) -> float:
    duration_bonus = min(0.18, len(cluster) * 0.015)
    proximity_bonus = max(0.0, min(0.22, (90.0 - min_distance) / 260.0))
    return round(max(0.32, min(0.82, 0.42 + duration_bonus + proximity_bonus)), 3)


# 후보를 찾아내는 탐지 보조 함수.
# detect_passes_and_steals 처리용 보조 함수.
# possession_segments, fps, shot_windows 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _detect_passes_and_steals(
    possession_segments: Sequence[PossessionSegment],
    fps: float,
    shot_windows: Sequence[ShotWindow],
) -> List[Dict[str, Any]]:
    event_rows: List[Dict[str, Any]] = []
    max_transition_gap = max(6, int(round(fps * 3.0)))
    # 끊긴 구간을 하나씩 보면서 이어도 되는지 판단.
    for left, right in zip(possession_segments, possession_segments[1:]):
        if left.player_track_id == right.player_track_id:
            continue
        gap = right.start_frame - left.end_frame
        if gap < 0 or gap > max_transition_gap:
            continue
        if _near_made_shot(right.start_frame, shot_windows, fps):
            continue

        same_known_team = left.team_id and right.team_id and left.team_id == right.team_id
        different_known_team = left.team_id and right.team_id and left.team_id != right.team_id
        if same_known_team:
            event_rows.append(
                _make_event_row(
                    event_type="pass",
                    frame=right.start_frame,
                    fps=fps,
                    team_id=right.team_id,
                    team_name=right.team_name,
                    player_track_id=str(right.player_track_id),
                    secondary_track_id=str(left.player_track_id),
                    confidence=0.58,
                    details=f"possession_changed_same_team;gap_frames={gap}",
                )
            )
        elif different_known_team:
            detail = "possession_changed_opponent_team"
            if _near_missed_shot(right.start_frame, shot_windows, fps):
                detail = "opponent_possession_after_missed_shot_or_block"
            event_rows.append(
                _make_event_row(
                    event_type="steal_or_block",
                    frame=right.start_frame,
                    fps=fps,
                    team_id=right.team_id,
                    team_name=right.team_name,
                    player_track_id=str(right.player_track_id),
                    secondary_track_id=str(left.player_track_id),
                    confidence=0.52,
                    details=f"{detail};gap_frames={gap}",
                )
            )
    return event_rows


# detect_rebounds 처리용 보조 함수.
# possession_segments, shot_windows, fps 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _detect_rebounds(
    possession_segments: Sequence[PossessionSegment],
    shot_windows: Sequence[ShotWindow],
    fps: float,
) -> List[Dict[str, Any]]:
    event_rows: List[Dict[str, Any]] = []
    max_lookahead = max(8, int(round(fps * 5.0)))
    for shot in shot_windows:
        if shot.made:
            continue
        rebound = _find_first_possession_after(
            possession_segments,
            frame=shot.end_frame,
            max_lookahead_frames=max_lookahead,
        )
        if rebound is None:
            continue
        kind = "offensive" if shot.team_id and rebound.team_id == shot.team_id else "defensive"
        event_rows.append(
            _make_event_row(
                event_type="rebound",
                frame=rebound.start_frame,
                fps=fps,
                team_id=rebound.team_id,
                team_name=rebound.team_name,
                player_track_id=str(rebound.player_track_id),
                confidence=0.5,
                details=f"{kind}_rebound_after_missed_shot_frame={shot.event_frame}",
            )
        )
    return event_rows


# detect_contact_fouls 처리용 보조 함수.
# frame_state, possession_segments, fps 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _detect_contact_fouls(
    frame_state: Mapping[str, Dict[int, Any]],
    possession_segments: Sequence[PossessionSegment],
    fps: float,
    frame_size: Tuple[int, int],
    shot_windows: Sequence[ShotWindow],
) -> List[Dict[str, Any]]:
    event_rows: List[Dict[str, Any]] = []
    cooldown = max(10, int(round(fps * 2.5)))
    last_foul_frame = -cooldown
    for left, right in zip(possession_segments, possession_segments[1:]):
        if not left.team_id or not right.team_id or left.team_id == right.team_id:
            continue
        frame = right.start_frame
        if frame - last_foul_frame < cooldown or _frame_in_any_shot_window(frame, shot_windows, fps):
            continue
        contact = _opponent_contact_near_ball(frame_state, frame, fps, frame_size)
        if contact is None:
            continue
        primary, secondary = contact
        event_rows.append(
            _make_event_row(
                event_type="foul",
                frame=frame,
                fps=fps,
                team_id=right.team_id,
                team_name=right.team_name,
                player_track_id=str(primary),
                secondary_track_id=str(secondary),
                confidence=0.28,
                details="heuristic_contact_near_ball_during_opponent_possession_change",
            )
        )
        last_foul_frame = frame
    return event_rows


# 공 후보나 공 좌표를 정리하는 보조 함수.
def _opponent_contact_near_ball(
    frame_state: Mapping[str, Dict[int, Any]],
    frame: int,
    fps: float,
    frame_size: Tuple[int, int],
) -> Optional[Tuple[int, int]]:
    scale = max(0.75, min(1.6, max(frame_size) / 1280.0))
    search_radius = max(2, int(round(fps * 0.35)))
    close_player_distance = 85.0 * scale
    ball_distance = 130.0 * scale
    balls = frame_state["balls"]
    players_by_frame = frame_state["players"]

    for near_frame in range(frame - search_radius, frame + search_radius + 1):
        ball = balls.get(near_frame)
        players = players_by_frame.get(near_frame, [])
        ball_center = _center(ball) if ball is not None else None
        if ball_center is None or len(players) < 2:
            continue
        for index, left in enumerate(players):
            left_team = _clean_team_id(left.get("team_id"))
            if not left_team:
                continue
            for right in players[index + 1 :]:
                right_team = _clean_team_id(right.get("team_id"))
                if not right_team or left_team == right_team:
                    continue
                if not _players_are_close(left, right, close_player_distance):
                    continue
                left_dist = _point_to_record_bbox_distance(ball_center, left)
                right_dist = _point_to_record_bbox_distance(ball_center, right)
                if min(left_dist, right_dist) > ball_distance:
                    continue
                try:
                    return int(float(left["track_id"])), int(float(right["track_id"]))
                except (KeyError, TypeError, ValueError):
                    return None
    return None


# _players_are_close는 선수 row나 선수 위치를 따로 정리하는 부분.
def _players_are_close(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    center_distance_threshold: float,
) -> bool:
    left_center = _center(left)
    right_center = _center(right)
    if left_center is None or right_center is None:
        return False
    center_distance = math.hypot(left_center[0] - right_center[0], left_center[1] - right_center[1])
    return center_distance <= center_distance_threshold or _bbox_iou(left, right) >= 0.04


# _bbox_iou에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
def _bbox_iou(left: Mapping[str, Any], right: Mapping[str, Any]) -> float:
    try:
        lx1, ly1, lx2, ly2 = float(left["x1"]), float(left["y1"]), float(left["x2"]), float(left["y2"])
        rx1, ry1, rx2, ry2 = float(right["x1"]), float(right["y1"]), float(right["x2"]), float(right["y2"])
    except (KeyError, TypeError, ValueError):
        return 0.0
    ix1, iy1 = max(lx1, rx1), max(ly1, ry1)
    ix2, iy2 = min(lx2, rx2), min(ly2, ry2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    intersection = iw * ih
    left_area = max(0.0, lx2 - lx1) * max(0.0, ly2 - ly1)
    right_area = max(0.0, rx2 - rx1) * max(0.0, ry2 - ry1)
    union = left_area + right_area - intersection
    return intersection / union if union > 0 else 0.0


# _make_event_row는 이벤트 row나 요약값을 만드는 쪽.
def _make_event_row(
    event_type: str,
    frame: int,
    fps: float,
    team_id: str = "",
    team_name: str = "",
    player_track_id: str = "",
    secondary_track_id: str = "",
    confidence: float = 0.0,
    details: str = "",
) -> Dict[str, Any]:
    return {
        "row_type": "event",
        "event_id": "",
        "event_type": event_type,
        "event_label_ko": EVENT_LABELS_KO.get(event_type, event_type),
        "count": "",
        "frame": int(frame),
        "time_sec": round(float(frame) / float(fps), 3) if fps > 0 else "",
        "team_id": team_id,
        "team_name": team_name,
        "player_track_id": player_track_id,
        "secondary_track_id": secondary_track_id,
        "confidence": round(float(confidence), 3),
        "details": details,
    }


# 입력 row를 정리한 뒤 필요한 컬럼 순서로 맞춰서 CSV/표 형태로 넘김.
def _cumulative_event_output_dataframe(
    event_rows: Sequence[Mapping[str, Any]],
    fps: float,
    max_frame: int,
    known_teams: Optional[Sequence[Mapping[str, str]]] = None,
    possession_by_frame: Optional[Mapping[int, Mapping[str, Any]]] = None,
    interval_seconds: float = CUMULATIVE_EVENT_INTERVAL_SECONDS,
) -> pd.DataFrame:
    event_rows = sorted(
        [dict(data_row) for data_row in event_rows],
        key=lambda data_row: (
            int(data_row.get("frame") or 0),
            EVENT_ORDER.index(data_row.get("event_type"))
            if data_row.get("event_type") in EVENT_ORDER
            else 999,
        ),
    )
    for index, data_row in enumerate(event_rows, start=1):
        data_row["event_id"] = f"E{index:05d}"

    snapshots = _cumulative_snapshot_specs(
        max_frame=max_frame,
        fps=fps,
        interval_seconds=interval_seconds,
    )

    data_rows: List[Dict[str, Any]] = [dict(data_row) for data_row in event_rows]
    possessions = possession_by_frame or {}
    for snapshot_index, (snapshot_frame, snapshot_time_sec) in enumerate(snapshots, start=1):
        snapshot_events = [
            data_row
            for data_row in event_rows
            if int(data_row.get("frame") or 0) <= int(snapshot_frame)
        ]
        snapshot_possessions = {
            int(frame): value
            for frame, value in possessions.items()
            if int(frame) <= int(snapshot_frame)
        }
        snapshot_possession_stats = _build_possession_stats(snapshot_possessions, fps)

        for team in _event_snapshot_team_keys(
            snapshot_events,
            known_teams,
            snapshot_possession_stats,
        ):
            team_id = str(team["team_id"])
            team_events = [
                data_row
                for data_row in snapshot_events
                if str(data_row.get("team_id", "")).strip() == team_id
            ]
            team_row = _make_wide_summary_row(
                row_type="cumulative_30s_team",
                event_type="team_cumulative_totals",
                event_label_ko="team_cumulative_totals",
                team_id=team_id,
                team_name=str(team.get("team_name", "")),
                event_rows=team_events,
                possession_stats=snapshot_possession_stats,
            )
            _stamp_cumulative_row(
                team_row,
                snapshot_frame=snapshot_frame,
                snapshot_time_sec=snapshot_time_sec,
                snapshot_index=snapshot_index,
                interval_seconds=interval_seconds,
            )
            data_rows.append(team_row)

    return pd.DataFrame(data_rows, columns=EVENT_COLUMNS)


# _event_snapshot_team_keys에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _event_snapshot_team_keys(
    event_rows: Sequence[Mapping[str, Any]],
    known_teams: Optional[Sequence[Mapping[str, str]]],
    possession_stats: Optional[Mapping[str, Any]],
) -> List[Dict[str, str]]:
    teams = _team_summary_keys(event_rows, known_teams, possession_stats)
    by_id = {str(team["team_id"]): dict(team) for team in teams}
    # 팀별로 같은 집계를 반복해서 마지막 표에 넣을 값을 만듦.
    for team_id in ("1", "2"):
        by_id.setdefault(
            team_id,
            {
                "team_id": team_id,
                "team_name": f"team_{team_id}",
            },
        )
    return [by_id[key] for key in sorted(by_id, key=_team_sort_key)[:2]]


# _cumulative_snapshot_specs에서는 슛으로 볼 만한 프레임 구간을 판단.
def _cumulative_snapshot_specs(
    max_frame: int,
    fps: float,
    interval_seconds: float,
) -> List[Tuple[int, float]]:
    fps_value = float(fps) if fps and fps > 0 else 30.0
    interval = max(1.0, float(interval_seconds or CUMULATIVE_EVENT_INTERVAL_SECONDS))
    last_frame = max(0, int(max_frame))
    duration_seconds = max(0.0, (last_frame + 1) / fps_value)
    if duration_seconds <= 0.0:
        return [(0, 0.0)]

    snapshots: List[Tuple[int, float]] = []
    tick = interval
    while tick < duration_seconds - 1e-6:
        frame = max(0, min(last_frame, int(round(tick * fps_value)) - 1))
        snapshots.append((frame, round(tick, 3)))
        tick += interval

    final_time = round(duration_seconds, 3)
    last_snapshot_time = snapshots[-1][1] if snapshots else None
    enough_tail_for_final_row = (
        last_snapshot_time is None
        or final_time - float(last_snapshot_time) >= min(1.0, interval * 0.25)
    )
    if not snapshots or (snapshots[-1][0] != last_frame and enough_tail_for_final_row):
        snapshots.append((last_frame, final_time))
    return snapshots


# _stamp_cumulative_row는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _stamp_cumulative_row(
    data_row: Dict[str, Any],
    snapshot_frame: int,
    snapshot_time_sec: float,
    snapshot_index: int,
    interval_seconds: float,
) -> None:
    data_row["frame"] = int(snapshot_frame)
    data_row["time_sec"] = round(float(snapshot_time_sec), 3)
    data_row["details"] = (
        f"cumulative_from_start_interval_{float(interval_seconds):g}s_"
        f"snapshot_{int(snapshot_index)}"
    )


# _event_output_dataframe에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _event_output_dataframe(
    event_rows: Sequence[Mapping[str, Any]],
    fps: float,
    known_teams: Optional[Sequence[Mapping[str, str]]] = None,
    possession_stats: Optional[Mapping[str, Any]] = None,
) -> pd.DataFrame:
    event_rows = sorted(
        [dict(data_row) for data_row in event_rows],
        key=lambda data_row: (int(data_row.get("frame") or 0), EVENT_ORDER.index(data_row.get("event_type")) if data_row.get("event_type") in EVENT_ORDER else 999),
    )
    for index, data_row in enumerate(event_rows, start=1):
        data_row["event_id"] = f"E{index:05d}"

    wide_summary_rows = [
        _make_wide_summary_row(
            row_type="total_summary",
            event_type="all_totals",
            event_label_ko="전체종합",
            team_id="all",
            team_name="all",
            event_rows=event_rows,
            possession_stats=possession_stats,
        )
    ]
    # team에서 필요한 좌표나 ID만 꺼내 다음 계산에 사용.
    for team in _team_summary_keys(event_rows, known_teams, possession_stats):
        team_id = str(team["team_id"])
        team_events = [
            data_row for data_row in event_rows if str(data_row.get("team_id", "")).strip() == team_id
        ]
        wide_summary_rows.append(
            _make_wide_summary_row(
                row_type="team_summary",
                event_type="team_totals",
                event_label_ko="팀별종합",
                team_id=team_id,
                team_name=str(team.get("team_name", "")),
                event_rows=team_events,
                possession_stats=possession_stats,
            )
        )

    total_summary_rows = []
    # event_type 항목을 돌면서 조건에 맞는 값만 남김.
    for event_type in EVENT_ORDER:
        total_summary_rows.append(
            {
                "row_type": "event_summary",
                "event_id": "",
                "event_type": event_type,
                "event_label_ko": EVENT_LABELS_KO[event_type],
                "count": sum(1 for data_row in event_rows if data_row["event_type"] == event_type),
                "frame": "",
                "time_sec": "",
                "team_id": "all",
                "team_name": "all",
                "player_track_id": "",
                "secondary_track_id": "",
                "confidence": "",
                "details": "total_count",
            }
        )

    team_summary_rows = []
    team_keys = sorted(
        {
            (str(data_row.get("team_id", "")), str(data_row.get("team_name", "")))
            for data_row in event_rows
            if str(data_row.get("team_id", "")).strip()
        }
    )
    for team_id, team_name in team_keys:
        for event_type in EVENT_ORDER:
            team_summary_rows.append(
                {
                    "row_type": "team_event_summary",
                    "event_id": "",
                    "event_type": event_type,
                    "event_label_ko": EVENT_LABELS_KO[event_type],
                    "count": sum(
                        1
                        for data_row in event_rows
                        if data_row["event_type"] == event_type and str(data_row.get("team_id", "")) == team_id
                    ),
                    "frame": "",
                    "time_sec": "",
                    "team_id": team_id,
                    "team_name": team_name,
                    "player_track_id": "",
                    "secondary_track_id": "",
                    "confidence": "",
                    "details": "team_count",
                }
            )

    note_row = {
        "row_type": "note",
        "event_id": "",
        "event_type": "measurement_note",
        "event_label_ko": "측정메모",
        "count": "",
        "frame": "",
        "time_sec": "",
        "team_id": "",
        "team_name": "",
        "player_track_id": "",
        "secondary_track_id": "",
        "confidence": "",
        "details": "객체 박스와 공 궤적 기반 추정값.",
    }

    data_rows = wide_summary_rows + total_summary_rows + team_summary_rows + [note_row] + event_rows
    return pd.DataFrame(data_rows, columns=EVENT_COLUMNS)


# make_wide_summary_row 처리용 보조 함수.
# row_type, event_type, event_label_ko 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def _make_wide_summary_row(
    row_type: str,
    event_type: str,
    event_label_ko: str,
    team_id: str,
    team_name: str,
    event_rows: Sequence[Mapping[str, Any]],
    possession_stats: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    counts = _event_count_values(event_rows)
    counts.update(_possession_feature_values(team_id, possession_stats))
    return {
        "row_type": row_type,
        "event_id": "",
        "event_type": event_type,
        "event_label_ko": event_label_ko,
        "count": sum(int(counts[f"{event_type}_count"]) for event_type in EVENT_ORDER),
        "frame": "",
        "time_sec": "",
        "team_id": team_id,
        "team_name": team_name,
        **counts,
        "player_track_id": "",
        "secondary_track_id": "",
        "confidence": "",
        "details": "wide_team_event_totals",
    }


# _event_count_values는 이벤트 row나 요약값을 만드는 쪽.
def _event_count_values(event_rows: Sequence[Mapping[str, Any]]) -> Dict[str, Any]:
    counts: Dict[str, Any] = {
        f"{event_type}_count": sum(
            1 for data_row in event_rows if data_row.get("event_type") == event_type
        )
        for event_type in EVENT_ORDER
    }
    attempts = int(counts["shot_attempt_count"])
    made = int(counts["shot_made_count"])
    counts["shot_success_rate_pct"] = round((made / attempts) * 100.0, 2) if attempts > 0 else 0.0
    return counts


# _possession_feature_values는 공 소유권이 이어지는 구간을 만들 때 사용.
def _possession_feature_values(
    team_id: str,
    possession_stats: Optional[Mapping[str, Any]],
) -> Dict[str, Any]:
    if not possession_stats:
        return {
            "possession_frame_count": 0,
            "possession_time_sec": 0.0,
            "possession_pct": 0.0,
        }

    if team_id == "all":
        return {
            "possession_frame_count": int(possession_stats.get("total_possession_frames", 0) or 0),
            "possession_time_sec": round(
                float(possession_stats.get("total_possession_time_sec", 0.0) or 0.0),
                3,
            ),
            "possession_pct": 100.0
            if int(possession_stats.get("total_possession_frames", 0) or 0) > 0
            else 0.0,
        }

    teams = possession_stats.get("teams", {})
    team_stats = teams.get(str(team_id), {}) if isinstance(teams, Mapping) else {}
    return {
        "possession_frame_count": int(team_stats.get("possession_frame_count", 0) or 0),
        "possession_time_sec": round(float(team_stats.get("possession_time_sec", 0.0) or 0.0), 3),
        "possession_pct": round(float(team_stats.get("possession_pct", 0.0) or 0.0), 2),
    }


# _team_summary_keys에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _team_summary_keys(
    event_rows: Sequence[Mapping[str, Any]],
    known_teams: Optional[Sequence[Mapping[str, str]]],
    possession_stats: Optional[Mapping[str, Any]] = None,
) -> List[Dict[str, str]]:
    teams: Dict[str, Dict[str, str]] = {}
    for team in known_teams or []:
        team_id = _clean_team_id(team.get("team_id"))
        if not team_id:
            continue
        teams[team_id] = {
            "team_id": team_id,
            "team_name": _clean_text(team.get("team_name")) or f"team_{team_id}",
        }

    # data_row에서 필요한 좌표나 ID만 꺼내 다음 계산에 사용.
    for data_row in event_rows:
        team_id = _clean_team_id(data_row.get("team_id"))
        if not team_id:
            continue
        teams.setdefault(
            team_id,
            {
                "team_id": team_id,
                "team_name": _clean_text(data_row.get("team_name")) or f"team_{team_id}",
            },
        )

    if possession_stats:
        stat_teams = possession_stats.get("teams", {})
        if isinstance(stat_teams, Mapping):
            for team_id, team in stat_teams.items():
                clean_id = _clean_team_id(team_id)
                if not clean_id:
                    continue
                teams.setdefault(
                    clean_id,
                    {
                        "team_id": clean_id,
                        "team_name": _clean_text(team.get("team_name")) or f"team_{clean_id}"
                        if isinstance(team, Mapping)
                        else f"team_{clean_id}",
                    },
                )

    return [teams[key] for key in sorted(teams, key=_team_sort_key)]


# _known_teams_from_tracking에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _known_teams_from_tracking(data_frame: pd.DataFrame) -> List[Dict[str, str]]:
    if data_frame.empty or "team_id" not in data_frame.columns:
        return []

    teams: Dict[str, Dict[str, str]] = {}
    player_mask = (
        data_frame["class"].astype(str).str.lower().isin(PLAYER_CLASSES)
        if "class" in data_frame.columns
        else pd.Series(True, index=data_frame.index)
    )
    for data_row in data_frame[player_mask].to_dict("records"):
        team_id = _clean_team_id(data_row.get("team_id"))
        if not team_id:
            continue
        teams.setdefault(
            team_id,
            {
                "team_id": team_id,
                "team_name": _clean_text(data_row.get("team_name")) or f"team_{team_id}",
            },
        )
    return [teams[key] for key in sorted(teams, key=_team_sort_key)]


# _team_sort_key에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _team_sort_key(team_id: str) -> Tuple[int, str]:
    try:
        return int(float(team_id)), team_id
    except (TypeError, ValueError):
        return 999999, str(team_id)


# _find_last_possession_before는 공 소유권이 이어지는 구간을 만들 때 사용.
def _find_last_possession_before(
    segments: Sequence[PossessionSegment],
    frame: int,
    max_lookback_frames: int,
) -> Optional[PossessionSegment]:
    candidates = [
        segment
        for segment in segments
        if segment.end_frame <= frame and frame - segment.end_frame <= max_lookback_frames
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda segment: segment.end_frame)


# _find_first_possession_after는 공 소유권이 이어지는 구간을 만들 때 사용.
def _find_first_possession_after(
    segments: Sequence[PossessionSegment],
    frame: int,
    max_lookahead_frames: int,
) -> Optional[PossessionSegment]:
    candidates = [
        segment
        for segment in segments
        if segment.start_frame >= frame and segment.start_frame - frame <= max_lookahead_frames
    ]
    if not candidates:
        return None
    return min(candidates, key=lambda segment: segment.start_frame)


# _near_made_shot는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _near_made_shot(frame: int, shots: Sequence[ShotWindow], fps: float) -> bool:
    margin = max(5, int(round(fps * 2.5)))
    return any(shot.made and shot.event_frame <= frame <= shot.end_frame + margin for shot in shots)


# _near_missed_shot는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _near_missed_shot(frame: int, shots: Sequence[ShotWindow], fps: float) -> bool:
    margin = max(5, int(round(fps * 2.0)))
    return any((not shot.made) and shot.start_frame - margin <= frame <= shot.end_frame + margin for shot in shots)


# _frame_in_any_shot_window에서는 프레임 번호를 기준으로 필요한 구간을 맞춤.
def _frame_in_any_shot_window(frame: int, shots: Sequence[ShotWindow], fps: float) -> bool:
    margin = max(4, int(round(fps * 0.7)))
    return any(shot.start_frame - margin <= frame <= shot.end_frame + margin for shot in shots)


# _point_to_record_bbox_distance는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _point_to_record_bbox_distance(point: Tuple[float, float], detection_record: Mapping[str, Any]) -> float:
    try:
        return _point_rect_distance(
            point[0],
            point[1],
            float(detection_record["x1"]),
            float(detection_record["y1"]),
            float(detection_record["x2"]),
            float(detection_record["y2"]),
        )
    except (KeyError, TypeError, ValueError):
        return float("inf")


# _point_rect_distance는 두 위치가 얼마나 가까운지 판단할 때 사용.
def _point_rect_distance(px: float, py: float, x1: float, y1: float, x2: float, y2: float) -> float:
    left, right = sorted((x1, x2))
    top, bottom = sorted((y1, y2))
    dx = max(left - px, 0.0, px - right)
    dy = max(top - py, 0.0, py - bottom)
    return math.hypot(dx, dy)


# _center는 bbox에서 중심 좌표를 뽑아 다음 계산에 넘김.
def _center(detection_record: Optional[Mapping[str, Any]]) -> Optional[Tuple[float, float]]:
    if detection_record is None:
        return None
    try:
        x = float(detection_record["x_center"])
        y = float(detection_record["y_center"])
    except (KeyError, TypeError, ValueError):
        return None
    if not math.isfinite(x) or not math.isfinite(y):
        return None
    return x, y


# _safe_bbox에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
def _safe_bbox(
    data_row: Mapping[str, Any],
    frame_size: Tuple[int, int],
) -> Optional[Tuple[int, int, int, int]]:
    width, height = frame_size
    try:
        x1 = int(max(0, min(width - 1, round(float(data_row["x1"])))))
        y1 = int(max(0, min(height - 1, round(float(data_row["y1"])))))
        x2 = int(max(0, min(width, round(float(data_row["x2"])))))
        y2 = int(max(0, min(height, round(float(data_row["y2"])))))
    except (KeyError, TypeError, ValueError):
        return None
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


# _clean_team_id에서는 팀 번호나 팀별 집계에 필요한 값을 맞춤.
def _clean_team_id(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "unknown"}:
        return ""
    if text.endswith(".0"):
        text = text[:-2]
    return text


# _clean_text는 값 형식을 뒤쪽 계산에서 쓰기 좋게 맞춤.
def _clean_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    text = str(value).strip()
    return "" if text.lower() in {"nan", "none"} else text


__all__ = [
    "assign_fallback_player_track_ids",
    "assign_player_teams_from_video",
    "measure_basketball_event_summary",
    "measure_basketball_events",
    "write_event_measurements",
    "write_event_summary",
]


## Core 6. YOLO 실행 흐름

실제로 영상을 읽고 YOLO를 돌린 다음 CSV까지 만드는 부분.
프레임마다 기본 탐지, 선수 보강, 림/공 보강을 합치고 마지막에 후처리를 태운다.


In [6]:
# - 실제 실행은 run_yolo_tracking_pipeline() 안에서 거의 이어짐.
# - 프레임마다 기본 탐지, 선수 보강, 림 보강, 공 보강을 합쳐 row를 만듦.
# - 공은 작고 자주 놓쳐서 전용 모델/보강 탐지를 한 번 더 돌림.
# - 코트 polygon은 매 프레임 찾지 않고 일정 간격으로 갱신해서 속도를 맞춤.
# - raw 결과를 바로 쓰지 않고 코트 필터, 공 보정, 림 안정화를 거쳐 CSV로 저장.
# - 마지막에 이벤트 CSV와 요약 CSV까지 같이 만듦.

# Core 6. YOLO 실행 흐름


# 영상 돌리는 함수들 모아둔 셀.

# 나중에 다시 볼 때 흐름은 이 정도.
# 노트북 실행부 -> run_yolo_tracking_pipeline() -> YOLO 추론 -> 후처리 -> CSV 저장
# 실제 처리는 거의 run_yolo_tracking_pipeline() 안에서 이어짐.
# 한 함수에 전부 넣으면 너무 길어서 공 탐지, 선수 보강, 림 보강, CSV 저장을 작은 함수로 나눴다.
# 마지막 CSV 컬럼은 아래쪽 함수에서 맞춤.

from __future__ import annotations

import math
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Set, Tuple

import cv2
import pandas as pd
from ultralytics import YOLO


CSV_COLUMNS = [
    "frame",
    "track_id",
    "class",
    "confidence",
    "x_center",
    "y_center",
    "x1",
    "y1",
    "x2",
    "y2",
]

# 모델마다 이름이 조금 달라서 ball/player/referee/hoop으로 맞춤.
CLASS_ALIASES = {
    "ball": "ball",
    "sports ball": "ball",
    "basketball": "ball",
    "frisbee": "ball",
    "player": "player",
    "person": "player",
    "ref": "referee",
    "referee": "referee",
    "hoop": "hoop",
    "rim": "hoop",
    "basket": "hoop",
    "basketball hoop": "hoop",
}

TARGET_CLASSES = {"ball", "player", "referee", "hoop"}
NON_BALL_CLASSES = {"player", "referee", "hoop"}
BALL_CLASSES = {"ball"}

# cv2는 BGR이라 색 지정할 때만 주의.
CLASS_COLORS = {
    "ball": (0, 80, 255),
    "player": (255, 120, 40),
    "referee": (80, 255, 255),
    "hoop": (80, 255, 80),
}

TEAM_COLORS = {
    "1": (255, 80, 80),
    "2": (80, 170, 255),
}

FULL_FRAME_POLYGON_NORM = [
    (0.0, 0.0),
    (1.0, 0.0),
    (1.0, 1.0),
    (0.0, 1.0),
]


# 영상 하나를 읽고 YOLO 탐지부터 CSV 저장까지 이어지는 큰 흐름.
# video_path, model_path, ball_model_path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def run_yolo_tracking_pipeline(
    video_path: str = "Video Project.mp4",
    model_path: str = "player_detector.pt",
    ball_model_path: Optional[str] = "best.pt",
    secondary_ball_model_path: Optional[str] = "ball_detector_model.pt",
    court_model_path: Optional[str] = "court_keypoint_detector.pt",
    csv_output_path: str = "runs/detect/improved_tracking_results.csv",
    event_csv_output_path: Optional[str] = "runs/detect/event.csv",
    event_summary_output_path: Optional[str] = "runs/detect/event_summary.csv",
    tracker_config: str = "bytetrack.yaml",
    conf_threshold: float = 0.12,
    imgsz: int = 1280,
    ball_conf_threshold: float = 0.02,
    ball_iou_threshold: float = 0.65,
    ball_imgsz: int = 1536,
    max_det: int = 300,
    ball_max_det: int = 120,
    enable_hoop_rescue: bool = True,
    hoop_conf_threshold: float = 0.035,
    hoop_iou_threshold: float = 0.65,
    hoop_imgsz: int = 1280,
    hoop_max_det: int = 40,
    enable_player_rescue: bool = True,
    player_rescue_conf_threshold: float = 0.045,
    player_rescue_iou_threshold: float = 0.65,
    player_rescue_imgsz: int = 1280,
    player_rescue_max_det: int = 160,
    player_rescue_min_players: int = 10,
    device: Optional[str] = None,
    use_half_if_cuda: bool = True,
    start_time_seconds: float = 0.0,
    max_duration_seconds: Optional[float] = None,
    enable_rim_ball_rescue: bool = True,
    rim_ball_rescue_conf_threshold: float = 0.006,
    rim_ball_rescue_iou_threshold: float = 0.70,
    rim_ball_rescue_margin_px: float = 220.0,
    rim_ball_rescue_imgsz: int = 1280,
    enable_ball_tile_rescue: bool = False,
    ball_rescue_conf_threshold: float = 0.01,
    ball_rescue_iou_threshold: float = 0.70,
    ball_rescue_imgsz: int = 1280,
    court_conf_threshold: float = 0.15,
    court_keypoint_conf_threshold: float = 0.05,
    court_imgsz: int = 640,
    court_update_interval_frames: int = 10,
    court_polygon_expand_px: float = 35.0,
    test_mode: bool = False,
    max_test_frames: int = 300,
) -> Dict[str, Optional[str]]:
    # 영상 하나 넣으면 tracking/event CSV가 나옴.
    

    _validate_input_file(video_path, "input video")
    _validate_input_file(model_path, "YOLO model")
    if ball_model_path:
        _validate_input_file(ball_model_path, "ball YOLO model")
    if secondary_ball_model_path:
        _validate_input_file(secondary_ball_model_path, "secondary ball YOLO model")
    if court_model_path:
        _validate_input_file(court_model_path, "court keypoint model")

    metadata = read_video_metadata(video_path)
    frame_size = (metadata["width"], metadata["height"])
    adaptive_profile = build_adaptive_processing_profile(metadata)

    effective_conf_threshold = min(float(conf_threshold), float(adaptive_profile["non_ball_conf_threshold"]))
    effective_imgsz = max(int(imgsz), int(adaptive_profile["imgsz"]))
    effective_ball_imgsz = max(int(ball_imgsz), int(adaptive_profile["ball_imgsz"]))
    effective_hoop_imgsz = max(int(hoop_imgsz), int(adaptive_profile["hoop_imgsz"]))
    effective_hoop_conf_threshold = min(
        float(hoop_conf_threshold),
        float(adaptive_profile["hoop_conf_threshold"]),
    )
    effective_player_rescue_imgsz = max(
        int(player_rescue_imgsz),
        int(adaptive_profile["imgsz"]),
    )
    effective_player_rescue_conf_threshold = min(
        float(player_rescue_conf_threshold),
        float(adaptive_profile["player_rescue_conf_threshold"]),
    )
    effective_max_det = max(int(max_det), int(adaptive_profile["max_det"]))
    effective_rim_rescue_margin_px = max(
        float(rim_ball_rescue_margin_px),
        float(adaptive_profile["rim_rescue_margin_px"]),
    )
    effective_rim_rescue_imgsz = max(
        int(rim_ball_rescue_imgsz),
        int(adaptive_profile["rim_rescue_imgsz"]),
    )
    effective_ball_rescue_imgsz = max(
        int(ball_rescue_imgsz),
        int(adaptive_profile["ball_rescue_imgsz"]),
    )
    effective_enable_ball_tile_rescue = bool(enable_ball_tile_rescue or adaptive_profile["enable_tile_rescue"])

    # 3) YOLO 모델을 불러오고, CUDA GPU가 가능하면 GPU로 돌림.
    model = YOLO(model_path)
    ball_model = YOLO(ball_model_path) if ball_model_path else model
    secondary_ball_model = (
        YOLO(secondary_ball_model_path)
        if should_use_secondary_model(ball_model_path, secondary_ball_model_path)
        else None
    )
    court_model = YOLO(court_model_path) if court_model_path else None
    resolved_device = resolve_yolo_device(device)
    use_half = use_half_if_cuda and resolved_device != "cpu"

    non_ball_class_ids = resolve_class_ids(model.names, NON_BALL_CLASSES) or None
    player_class_ids = resolve_class_ids(model.names, {"player"})
    ball_class_ids = resolve_class_ids(ball_model.names, BALL_CLASSES)
    secondary_ball_class_ids = (
        resolve_class_ids(secondary_ball_model.names, BALL_CLASSES)
        if secondary_ball_model is not None
        else []
    )
    hoop_class_ids = resolve_class_ids(model.names, {"hoop"})
    if not ball_class_ids:
        raise ValueError("Core6 ball class 확인")
    if secondary_ball_model is not None and not secondary_ball_class_ids:
        raise ValueError("Core6 secondary ball class 확인")

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    start_frame = max(0, int(round(float(start_time_seconds) * float(metadata["fps"]))))
    if start_frame > 0:
        capture.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    max_processing_frames = resolve_processing_frame_limit(
        fps=float(metadata["fps"]),
        max_duration_seconds=max_duration_seconds,
        test_mode=test_mode,
        max_test_frames=max_test_frames,
    )

    detection_records: List[Dict[str, Any]] = []
    frame_number = 0
    current_court_polygon: Optional[List[Tuple[float, float]]] = None
    court_margin_by_class = {
        "ball": float(adaptive_profile["court_ball_margin_px"]),
        "sports ball": float(adaptive_profile["court_ball_margin_px"]),
        "basketball": float(adaptive_profile["court_ball_margin_px"]),
        "frisbee": float(adaptive_profile["court_ball_margin_px"]),
        "hoop": float(adaptive_profile["court_hoop_margin_px"]),
        "rim": float(adaptive_profile["court_hoop_margin_px"]),
        "basket": float(adaptive_profile["court_hoop_margin_px"]),
        "basketball hoop": float(adaptive_profile["court_hoop_margin_px"]),
    }

    # 프레임마다 기본 탐지, 선수 보강, 림/공 보강을 차례로 합침.
    while capture.isOpened():
        has_frame, frame = capture.read()
        if not has_frame:
            break

        tracked_results = model.track(
            frame,
            persist=True,
            tracker=tracker_config,
            conf=effective_conf_threshold,
            imgsz=effective_imgsz,
            classes=non_ball_class_ids,
            max_det=effective_max_det,
            device=resolved_device,
            half=use_half,
            verbose=False,
        )
        frame_records = extract_detection_records(tracked_results, model.names, frame_number)

        if enable_player_rescue and count_class_records(frame_records, "player") < int(player_rescue_min_players):
            # 선수 수가 너무 적으면 낮은 confidence로 한 번 더 탐지해서 누락된 선수를 보강함.
            player_records = detect_player_records(
                frame=frame,
                model=model,
                model_names=model.names,
                frame_number=frame_number,
                frame_size=frame_size,
                player_class_ids=player_class_ids,
                device=resolved_device,
                use_half=use_half,
                confidence_threshold=effective_player_rescue_conf_threshold,
                iou_threshold=player_rescue_iou_threshold,
                imgsz=effective_player_rescue_imgsz,
                max_det=player_rescue_max_det,
            )
            frame_records = merge_player_records(frame_records, player_records, frame_size)

        # 링이 안 잡힌 프레임만 한 번 더 찾아서 슛 판단에 사용.
        if enable_hoop_rescue and not has_class_record(frame_records, "hoop"):
            hoop_records = detect_hoop_records(
                frame=frame,
                model=model,
                model_names=model.names,
                frame_number=frame_number,
                frame_size=frame_size,
                hoop_class_ids=hoop_class_ids,
                device=resolved_device,
                use_half=use_half,
                confidence_threshold=effective_hoop_conf_threshold,
                iou_threshold=hoop_iou_threshold,
                imgsz=effective_hoop_imgsz,
                max_det=hoop_max_det,
            )
            frame_records = merge_hoop_records(frame_records, hoop_records, frame_size)

        # 공은 작고 confidence가 낮게 나오는 경우가 많아서 공 전용 탐지를 한 번 더 돌림.
        ball_records = detect_ball_records(
            frame=frame,
            model=ball_model,
            model_names=ball_model.names,
            frame_number=frame_number,
            frame_size=frame_size,
            ball_class_ids=ball_class_ids,
            context_records=frame_records,
            device=resolved_device,
            use_half=use_half,
            ball_conf_threshold=ball_conf_threshold,
            ball_iou_threshold=ball_iou_threshold,
            ball_imgsz=effective_ball_imgsz,
            ball_max_det=ball_max_det,
            enable_rim_ball_rescue=enable_rim_ball_rescue,
            rim_rescue_conf_threshold=rim_ball_rescue_conf_threshold,
            rim_rescue_iou_threshold=rim_ball_rescue_iou_threshold,
            rim_rescue_margin_px=effective_rim_rescue_margin_px,
            rim_rescue_imgsz=effective_rim_rescue_imgsz,
            enable_tile_rescue=effective_enable_ball_tile_rescue,
            rescue_conf_threshold=ball_rescue_conf_threshold,
            rescue_iou_threshold=ball_rescue_iou_threshold,
            rescue_imgsz=effective_ball_rescue_imgsz,
            model_label=Path(ball_model_path or model_path).name,
            model_priority_bonus=0.0,
        )
        if secondary_ball_model is not None:
            # 두 번째 공 모델이 있으면 후보를 합친 뒤 크기/위치/중복 기준으로 다시 거른다.
            secondary_ball_records = detect_ball_records(
                frame=frame,
                model=secondary_ball_model,
                model_names=secondary_ball_model.names,
                frame_number=frame_number,
                frame_size=frame_size,
                ball_class_ids=secondary_ball_class_ids,
                context_records=frame_records,
                device=resolved_device,
                use_half=use_half,
                ball_conf_threshold=ball_conf_threshold,
                ball_iou_threshold=ball_iou_threshold,
                ball_imgsz=effective_ball_imgsz,
                ball_max_det=ball_max_det,
                enable_rim_ball_rescue=enable_rim_ball_rescue,
                rim_rescue_conf_threshold=rim_ball_rescue_conf_threshold,
                rim_rescue_iou_threshold=rim_ball_rescue_iou_threshold,
                rim_rescue_margin_px=effective_rim_rescue_margin_px,
                rim_rescue_imgsz=effective_rim_rescue_imgsz,
                enable_tile_rescue=effective_enable_ball_tile_rescue,
                rescue_conf_threshold=ball_rescue_conf_threshold,
                rescue_iou_threshold=ball_rescue_iou_threshold,
                rescue_imgsz=effective_ball_rescue_imgsz,
                model_label=Path(secondary_ball_model_path).name,
                model_priority_bonus=0.0,
            )
            ball_records = filter_ball_candidates(
                [*ball_records, *secondary_ball_records],
                frame_size,
                context_records=frame_records,
            )

        combined_records = frame_records + ball_records
        if court_model is not None:
            # 코트 polygon은 매 프레임 새로 찾으면 느리므로 일정 간격마다 갱신해서 쓴다.
            if should_update_court_polygon(
                frame_number=frame_number,
                interval_frames=court_update_interval_frames,
                current_polygon=current_court_polygon,
            ):
                detected_polygon = detect_court_polygon(
                    frame=frame,
                    model=court_model,
                    frame_size=frame_size,
                    device=resolved_device,
                    use_half=use_half,
                    confidence_threshold=court_conf_threshold,
                    keypoint_conf_threshold=court_keypoint_conf_threshold,
                    imgsz=court_imgsz,
                    expand_px=court_polygon_expand_px,
                )
                if detected_polygon is not None:
                    current_court_polygon = detected_polygon

            combined_records = filter_records_by_court_polygon(
                combined_records,
                frame_size=frame_size,
                court_polygon=current_court_polygon,
                margin_by_class=court_margin_by_class,
            )
        detection_records.extend(combined_records)

        frame_number += 1
        if frame_number % 100 == 0:
            if max_processing_frames is None:
                print(f"YOLO processing frame: {frame_number}", flush=True)
            else:
                print(f"YOLO processing frame: {frame_number}/{max_processing_frames}", flush=True)

        if max_processing_frames is not None and frame_number >= max_processing_frames:
            break

    capture.release()

    # 6) YOLO 원본 결과를 CSV에 맞는 컬럼 구조로 정리.
    raw_df = make_tracking_dataframe(detection_records)
    # raw 결과는 노이즈가 있어서 코트 필터랑 공 보정을 한 번 더 태움.
    # 7) 코트 밖 객체 빼고 공 궤적 보정.
    postprocessed_df = postprocess_basketball_detections(
        raw_df,
        total_frames=frame_number,
        frame_size=frame_size,
        video_path=video_path,
        court_polygon=FULL_FRAME_POLYGON_NORM if court_model is not None else None,
        court_margin_by_class=court_margin_by_class,
        tracking_kwargs={
            "ball_classes": ("ball", "sports ball", "basketball", "frisbee"),
            "ball_label": "ball",
            "player_classes": ("player", "person"),
            "enable_player_occlusion": True,
            "enable_rim_proximity_boost": False,
            "rim_classes": ("hoop", "rim", "basketball hoop"),
            "rim_proximity_px": float(adaptive_profile["rim_proximity_px"]),
            "rim_interpolation_extra_gap": 0,
            "use_motion_association": True,
            "association_max_distance_px": float(adaptive_profile["association_max_distance_px"]),
            "association_gap_growth_px": float(adaptive_profile["association_gap_growth_px"]),
            "association_confidence_weight": 18.0,
            "association_restart_after_frames": int(adaptive_profile["association_restart_after_frames"]),
            "association_restart_min_confidence": 0.004,
            "max_gap": int(adaptive_profile["max_gap"]),
            "max_prediction_frames": int(adaptive_profile["max_prediction_frames"]),
            "process_noise": 90.0,
            "measurement_noise": 7.0,
            "interpolated_measurement_noise": 8.0,
            "interpolated_confidence_floor": 0.45,
            "auto_gravity": True,
        },
    )
    # 림 탐지는 이벤트 판단의 기준점이라 짧은 누락 구간을 보간해서 흔들림을 줄인다.
    postprocessed_df = stabilize_hoop_detections(
        postprocessed_df,
        total_frames=frame_number,
        frame_size=frame_size,
        fps=float(metadata["fps"]),
    )

    # 8) 최종 CSV에 필요한 컬럼만 남기고, 공 좌표에 bbox가 없으면 작은 bbox를 채운다.
    final_df = make_output_dataframe(postprocessed_df)
    final_df = remove_invalid_ball_positions(final_df, frame_size)
    final_df = remove_rim_false_ball_positions(final_df, frame_size, fps=float(metadata["fps"]))
    final_df = assign_fallback_player_track_ids(final_df, frame_size)
    # 원본 공 후보는 확인용으로 두고, 최종 데이터는 이상치 제거/보간/평활화에 쓴다.
    refiner = BallCoordinateRefiner(build_adaptive_refiner_config(frame_size, float(metadata["fps"])))
    improved_tracking_data = refiner.create_cleaned_tracking_results(
        tracking_data=final_df,
        frame_size=frame_size,
        total_frames=frame_number,
        output_path=csv_output_path,
    )
    improved_tracking_data = remove_invalid_ball_positions(improved_tracking_data, frame_size)
    improved_tracking_data = remove_rim_false_ball_positions(improved_tracking_data, frame_size, fps=float(metadata["fps"]))

    # 선수 유니폼 색과 위치를 이용해 팀을 추정하고, 이벤트 요약에 쓸 team_id를 채운다.
    improved_tracking_data, _team_summary = assign_player_teams_from_video(
        tracking_data=improved_tracking_data,
        video_path=video_path,
        frame_size=frame_size,
        start_frame=start_frame,
    )
    Path(csv_output_path).parent.mkdir(parents=True, exist_ok=True)
    improved_tracking_data.to_csv(csv_output_path, index=False, encoding="utf-8-sig")

    # 정제된 tracking 결과를 기준으로 실시간 이벤트 CSV와 최종 요약 CSV를 만듦.
    if event_csv_output_path:
        Path(event_csv_output_path).parent.mkdir(parents=True, exist_ok=True)
        write_event_measurements(
            tracking_data=improved_tracking_data,
            output_csv_path=event_csv_output_path,
            fps=float(metadata["fps"]),
            frame_size=frame_size,
            video_path=video_path,
        )
    if event_summary_output_path:
        Path(event_summary_output_path).parent.mkdir(parents=True, exist_ok=True)
        write_event_summary(
            tracking_data=improved_tracking_data,
            output_csv_path=event_summary_output_path,
            fps=float(metadata["fps"]),
            frame_size=frame_size,
            video_path=video_path,
        )


    print(f"Improved tracking CSV saved: {csv_output_path}")
    if event_csv_output_path:
        print(f"Event measurement CSV saved: {event_csv_output_path}")
    if event_summary_output_path:
        print(f"Event summary CSV saved: {event_summary_output_path}")

    return {
        "improved_csv": csv_output_path,
        "event_csv": event_csv_output_path,
        "event_summary_csv": event_summary_output_path,
    }


# 팀 구분이나 팀별 집계에 쓰는 부분.
# 팀 구분이나 팀별 결과를 맞추는 함수.
# 선수/공 위치를 프레임별로 묶고, 가장 가까운 선수나 팀 기준으로 값을 붙임.
def has_team_specific_classes(model_names: Mapping[int, str]) -> bool:
    # 모델 클래스 이름에 팀 구분 정보가 있는지 확인.

    team_tokens = ("team", "home", "away", "offense", "defense", "white", "dark")
    for raw_name in model_names.values():
        name = str(raw_name).strip().lower()
        if any(token in name for token in team_tokens):
            return True
    return False


# 영상 fps, 해상도, 프레임 수를 읽어옴.
# video_path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def read_video_metadata(video_path: str) -> Dict[str, int | float]:

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.release()

    if fps <= 0:
        fps = 30.0
    if width <= 0 or height <= 0:
        raise RuntimeError("Cannot read video size.")

    return {
        "fps": fps,
        "width": width,
        "height": height,
        "frame_count": frame_count,
    }


# 영상 크기/fps에 맞춰 탐지 기준값을 조절.
# metadata 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def build_adaptive_processing_profile(metadata: Mapping[str, int | float]) -> Dict[str, int | float | bool]:

    width = float(metadata.get("width", 1280) or 1280)
    height = float(metadata.get("height", 720) or 720)
    fps = float(metadata.get("fps", 30.0) or 30.0)
    max_dim = max(width, height, 1.0)

    resolution_scale = max(0.85, min(1.45, max_dim / 1280.0))
    fps_scale = max(0.72, min(1.25, 30.0 / fps))
    motion_scale = resolution_scale * fps_scale

    base_imgsz = _round_to_stride(max(1280.0, min(1600.0, max_dim * 1.05)))
    ball_imgsz = _round_to_stride(max(1536.0, min(1920.0, max_dim * 1.25)))
    rescue_imgsz = _round_to_stride(max(1280.0, min(1600.0, max_dim * 1.05)))

    return {
        "imgsz": int(base_imgsz),
        "ball_imgsz": int(ball_imgsz),
        "hoop_imgsz": int(rescue_imgsz),
        "rim_rescue_imgsz": int(rescue_imgsz),
        "ball_rescue_imgsz": int(rescue_imgsz),
        "non_ball_conf_threshold": 0.08,
        "player_rescue_conf_threshold": 0.032,
        "hoop_conf_threshold": 0.035,
        "max_det": 420,
        "rim_rescue_margin_px": 220.0 * resolution_scale,
        "enable_tile_rescue": True,
        "court_ball_margin_px": 320.0 * resolution_scale,
        "court_hoop_margin_px": 360.0 * resolution_scale,
        "rim_proximity_px": 190.0 * resolution_scale,
        "rim_interpolation_extra_gap": 0,
        "association_max_distance_px": 280.0 * motion_scale,
        "association_gap_growth_px": 55.0 * motion_scale,
        "association_restart_after_frames": 8 if fps >= 45.0 else 7,
        "max_gap": 16 if fps >= 45.0 else 12,
        "max_prediction_frames": 12 if fps >= 45.0 else 9,
    }


# _round_to_stride는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _round_to_stride(value: float, stride: int = 32) -> int:
    # YOLO 입력 크기를 stride 배수로 반올림함.

    return int(round(float(value) / float(stride)) * stride)


# 결과를 DataFrame/CSV 형태로 맞추는 부분.
# derive_refinement_output_prefix 처리용 보조 함수.
# 입력 row를 정리한 뒤 필요한 컬럼 순서로 맞춰서 CSV/표 형태로 넘김.
def derive_refinement_output_prefix(csv_output_path: str) -> str:

    name = Path(csv_output_path).name
    suffix = "tracking_results.csv"
    if name == suffix:
        return ""
    if name.endswith(suffix):
        return name[: -len(suffix)]
    return f"{Path(name).stem}_"


# resolve_processing_frame_limit 처리용 보조 함수.
# fps, max_duration_seconds, test_mode 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def resolve_processing_frame_limit(
    fps: float,
    max_duration_seconds: Optional[float],
    test_mode: bool,
    max_test_frames: int,
) -> Optional[int]:

    limits: List[int] = []
    if max_duration_seconds is not None and max_duration_seconds > 0:
        limits.append(max(1, int(round(float(max_duration_seconds) * float(fps)))))
    if test_mode:
        limits.append(max(1, int(max_test_frames)))
    if not limits:
        return None
    return min(limits)


# should_update_court_polygon 처리용 보조 함수.
# frame_number, interval_frames, current_polygon 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def should_update_court_polygon(
    frame_number: int,
    interval_frames: int,
    current_polygon: Optional[Sequence[Tuple[float, float]]],
) -> bool:

    interval = max(1, int(interval_frames))
    return current_polygon is None or frame_number % interval == 0


# 코트 모델 결과로 현재 화면의 코트 영역을 잡음.
# 코트 모델로 현재 프레임의 코트 영역을 찾음.
# frame, model, frame_size 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def detect_court_polygon(
    frame: Any,
    model: YOLO,
    frame_size: Tuple[int, int],
    device: str,
    use_half: bool,
    confidence_threshold: float,
    keypoint_conf_threshold: float,
    imgsz: int,
    expand_px: float,
) -> Optional[List[Tuple[float, float]]]:

    try:
        results = model.predict(
            frame,
            conf=confidence_threshold,
            imgsz=imgsz,
            max_det=1,
            device=device,
            half=use_half,
            verbose=False,
        )
    except Exception:
        return None

    if not results:
        return None

    result = results[0]
    polygon = court_polygon_from_keypoints(
        result,
        frame_size=frame_size,
        keypoint_conf_threshold=keypoint_conf_threshold,
        expand_px=expand_px,
    )
    if polygon is not None:
        return polygon
    return court_polygon_from_box(result, frame_size=frame_size, expand_px=expand_px)


# 코트 keypoint로 polygon을 만드는 부분.
# 코트 keypoint를 이용해서 polygon을 만듦.
# result, frame_size, keypoint_conf_threshold 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def court_polygon_from_keypoints(
    result: Any,
    frame_size: Tuple[int, int],
    keypoint_conf_threshold: float,
    expand_px: float,
    min_area_ratio: float = 0.04,
) -> Optional[List[Tuple[float, float]]]:

    keypoints = getattr(result, "keypoints", None)
    if keypoints is None or getattr(keypoints, "xy", None) is None:
        return None

    xy_instances = keypoints.xy.cpu().tolist()
    conf_tensor = getattr(keypoints, "conf", None)
    conf_instances = conf_tensor.cpu().tolist() if conf_tensor is not None else None
    width, height = frame_size
    min_area = float(width * height) * float(min_area_ratio)

    best_polygon: Optional[List[Tuple[float, float]]] = None
    best_area = 0.0

    for instance_index, points in enumerate(xy_instances):
        confidences = (
            conf_instances[instance_index]
            if conf_instances is not None
            else [1.0] * len(points)
        )
        valid_points: List[Tuple[float, float]] = []
        for point, confidence in zip(points, confidences):
            if float(confidence) < float(keypoint_conf_threshold):
                continue
            if len(point) < 2:
                continue
            x, y = float(point[0]), float(point[1])
            if not point_is_in_frame((x, y), frame_size):
                continue
            if x == 0.0 and y == 0.0:
                continue
            valid_points.append((x, y))

        hull = convex_hull(valid_points)
        if len(hull) < 3:
            continue

        area = abs(polygon_area(hull))
        if area >= min_area and area > best_area:
            best_area = area
            best_polygon = hull

    if best_polygon is None:
        return None
    return expand_polygon(best_polygon, frame_size=frame_size, expand_px=expand_px)


# keypoint가 부족할 때 bbox로 코트 영역을 대체.
# result, frame_size, expand_px 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def court_polygon_from_box(
    result: Any,
    frame_size: Tuple[int, int],
    expand_px: float,
) -> Optional[List[Tuple[float, float]]]:

    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return None

    best_box = max(boxes, key=lambda box: float(box.conf[0]))
    x1, y1, x2, y2 = map(float, best_box.xyxy[0])
    polygon = [
        (x1, y1),
        (x2, y1),
        (x2, y2),
        (x1, y2),
    ]
    return expand_polygon(polygon, frame_size=frame_size, expand_px=expand_px)


# YOLO 결과 중 코트 밖 row를 제거.
# YOLO row 중 코트 밖 객체를 제거.
# 기준값을 계산하고, 조건에 맞지 않는 row를 mask/거리 기준으로 제거함.
def filter_records_by_court_polygon(
    detection_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
    court_polygon: Optional[Sequence[Tuple[float, float]]],
    margin_by_class: Optional[Mapping[str, float]] = None,
) -> List[Dict[str, Any]]:

    if not detection_records:
        return []

    polygon = court_polygon if court_polygon is not None else DEFAULT_BROADCAST_COURT_POLYGON_NORM
    polygon_is_normalized = court_polygon is None
    try:
        court_filter = CourtObjectFilter(
            polygon,
            frame_size=frame_size,
            polygon_is_normalized=polygon_is_normalized,
            margin_px=12.0,
            margin_by_class=margin_by_class,
        )
    except ValueError:
        return [dict(detection_record) for detection_record in detection_records]

    return court_filter.filter_records(detection_records)


# convex_hull 처리용 보조 함수.
# points 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def convex_hull(points: Sequence[Tuple[float, float]]) -> List[Tuple[float, float]]:

    unique_points = sorted({(round(float(x), 3), round(float(y), 3)) for x, y in points})
    if len(unique_points) <= 1:
        return list(unique_points)

    # cross 처리용 보조 함수.
    # origin, left, right 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
    def cross(
        origin: Tuple[float, float],
        left: Tuple[float, float],
        right: Tuple[float, float],
    ) -> float:
        return (left[0] - origin[0]) * (right[1] - origin[1]) - (
            left[1] - origin[1]
        ) * (right[0] - origin[0])

    lower: List[Tuple[float, float]] = []
    for point in unique_points:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], point) <= 0.0:
            lower.pop()
        lower.append(point)

    upper: List[Tuple[float, float]] = []
    for point in reversed(unique_points):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], point) <= 0.0:
            upper.pop()
        upper.append(point)

    return lower[:-1] + upper[:-1]


# expand_polygon 처리용 보조 함수.
# polygon, frame_size, expand_px 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def expand_polygon(
    polygon: Sequence[Tuple[float, float]],
    frame_size: Tuple[int, int],
    expand_px: float,
) -> List[Tuple[float, float]]:

    if not polygon:
        return []

    center_x = sum(point[0] for point in polygon) / float(len(polygon))
    center_y = sum(point[1] for point in polygon) / float(len(polygon))
    expanded: List[Tuple[float, float]] = []

    for x, y in polygon:
        dx = float(x) - center_x
        dy = float(y) - center_y
        distance = math.hypot(dx, dy)
        if distance <= 1e-6:
            expanded.append(clamp_point((x, y), frame_size))
            continue
        scale = (distance + float(expand_px)) / distance
        expanded.append(clamp_point((center_x + dx * scale, center_y + dy * scale), frame_size))

    return expanded


# polygon_area 처리용 보조 함수.
# polygon 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def polygon_area(polygon: Sequence[Tuple[float, float]]) -> float:

    if len(polygon) < 3:
        return 0.0
    area = 0.0
    for index, point in enumerate(polygon):
        next_point = polygon[(index + 1) % len(polygon)]
        area += point[0] * next_point[1] - next_point[0] * point[1]
    return area / 2.0


# point_is_in_frame 처리용 보조 함수.
# point, frame_size 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def point_is_in_frame(point: Tuple[float, float], frame_size: Tuple[int, int]) -> bool:
    width, height = frame_size
    x, y = point
    return -1.0 <= x <= float(width) + 1.0 and -1.0 <= y <= float(height) + 1.0


# clamp_point 처리용 보조 함수.
def clamp_point(point: Tuple[float, float], frame_size: Tuple[int, int]) -> Tuple[float, float]:
    width, height = frame_size
    x, y = point
    return (
        max(0.0, min(float(width), float(x))),
        max(0.0, min(float(height), float(y))),
    )


# YOLO 결과를 CSV row 형태로 바꿈.
# results, model_names, frame_number 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def extract_detection_records(
    results: Sequence[Any],
    model_names: Mapping[int, str],
    frame_number: int,
    forced_track_id: Optional[int] = None,
) -> List[Dict[str, Any]]:

    if not results:
        return []

    boxes = results[0].boxes
    if boxes is None:
        return []

    frame_records: List[Dict[str, Any]] = []
    for box in boxes:
        cls_id = int(box.cls[0])
        raw_class_name = str(model_names.get(cls_id, cls_id))
        class_name = normalize_class_name(raw_class_name)
        if class_name not in TARGET_CLASSES:
            continue

        x1, y1, x2, y2 = map(float, box.xyxy[0])
        confidence = float(box.conf[0])

        track_id = -1
        if forced_track_id is not None:
            track_id = forced_track_id
        elif box.id is not None:
            track_id = int(box.id[0])

        frame_records.append(
            {
                "frame": frame_number,
                "track_id": track_id,
                "class": class_name,
                "confidence": round(confidence, 3),
                "x_center": round((x1 + x2) / 2.0, 2),
                "y_center": round((y1 + y2) / 2.0, 2),
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
            }
        )

    return frame_records


# 공 전용 모델까지 써서 공 후보를 찾는 부분.
# 공 전용 탐지까지 써서 공 후보를 찾음.
# 공 후보를 모은 뒤 confidence, 위치, 이전 이동 방향을 같이 보고 남길 후보를 고름.
def detect_ball_records(
    frame: Any,
    model: YOLO,
    model_names: Mapping[int, str],
    frame_number: int,
    frame_size: Tuple[int, int],
    ball_class_ids: Sequence[int],
    context_records: Sequence[Mapping[str, Any]],
    device: str,
    use_half: bool,
    ball_conf_threshold: float,
    ball_iou_threshold: float,
    ball_imgsz: int,
    ball_max_det: int,
    enable_rim_ball_rescue: bool,
    rim_rescue_conf_threshold: float,
    rim_rescue_iou_threshold: float,
    rim_rescue_margin_px: float,
    rim_rescue_imgsz: int,
    enable_tile_rescue: bool,
    rescue_conf_threshold: float,
    rescue_iou_threshold: float,
    rescue_imgsz: int,
    model_label: str = "ball_model",
    model_priority_bonus: float = 0.0,
) -> List[Dict[str, Any]]:

    if not ball_class_ids:
        return []

    ball_results = model.predict(
        frame,
        conf=ball_conf_threshold,
        iou=ball_iou_threshold,
        imgsz=ball_imgsz,
        classes=list(ball_class_ids),
        max_det=ball_max_det,
        device=device,
        half=use_half,
        verbose=False,
    )
    ball_records = extract_detection_records(
        ball_results,
        model_names,
        frame_number,
        forced_track_id=-1,
    )
    ball_records = annotate_ball_model_records(ball_records, model_label, model_priority_bonus)
    kept_records = filter_ball_candidates(
        ball_records,
        frame_size,
        context_records=context_records,
    )

    if enable_rim_ball_rescue:
        rim_crops = make_rim_rescue_crops(
            frame_size=frame_size,
            context_records=context_records,
            margin_px=rim_rescue_margin_px,
        )
        if rim_crops and should_run_rim_ball_rescue(
            kept_records,
            context_records=context_records,
            frame_size=frame_size,
            rescue_margin_px=rim_rescue_margin_px,
        ):
            rim_records = detect_ball_records_in_crops(
                frame=frame,
                model=model,
                model_names=model_names,
                frame_number=frame_number,
                ball_class_ids=ball_class_ids,
                crops=rim_crops,
                device=device,
                use_half=use_half,
                confidence_threshold=rim_rescue_conf_threshold,
                iou_threshold=rim_rescue_iou_threshold,
                imgsz=rim_rescue_imgsz,
                max_det=max(12, ball_max_det // 3),
            )
            rim_records = annotate_ball_model_records(
                rim_records,
                f"{model_label}:rim",
                max(model_priority_bonus, 0.035),
            )
            if rim_records:
                kept_records = filter_ball_candidates(
                    [*kept_records, *rim_records],
                    frame_size,
                    context_records=context_records,
                )

    if kept_records or not enable_tile_rescue or not context_records:
        return kept_records

    rescue_records: List[Dict[str, Any]] = []
    for crop in make_ball_rescue_crops(frame_size):
        x1, y1, x2, y2 = crop
        crop_frame = frame[y1:y2, x1:x2]
        if crop_frame.size == 0:
            continue

        crop_records = detect_ball_records_in_crops(
            frame=frame,
            model=model,
            model_names=model_names,
            frame_number=frame_number,
            ball_class_ids=ball_class_ids,
            crops=[crop],
            device=device,
            use_half=use_half,
            confidence_threshold=rescue_conf_threshold,
            iou_threshold=rescue_iou_threshold,
            imgsz=rescue_imgsz,
            max_det=max(20, ball_max_det // 2),
        )
        crop_records = annotate_ball_model_records(crop_records, model_label, model_priority_bonus)
        crop_records = attenuate_rescue_ball_confidence(crop_records)
        rescue_records.extend(crop_records)

    if not rescue_records:
        return kept_records

    rescue_records = filter_rescue_ball_candidates_by_context(
        rescue_records,
        context_records=context_records,
        frame_size=frame_size,
    )
    if not rescue_records:
        return kept_records

    return filter_ball_candidates(
        kept_records + rescue_records,
        frame_size,
        context_records=context_records,
    )


# has_class_record 처리용 보조 함수.
# detection_records, class_name 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def has_class_record(detection_records: Sequence[Mapping[str, Any]], class_name: str) -> bool:

    return any(str(detection_record.get("class")) == class_name for detection_record in detection_records)


# count_class_records 처리용 보조 함수.
def count_class_records(detection_records: Sequence[Mapping[str, Any]], class_name: str) -> int:

    return sum(1 for detection_record in detection_records if str(detection_record.get("class")) == class_name)


# should_use_secondary_model 처리용 보조 함수.
# primary_model_path, secondary_model_path 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def should_use_secondary_model(
    primary_model_path: Optional[str],
    secondary_model_path: Optional[str],
) -> bool:

    if not secondary_model_path:
        return False
    if not primary_model_path:
        return True
    try:
        return Path(primary_model_path).resolve() != Path(secondary_model_path).resolve()
    except OSError:
        return str(primary_model_path) != str(secondary_model_path)


# 공 후보나 공 좌표를 정리하는 보조 함수.
def annotate_ball_model_records(
    detection_records: Sequence[Mapping[str, Any]],
    model_label: str,
    model_priority_bonus: float,
) -> List[Dict[str, Any]]:

    annotated: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        data_row = dict(detection_record)
        data_row["ball_model_source"] = model_label
        data_row["ball_model_sources"] = model_label
        data_row["ball_model_priority_bonus"] = float(model_priority_bonus)
        data_row["model_agreement_count"] = 1
        annotated.append(data_row)
    return annotated


# 선수 누락이 있을 때 낮은 confidence로 보강 탐지.
# frame, model, model_names 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def detect_player_records(
    frame: Any,
    model: YOLO,
    model_names: Mapping[int, str],
    frame_number: int,
    frame_size: Tuple[int, int],
    player_class_ids: Sequence[int],
    device: str,
    use_half: bool,
    confidence_threshold: float,
    iou_threshold: float,
    imgsz: int,
    max_det: int,
) -> List[Dict[str, Any]]:

    if not player_class_ids:
        return []

    player_results = model.predict(
        frame,
        conf=confidence_threshold,
        iou=iou_threshold,
        imgsz=imgsz,
        classes=list(player_class_ids),
        max_det=max_det,
        device=device,
        half=use_half,
        verbose=False,
    )
    player_records = extract_detection_records(
        player_results,
        model_names,
        frame_number,
        forced_track_id=-1,
    )
    return filter_player_candidates(player_records, frame_size)


# 선수 후보나 선수 위치를 정리하는 보조 함수.
# 겹치는 후보를 비교하고 더 믿을 만한 값 하나로 합쳐서 중복을 줄임.
def merge_player_records(
    base_records: Sequence[Mapping[str, Any]],
    rescue_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    if not rescue_records:
        return [dict(detection_record) for detection_record in base_records]

    non_player_records = [
        dict(detection_record)
        for detection_record in base_records
        if str(detection_record.get("class")) != "player"
    ]
    player_records = [
        dict(detection_record)
        for detection_record in [*base_records, *rescue_records]
        if str(detection_record.get("class")) == "player"
    ]
    return non_player_records + deduplicate_player_candidates(
        filter_player_candidates(player_records, frame_size),
        frame_size,
    )


# 림/골대 후보를 찾는 부분.
# 림 후보를 고른 뒤 짧은 누락을 보정해서 슛 판단 기준점으로 사용함.
def detect_hoop_records(
    frame: Any,
    model: YOLO,
    model_names: Mapping[int, str],
    frame_number: int,
    frame_size: Tuple[int, int],
    hoop_class_ids: Sequence[int],
    device: str,
    use_half: bool,
    confidence_threshold: float,
    iou_threshold: float,
    imgsz: int,
    max_det: int,
) -> List[Dict[str, Any]]:

    if not hoop_class_ids:
        return []

    hoop_results = model.predict(
        frame,
        conf=confidence_threshold,
        iou=iou_threshold,
        imgsz=imgsz,
        classes=list(hoop_class_ids),
        max_det=max_det,
        device=device,
        half=use_half,
        verbose=False,
    )
    hoop_records = extract_detection_records(
        hoop_results,
        model_names,
        frame_number,
        forced_track_id=-1,
    )
    return filter_hoop_candidates(hoop_records, frame_size)


# 림/골대 후보를 정리할 때 쓰는 함수.
# 림/골대 후보를 다룰 때 쓰는 보조 함수.
def merge_hoop_records(
    base_records: Sequence[Mapping[str, Any]],
    rescue_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    if not rescue_records:
        return [dict(detection_record) for detection_record in base_records]

    non_hoop_records = [
        dict(detection_record)
        for detection_record in base_records
        if str(detection_record.get("class")) != "hoop"
    ]
    hoop_records = [
        dict(detection_record)
        for detection_record in [*base_records, *rescue_records]
        if str(detection_record.get("class")) == "hoop"
    ]
    return non_hoop_records + filter_hoop_candidates(hoop_records, frame_size)


# filter_hoop_candidates에서는 여러 후보 중 쓸 만한 값만 남김.
def filter_hoop_candidates(
    detection_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    width, height = frame_size
    frame_area = float(width * height)
    scale = max(0.85, min(1.45, max(width, height) / 1280.0))
    kept: List[Dict[str, Any]] = []

    # row를 하나씩 보면서 필요한 값만 모으거나 바꿈.
    for detection_record in detection_records:
        if str(detection_record.get("class")) != "hoop" or not _record_has_bbox(detection_record):
            continue

        box_width = float(detection_record["x2"]) - float(detection_record["x1"])
        box_height = float(detection_record["y2"]) - float(detection_record["y1"])
        if box_width <= 4.0 * scale or box_height <= 4.0 * scale:
            continue

        area = box_width * box_height
        aspect_ratio = box_width / max(1.0, box_height)
        x_center = float(detection_record["x_center"])
        y_center = float(detection_record["y_center"])

        if y_center > height * 0.78:
            continue
        if x_center < -width * 0.03 or x_center > width * 1.03:
            continue
        if area > frame_area * 0.035:
            continue
        if max(box_width, box_height) > max(width, height) * 0.23:
            continue
        if aspect_ratio < 0.25 or aspect_ratio > 4.5:
            continue

        kept.append(dict(detection_record))

    kept.sort(key=lambda detection_record: _hoop_candidate_quality(detection_record, frame_size), reverse=True)
    return deduplicate_hoop_candidates(kept, frame_size)[:4]


# deduplicate_hoop_candidates에서는 여러 후보 중 쓸 만한 값만 남김.
def deduplicate_hoop_candidates(
    detection_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    scale = max(0.85, min(1.45, max(frame_size) / 1280.0))
    deduped: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        duplicate = False
        for kept in deduped:
            if _ball_candidate_overlap(detection_record, kept) > 0.35:
                duplicate = True
                break
            center_distance = math.hypot(
                float(detection_record["x_center"]) - float(kept["x_center"]),
                float(detection_record["y_center"]) - float(kept["y_center"]),
            )
            if center_distance <= 28.0 * scale:
                duplicate = True
                break
        if not duplicate:
            deduped.append(dict(detection_record))
    return deduped


# _hoop_candidate_quality에서는 여러 후보 중 쓸 만한 값만 남김.
def _hoop_candidate_quality(detection_record: Mapping[str, Any], frame_size: Tuple[int, int]) -> float:

    width, height = frame_size
    confidence = float(detection_record.get("confidence", 0.0) or 0.0)
    box_width = max(1.0, float(detection_record["x2"]) - float(detection_record["x1"]))
    box_height = max(1.0, float(detection_record["y2"]) - float(detection_record["y1"]))
    area_ratio = (box_width * box_height) / max(1.0, float(width * height))
    aspect_ratio = box_width / box_height
    y_center = float(detection_record["y_center"])

    aspect_penalty = abs(1.25 - aspect_ratio) * 0.025
    area_penalty = max(0.0, area_ratio - 0.006) * 6.0
    upper_bonus = max(0.0, (height * 0.65 - y_center) / max(1.0, height)) * 0.04
    return confidence + upper_bonus - aspect_penalty - area_penalty


# filter_player_candidates에서는 여러 후보 중 쓸 만한 값만 남김.
def filter_player_candidates(
    detection_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    width, height = frame_size
    frame_area = float(width * height)
    scale = max(0.85, min(1.45, max(width, height) / 1280.0))
    kept: List[Dict[str, Any]] = []

    for detection_record in detection_records:
        if str(detection_record.get("class")) != "player" or not _record_has_bbox(detection_record):
            continue

        box_width = float(detection_record["x2"]) - float(detection_record["x1"])
        box_height = float(detection_record["y2"]) - float(detection_record["y1"])
        if box_width <= 10.0 * scale or box_height <= 22.0 * scale:
            continue
        if box_height > height * 0.95 or box_width > width * 0.38:
            continue

        area = box_width * box_height
        if area < frame_area * 0.00018 or area > frame_area * 0.22:
            continue

        aspect_ratio = box_width / max(1.0, box_height)
        if aspect_ratio < 0.12 or aspect_ratio > 1.35:
            continue

        kept.append(dict(detection_record))

    kept.sort(key=_player_candidate_quality, reverse=True)
    return kept[:18]


# deduplicate_player_candidates에서는 여러 후보 중 쓸 만한 값만 남김.
def deduplicate_player_candidates(
    detection_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    scale = max(0.85, min(1.45, max(frame_size) / 1280.0))
    deduped: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        duplicate = False
        for kept in deduped:
            if _ball_candidate_overlap(detection_record, kept) > 0.42:
                duplicate = True
                break
            center_distance = math.hypot(
                float(detection_record["x_center"]) - float(kept["x_center"]),
                float(detection_record["y_center"]) - float(kept["y_center"]),
            )
            if center_distance <= 32.0 * scale:
                duplicate = True
                break
        if not duplicate:
            deduped.append(dict(detection_record))
    return deduped[:14]


# _player_candidate_quality에서는 여러 후보 중 쓸 만한 값만 남김.
def _player_candidate_quality(detection_record: Mapping[str, Any]) -> float:
    confidence = float(detection_record.get("confidence", 0.0) or 0.0)
    try:
        track_id = int(float(detection_record.get("track_id", -1) or -1))
    except (TypeError, ValueError):
        track_id = -1
    tracked_bonus = 0.10 if track_id >= 0 else 0.0
    box_width = max(1.0, float(detection_record["x2"]) - float(detection_record["x1"]))
    box_height = max(1.0, float(detection_record["y2"]) - float(detection_record["y1"]))
    aspect_ratio = box_width / box_height
    aspect_penalty = abs(0.42 - aspect_ratio) * 0.035
    return confidence + tracked_bonus - aspect_penalty


# make_ball_rescue_crops는 공 좌표나 공 후보를 따로 정리할 때 사용.
def make_ball_rescue_crops(frame_size: Tuple[int, int]) -> List[Tuple[int, int, int, int]]:

    width, height = frame_size
    left_end = int(round(width * 0.58))
    right_start = int(round(width * 0.42))
    center_start = int(round(width * 0.20))
    center_end = int(round(width * 0.80))

    crops = [
        (0, 0, left_end, height),
        (right_start, 0, width, height),
        (center_start, 0, center_end, height),
    ]
    return [
        (max(0, x1), max(0, y1), min(width, x2), min(height, y2))
        for x1, y1, x2, y2 in crops
        if x2 - x1 >= 64 and y2 - y1 >= 64
    ]


# make_rim_rescue_crops는 림/골대 위치를 안정적으로 잡기 위한 보조 처리.
def make_rim_rescue_crops(
    frame_size: Tuple[int, int],
    context_records: Sequence[Mapping[str, Any]],
    margin_px: float,
) -> List[Tuple[int, int, int, int]]:

    crops: List[Tuple[int, int, int, int]] = []
    for detection_record in context_records:
        if str(detection_record.get("class")) != "hoop" or not _record_has_bbox(detection_record):
            continue
        x1 = float(detection_record["x1"]) - float(margin_px)
        y1 = float(detection_record["y1"]) - float(margin_px)
        x2 = float(detection_record["x2"]) + float(margin_px)
        y2 = float(detection_record["y2"]) + float(margin_px)
        crop = clamp_crop((x1, y1, x2, y2), frame_size)
        if crop is not None:
            crops.append(crop)
    return deduplicate_crops(crops)


# 잘라낸 영역에서 공 후보를 다시 찾음.
# crop 영역 안에서 공 후보를 다시 찾음.
def detect_ball_records_in_crops(
    frame: Any,
    model: YOLO,
    model_names: Mapping[int, str],
    frame_number: int,
    ball_class_ids: Sequence[int],
    crops: Sequence[Tuple[int, int, int, int]],
    device: str,
    use_half: bool,
    confidence_threshold: float,
    iou_threshold: float,
    imgsz: int,
    max_det: int,
) -> List[Dict[str, Any]]:

    crop_records: List[Dict[str, Any]] = []
    for crop in crops:
        x1, y1, x2, y2 = crop
        crop_frame = frame[y1:y2, x1:x2]
        if crop_frame.size == 0:
            continue

        crop_results = model.predict(
            crop_frame,
            conf=confidence_threshold,
            iou=iou_threshold,
            imgsz=imgsz,
            classes=list(ball_class_ids),
            max_det=max_det,
            device=device,
            half=use_half,
            verbose=False,
        )
        detection_records = extract_detection_records(
            crop_results,
            model_names,
            frame_number,
            forced_track_id=-1,
        )
        crop_records.extend(offset_detection_records(detection_records, x_offset=x1, y_offset=y1))
    return crop_records


# clamp_crop 처리용 보조 함수.
# crop, frame_size 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def clamp_crop(
    crop: Tuple[float, float, float, float],
    frame_size: Tuple[int, int],
) -> Optional[Tuple[int, int, int, int]]:

    width, height = frame_size
    x1, y1, x2, y2 = crop
    output = (
        max(0, min(width, int(round(x1)))),
        max(0, min(height, int(round(y1)))),
        max(0, min(width, int(round(x2)))),
        max(0, min(height, int(round(y2)))),
    )
    if output[2] - output[0] < 64 or output[3] - output[1] < 64:
        return None
    return output


# 중복으로 잡힌 후보를 합치거나 줄이는 부분.
# 중복 후보를 합치거나 줄이는 함수.
def deduplicate_crops(
    crops: Sequence[Tuple[int, int, int, int]],
) -> List[Tuple[int, int, int, int]]:

    seen = set()
    output: List[Tuple[int, int, int, int]] = []
    for crop in crops:
        if crop in seen:
            continue
        seen.add(crop)
        output.append(crop)
    return output


# 후보를 찾아내는 탐지 보조 함수.
# offset_detection_records 처리용 보조 함수.
# detection_records, x_offset, y_offset 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def offset_detection_records(
    detection_records: Sequence[Mapping[str, Any]],
    x_offset: float,
    y_offset: float,
) -> List[Dict[str, Any]]:

    adjusted: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        data_row = dict(detection_record)
        for column in ["x_center", "x1", "x2"]:
            data_row[column] = round(float(data_row[column]) + float(x_offset), 2)
        for column in ["y_center", "y1", "y2"]:
            data_row[column] = round(float(data_row[column]) + float(y_offset), 2)
        adjusted.append(data_row)
    return adjusted


# attenuate_rescue_ball_confidence는 공 좌표나 공 후보를 따로 정리할 때 사용.
def attenuate_rescue_ball_confidence(detection_records: Sequence[Mapping[str, Any]]) -> List[Dict[str, Any]]:

    adjusted: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        data_row = dict(detection_record)
        confidence = float(data_row.get("confidence", 0.0) or 0.0)
        data_row["confidence"] = round(min(0.07, confidence * 0.35), 3)
        adjusted.append(data_row)
    return adjusted


# filter_rescue_ball_candidates_by_context에서는 공 후보를 비교해서 남길 것만 고름.
def filter_rescue_ball_candidates_by_context(
    detection_records: Sequence[Mapping[str, Any]],
    context_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> List[Dict[str, Any]]:

    width, height = frame_size
    scale = max(0.85, min(1.45, max(width, height) / 1280.0))
    context = [
        detection_record
        for detection_record in context_records
        if str(detection_record.get("class")) in {"player", "referee", "hoop"}
        and _record_has_bbox(detection_record)
    ]
    if not context:
        return []

    filtered: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        ball_x = float(detection_record["x_center"])
        ball_y = float(detection_record["y_center"])
        if ball_y < height * 0.08:
            continue

        for context_record in context:
            class_name = str(context_record.get("class"))
            margin = (250.0 if class_name == "hoop" else 185.0) * scale
            if _point_inside_expanded_bbox(ball_x, ball_y, context_record, margin):
                filtered.append(dict(detection_record))
                break

    return filtered


# 공 후보 중 크기, 위치가 이상한 것을 제거.
# 크기, 위치, confidence 기준으로 공 후보를 거름.
def filter_ball_candidates(
    detection_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
    context_records: Optional[Sequence[Mapping[str, Any]]] = None,
) -> List[Dict[str, Any]]:

    width, height = frame_size
    frame_area = float(width * height)
    resolution_scale = max(0.85, min(1.45, max(width, height) / 1280.0))
    max_ball_area = min(frame_area * 0.0032, 5200.0 * resolution_scale * resolution_scale)
    max_ball_side = 95.0 * resolution_scale
    kept: List[Dict[str, Any]] = []

    for detection_record in detection_records:
        if detection_record.get("class") != "ball":
            continue

        box_width = float(detection_record["x2"]) - float(detection_record["x1"])
        box_height = float(detection_record["y2"]) - float(detection_record["y1"])
        if box_width <= 3.0 or box_height <= 3.0:
            continue

        area = box_width * box_height
        aspect_ratio = box_width / max(1.0, box_height)

        if area > max_ball_area:
            continue
        if max(box_width, box_height) > max_ball_side:
            continue
        if aspect_ratio < 0.45 or aspect_ratio > 2.2:
            continue
        if _is_likely_player_body_false_ball(detection_record, context_records, frame_size):
            continue
        if _is_likely_rim_false_ball(detection_record, context_records, frame_size):
            continue

        kept.append(dict(detection_record))

    kept.sort(key=_ball_candidate_quality, reverse=True)
    return deduplicate_ball_candidates(kept)[:12]


# 같은 공이 여러 번 잡힌 후보를 하나로 줄임.
def deduplicate_ball_candidates(detection_records: Sequence[Mapping[str, Any]]) -> List[Dict[str, Any]]:

    deduped: List[Dict[str, Any]] = []
    for detection_record in detection_records:
        duplicate = False
        for kept_index, kept in enumerate(deduped):
            if _ball_candidate_overlap(detection_record, kept) > 0.45:
                deduped[kept_index] = merge_duplicate_ball_candidate(detection_record, kept)
                duplicate = True
                break
            center_distance = math.hypot(
                float(detection_record["x_center"]) - float(kept["x_center"]),
                float(detection_record["y_center"]) - float(kept["y_center"]),
            )
            if center_distance <= max(
                10.0,
                _ball_candidate_size(detection_record) * 0.65,
                _ball_candidate_size(kept) * 0.65,
            ):
                deduped[kept_index] = merge_duplicate_ball_candidate(detection_record, kept)
                duplicate = True
                break
        if not duplicate:
            deduped.append(dict(detection_record))
    return deduped


# merge_duplicate_ball_candidate에서는 공 후보를 비교해서 남길 것만 고름.
def merge_duplicate_ball_candidate(
    first: Mapping[str, Any],
    second: Mapping[str, Any],
) -> Dict[str, Any]:

    first_sources = _candidate_sources(first)
    second_sources = _candidate_sources(second)
    combined_sources = sorted(first_sources | second_sources)

    best = dict(first if _ball_candidate_quality(first) >= _ball_candidate_quality(second) else second)
    first_confidence = float(first.get("confidence", 0.0) or 0.0)
    second_confidence = float(second.get("confidence", 0.0) or 0.0)
    agreement_bonus = 0.04 * max(0, len(combined_sources) - 1)
    best["confidence"] = round(min(0.99, max(first_confidence, second_confidence) + agreement_bonus), 3)
    best["ball_model_sources"] = "|".join(combined_sources)
    best["ball_model_source"] = best.get("ball_model_source") or combined_sources[0]
    best["model_agreement_count"] = len(combined_sources)
    best["ball_model_priority_bonus"] = max(
        float(first.get("ball_model_priority_bonus", 0.0) or 0.0),
        float(second.get("ball_model_priority_bonus", 0.0) or 0.0),
    )
    return best


# _candidate_sources에서는 여러 후보 중 쓸 만한 값만 남김.
def _candidate_sources(detection_record: Mapping[str, Any]) -> Set[str]:
    sources_value = detection_record.get("ball_model_sources") or detection_record.get("ball_model_source") or ""
    sources = {
        source.strip()
        for source in str(sources_value).split("|")
        if source and source.strip()
    }
    if not sources:
        sources.add("unknown")
    return sources


# _ball_candidate_quality에서는 공 후보를 비교해서 남길 것만 고름.
def _ball_candidate_quality(detection_record: Mapping[str, Any]) -> float:

    confidence = float(detection_record.get("confidence", 0.0) or 0.0)
    width = float(detection_record["x2"]) - float(detection_record["x1"])
    height = float(detection_record["y2"]) - float(detection_record["y1"])
    area = max(1.0, width * height)
    aspect_ratio = width / max(1.0, height)

    aspect_penalty = abs(1.0 - aspect_ratio) * 0.08
    size_penalty = max(0.0, area - 1600.0) / 8000.0
    agreement_bonus = 0.035 * max(0, int(detection_record.get("model_agreement_count", 1) or 1) - 1)
    source_bonus = float(detection_record.get("ball_model_priority_bonus", 0.0) or 0.0)
    return confidence + agreement_bonus + source_bonus - aspect_penalty - size_penalty


# _is_likely_rim_false_ball는 공 좌표나 공 후보를 따로 정리할 때 사용.
def _is_likely_rim_false_ball(
    detection_record: Mapping[str, Any],
    context_records: Optional[Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
) -> bool:

    if not context_records:
        return False

    confidence = float(detection_record.get("confidence", 0.0) or 0.0)
    ball_size = _ball_candidate_size(detection_record)
    ball_area = _candidate_area(detection_record)
    if ball_area <= 0.0:
        return True

    width, height = frame_size
    scale = max(0.85, min(1.45, max(width, height) / 1280.0))
    ball_x = float(detection_record["x_center"])
    ball_y = float(detection_record["y_center"])

    for context in context_records:
        if str(context.get("class")) != "hoop" or not _record_has_bbox(context):
            continue

        hoop_width = float(context["x2"]) - float(context["x1"])
        hoop_height = float(context["y2"]) - float(context["y1"])
        if hoop_width <= 6.0 or hoop_height <= 6.0:
            continue

        if ball_is_above_hoop_arc_zone(ball_y, context, scale):
            continue

        hoop_size = max(hoop_width, hoop_height)
        center_inside_hoop = _point_inside_expanded_bbox(ball_x, ball_y, context, 3.0 * scale)
        ball_inside_ratio = _candidate_intersection_ratio(detection_record, context)
        near_hoop_center = math.hypot(
            ball_x - float(context["x_center"]),
            ball_y - float(context["y_center"]),
        ) <= max(18.0 * scale, hoop_size * 0.28)

        if (
            confidence < 0.06
            and center_inside_hoop
            and ball_inside_ratio >= 0.70
            and ball_size <= max(42.0 * scale, hoop_size * 0.48)
        ):
            return True
        if (
            confidence < 0.10
            and near_hoop_center
            and ball_inside_ratio >= 0.88
            and ball_size <= max(54.0 * scale, hoop_size * 0.62)
        ):
            return True

    return False


# should_run_rim_ball_rescue는 공 좌표나 공 후보를 따로 정리할 때 사용.
def should_run_rim_ball_rescue(
    kept_records: Sequence[Mapping[str, Any]],
    context_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
    rescue_margin_px: float,
) -> bool:

    hoops = [
        detection_record
        for detection_record in context_records
        if str(detection_record.get("class")) == "hoop" and _record_has_bbox(detection_record)
    ]
    if not hoops:
        return False
    if not kept_records:
        return True

    for ball in kept_records:
        ball_x = float(ball["x_center"])
        ball_y = float(ball["y_center"])
        for hoop in hoops:
            if _point_inside_expanded_bbox(
                ball_x,
                ball_y,
                hoop,
                max(80.0, float(rescue_margin_px) * 0.55),
            ):
                return False

    best_quality = max(_ball_candidate_quality(detection_record) for detection_record in kept_records)
    if best_quality < 0.82:
        return True

    return any(
        _ball_candidate_inside_player_body_core(detection_record, context_records, frame_size)
        for detection_record in kept_records
    )


# _is_likely_player_body_false_ball는 공 좌표나 공 후보를 따로 정리할 때 사용.
def _is_likely_player_body_false_ball(
    detection_record: Mapping[str, Any],
    context_records: Optional[Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
) -> bool:

    if not context_records:
        return False

    confidence = float(detection_record.get("confidence", 0.0) or 0.0)
    agreement_count = int(detection_record.get("model_agreement_count", 1) or 1)
    if confidence >= 0.62 or (agreement_count >= 2 and confidence >= 0.25):
        return False

    return _ball_candidate_inside_player_body_core(detection_record, context_records, frame_size)


# _ball_candidate_inside_player_body_core에서는 공 후보를 비교해서 남길 것만 고름.
def _ball_candidate_inside_player_body_core(
    detection_record: Mapping[str, Any],
    context_records: Sequence[Mapping[str, Any]],
    frame_size: Tuple[int, int],
) -> bool:

    ball_x = float(detection_record["x_center"])
    ball_y = float(detection_record["y_center"])
    width, height = frame_size
    scale = max(0.85, min(1.45, max(width, height) / 1280.0))

    for context in context_records:
        if str(context.get("class")) not in {"player", "person"}:
            continue
        if not _record_has_bbox(context):
            continue
        try:
            x1 = float(context["x1"])
            y1 = float(context["y1"])
            x2 = float(context["x2"])
            y2 = float(context["y2"])
        except (KeyError, TypeError, ValueError):
            continue

        box_width = max(1.0, abs(x2 - x1))
        box_height = max(1.0, abs(y2 - y1))
        margin = 4.0 * scale
        if not (x1 - margin <= ball_x <= x2 + margin and y1 - margin <= ball_y <= y2 + margin):
            continue

        rel_x = (ball_x - x1) / box_width
        rel_y = (ball_y - y1) / box_height
        if 0.12 <= rel_x <= 0.88 and 0.10 <= rel_y <= 0.70:
            return True

    return False


# _record_has_bbox에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
def _record_has_bbox(detection_record: Mapping[str, Any]) -> bool:
    return all(
        _as_finite_float(detection_record.get(column)) is not None
        for column in ["x1", "y1", "x2", "y2"]
    )


# _point_inside_expanded_bbox에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
def _point_inside_expanded_bbox(
    x: float,
    y: float,
    detection_record: Mapping[str, Any],
    margin: float,
) -> bool:
    x1 = float(detection_record["x1"]) - margin
    y1 = float(detection_record["y1"]) - margin
    x2 = float(detection_record["x2"]) + margin
    y2 = float(detection_record["y2"]) + margin
    return x1 <= x <= x2 and y1 <= y <= y2


# ball_is_above_hoop_arc_zone는 공 좌표나 공 후보를 따로 정리할 때 사용.
def ball_is_above_hoop_arc_zone(
    ball_y: float,
    hoop_record: Mapping[str, Any],
    scale: float,
) -> bool:

    hoop_height = max(1.0, float(hoop_record["y2"]) - float(hoop_record["y1"]))
    hoop_center_y = float(hoop_record["y_center"])
    arc_margin = max(8.0 * scale, hoop_height * 0.20)
    return float(ball_y) < hoop_center_y - arc_margin


# _as_finite_float는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _as_finite_float(value: Any) -> Optional[float]:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(result) or math.isinf(result):
        return None
    return result


# _ball_candidate_size에서는 공 후보를 비교해서 남길 것만 고름.
def _ball_candidate_size(detection_record: Mapping[str, Any]) -> float:

    width = float(detection_record["x2"]) - float(detection_record["x1"])
    height = float(detection_record["y2"]) - float(detection_record["y1"])
    return max(width, height)


# _ball_candidate_overlap에서는 공 후보를 비교해서 남길 것만 고름.
def _ball_candidate_overlap(first: Mapping[str, Any], second: Mapping[str, Any]) -> float:

    first_x1, first_y1, first_x2, first_y2 = _candidate_bbox(first)
    second_x1, second_y1, second_x2, second_y2 = _candidate_bbox(second)

    inter_x1 = max(first_x1, second_x1)
    inter_y1 = max(first_y1, second_y1)
    inter_x2 = min(first_x2, second_x2)
    inter_y2 = min(first_y2, second_y2)
    inter_width = max(0.0, inter_x2 - inter_x1)
    inter_height = max(0.0, inter_y2 - inter_y1)
    intersection = inter_width * inter_height

    first_area = max(0.0, first_x2 - first_x1) * max(0.0, first_y2 - first_y1)
    second_area = max(0.0, second_x2 - second_x1) * max(0.0, second_y2 - second_y1)
    union = first_area + second_area - intersection
    if union <= 0.0:
        return 0.0
    return intersection / union


# _candidate_intersection_ratio에서는 여러 후보 중 쓸 만한 값만 남김.
def _candidate_intersection_ratio(
    first: Mapping[str, Any],
    second: Mapping[str, Any],
) -> float:

    first_x1, first_y1, first_x2, first_y2 = _candidate_bbox(first)
    second_x1, second_y1, second_x2, second_y2 = _candidate_bbox(second)
    inter_x1 = max(first_x1, second_x1)
    inter_y1 = max(first_y1, second_y1)
    inter_x2 = min(first_x2, second_x2)
    inter_y2 = min(first_y2, second_y2)
    intersection = max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)
    first_area = _candidate_area(first)
    if first_area <= 0.0:
        return 0.0
    return intersection / first_area


# _candidate_area에서는 여러 후보 중 쓸 만한 값만 남김.
def _candidate_area(detection_record: Mapping[str, Any]) -> float:
    x1, y1, x2, y2 = _candidate_bbox(detection_record)
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


# _candidate_bbox에서는 bbox 좌표를 계산하거나 비교하기 좋게 정리.
def _candidate_bbox(detection_record: Mapping[str, Any]) -> Tuple[float, float, float, float]:
    return (
        float(detection_record["x1"]),
        float(detection_record["y1"]),
        float(detection_record["x2"]),
        float(detection_record["y2"]),
    )


# 값 형식이 흔들리지 않게 정리하는 함수.
# 값을 같은 형식으로 맞춘 뒤 뒤쪽 계산에서 비교하기 쉽게 넘김.
def normalize_class_name(raw_name: str) -> str:

    normalized = raw_name.strip().lower()
    return CLASS_ALIASES.get(normalized, normalized)


# resolve_class_ids 처리용 보조 함수.
# model_names, wanted_classes 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def resolve_class_ids(model_names: Mapping[int, str], wanted_classes: Set[str]) -> List[int]:

    class_ids: List[int] = []
    for class_id, raw_name in model_names.items():
        if normalize_class_name(str(raw_name)) in wanted_classes:
            class_ids.append(int(class_id))
    return class_ids


# resolve_yolo_device 처리용 보조 함수.
# device 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def resolve_yolo_device(device: Optional[str] = None) -> str:

    if device:
        normalized = str(device).strip().lower()
        if normalized in {"cuda", "gpu"}:
            return "0"
        if normalized.startswith("cuda:"):
            return normalized.split(":", 1)[1] or "0"
        return normalized

    try:
        import torch
    except ImportError:
        return "cpu"

    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        return "0"
    return "cpu"


# make_tracking_dataframe 처리용 보조 함수.
def make_tracking_dataframe(detection_records: Iterable[Mapping[str, Any]]) -> pd.DataFrame:

    data_frame = pd.DataFrame(detection_records, columns=CSV_COLUMNS)
    if data_frame.empty:
        return data_frame

    return data_frame.sort_values(["frame", "class", "track_id"]).reset_index(drop=True)


# make_output_dataframe 처리용 보조 함수.
def make_output_dataframe(data_frame: pd.DataFrame) -> pd.DataFrame:

    result_data = data_frame.copy()
    for column in CSV_COLUMNS:
        if column not in result_data.columns:
            result_data[column] = None

    result_data = fill_missing_bbox_from_center(result_data)
    result_data = result_data[CSV_COLUMNS]

    if not result_data.empty:
        result_data = result_data.sort_values(["frame", "class", "track_id"]).reset_index(drop=True)
    return result_data


# bbox나 중심 좌표를 계산할 때 쓰는 함수.
# bbox 좌표에서 중심, 크기, 거리 값을 계산해서 다음 판단에 넘김.
def fill_missing_bbox_from_center(data_frame: pd.DataFrame) -> pd.DataFrame:

    result_data = data_frame.copy()
    default_box_size = {
        "ball": 18.0,
        "player": 70.0,
        "referee": 70.0,
        "hoop": 80.0,
    }

    for index, data_row in result_data.iterrows():
        has_bbox = (
            pd.notna(data_row.get("x1"))
            and pd.notna(data_row.get("y1"))
            and pd.notna(data_row.get("x2"))
            and pd.notna(data_row.get("y2"))
        )
        if has_bbox:
            continue

        x_center = data_row.get("x_center")
        y_center = data_row.get("y_center")
        if pd.isna(x_center) or pd.isna(y_center):
            continue

        size = default_box_size.get(str(data_row.get("class")), 40.0)
        half = size / 2.0
        result_data.at[index, "x1"] = round(float(x_center) - half, 2)
        result_data.at[index, "y1"] = round(float(y_center) - half, 2)
        result_data.at[index, "x2"] = round(float(x_center) + half, 2)
        result_data.at[index, "y2"] = round(float(y_center) + half, 2)

    return result_data


# 화면 밖으로 나간 공 좌표를 제거.
def remove_invalid_ball_positions(data_frame: pd.DataFrame, frame_size: Tuple[int, int]) -> pd.DataFrame:

    if data_frame.empty:
        return data_frame

    width, height = frame_size
    result_data = data_frame.copy()
    is_ball = result_data["class"].eq("ball")
    x = pd.to_numeric(result_data["x_center"], errors="coerce")
    y = pd.to_numeric(result_data["y_center"], errors="coerce")
    inside_frame = x.between(0, width) & y.between(0, height)

    result_data = result_data[~is_ball | inside_frame].reset_index(drop=True)
    return result_data


# 림 근처에 고정된 공 오탐을 한 번 더 제거.
# 림 주변에 붙어 있는 공 오탐을 줄임.
def remove_rim_false_ball_positions(
    data_frame: pd.DataFrame,
    frame_size: Tuple[int, int],
    fps: float,
) -> pd.DataFrame:

    if data_frame.empty or "class" not in data_frame.columns:
        return data_frame

    result_data = data_frame.copy()
    numeric_columns = ["frame", "confidence", "x_center", "y_center", "x1", "y1", "x2", "y2"]
    for column in numeric_columns:
        if column in result_data.columns:
            result_data[column] = pd.to_numeric(result_data[column], errors="coerce")

    ball_mask = result_data["class"].eq("ball")
    hoop_mask = result_data["class"].eq("hoop")
    if not bool(ball_mask.any()):
        return data_frame

    width, height = frame_size
    scale = max(0.85, min(1.45, max(width, height) / 1280.0))
    hoop_by_frame: Dict[int, List[Dict[str, Any]]] = {}
    for data_row in result_data[hoop_mask].dropna(subset=["frame"]).to_dict("records"):
        try:
            frame = int(data_row["frame"])
        except (TypeError, ValueError):
            continue
        hoop_by_frame.setdefault(frame, []).append(data_row)
    player_by_frame: Dict[int, List[Dict[str, Any]]] = {}
    player_mask = result_data["class"].eq("player")
    for data_row in result_data[player_mask].dropna(subset=["frame"]).to_dict("records"):
        try:
            frame = int(data_row["frame"])
        except (TypeError, ValueError):
            continue
        player_by_frame.setdefault(frame, []).append(data_row)

    remove_indices: Set[int] = set()
    near_rim_rows: List[Tuple[int, int, float, float, float]] = []
    near_edge_rows: List[Tuple[int, int, float, float, float]] = []
    # 프레임을 돌면서 탐지 결과와 누적값을 계속 갱신함.
    for index, data_row in result_data[ball_mask].dropna(subset=["frame"]).iterrows():
        detection_record = data_row.to_dict()
        try:
            frame = int(detection_record["frame"])
        except (TypeError, ValueError):
            continue
        if edge_static_ball_candidate(detection_record, frame_size, scale):
            near_edge_rows.append(
                (
                    int(index),
                    frame,
                    float(detection_record["x_center"]),
                    float(detection_record["y_center"]),
                    float(detection_record.get("confidence", 0.0) or 0.0),
                )
            )
        hoop = nearest_hoop_record(frame, detection_record, hoop_by_frame, search_window=3)
        if hoop is None:
            continue

        rim_score = rim_false_ball_score(detection_record, hoop, scale)
        if rim_score["near_rim"]:
            near_rim_rows.append(
                (
                    int(index),
                    frame,
                    float(detection_record["x_center"]),
                    float(detection_record["y_center"]),
                    float(rim_score["distance_to_hoop_center"]),
                )
            )
        if rim_score["remove_now"]:
            remove_indices.add(int(index))

    remove_indices.update(
        non_shot_rim_ball_indices(
            result_data=result_data,
            near_rim_rows=near_rim_rows,
            hoop_by_frame=hoop_by_frame,
            frame_size=frame_size,
            scale=scale,
            fps=fps,
        )
    )
    remove_indices.update(static_edge_false_ball_indices(near_edge_rows, scale, fps))
    remove_indices.update(
        static_contextless_ball_indices(
            result_data=result_data,
            player_by_frame=player_by_frame,
            frame_size=frame_size,
            scale=scale,
            fps=fps,
        )
    )
    if not remove_indices:
        return data_frame

    return result_data.drop(index=sorted(remove_indices)).reset_index(drop=True)


# nearest_hoop_record는 두 위치가 얼마나 가까운지 판단할 때 사용.
def nearest_hoop_record(
    frame: int,
    ball_record: Mapping[str, Any],
    hoop_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    search_window: int,
) -> Optional[Mapping[str, Any]]:

    best_hoop: Optional[Mapping[str, Any]] = None
    best_score = float("inf")
    ball_x = _as_finite_float(ball_record.get("x_center"))
    ball_y = _as_finite_float(ball_record.get("y_center"))
    if ball_x is None or ball_y is None:
        return None

    for current_frame in range(frame - search_window, frame + search_window + 1):
        for hoop in hoop_by_frame.get(current_frame, []):
            if not _record_has_bbox(hoop):
                continue
            hoop_x = float(hoop["x_center"])
            hoop_y = float(hoop["y_center"])
            frame_penalty = abs(current_frame - frame) * 18.0
            score = math.hypot(ball_x - hoop_x, ball_y - hoop_y) + frame_penalty
            if score < best_score:
                best_score = score
                best_hoop = hoop
    return best_hoop


# rim_false_ball_score는 공 좌표나 공 후보를 따로 정리할 때 사용.
def rim_false_ball_score(
    ball_record: Mapping[str, Any],
    hoop_record: Mapping[str, Any],
    scale: float,
) -> Dict[str, Any]:

    confidence = float(ball_record.get("confidence", 0.0) or 0.0)
    ball_x = float(ball_record["x_center"])
    ball_y = float(ball_record["y_center"])
    hoop_width = float(hoop_record["x2"]) - float(hoop_record["x1"])
    hoop_height = float(hoop_record["y2"]) - float(hoop_record["y1"])
    hoop_size = max(hoop_width, hoop_height, 1.0)
    distance = math.hypot(
        ball_x - float(hoop_record["x_center"]),
        ball_y - float(hoop_record["y_center"]),
    )
    inside_ratio = _candidate_intersection_ratio(ball_record, hoop_record)
    center_inside = _point_inside_expanded_bbox(ball_x, ball_y, hoop_record, 5.0 * scale)
    near_center = distance <= max(22.0 * scale, hoop_size * 0.34)
    ball_size = _ball_candidate_size(ball_record)
    small_enough = ball_size <= max(52.0 * scale, hoop_size * 0.62)
    above_arc_zone = ball_is_above_hoop_arc_zone(ball_y, hoop_record, scale)

    remove_now = (
        not above_arc_zone
        and
        confidence < 0.16
        and small_enough
        and ((center_inside and inside_ratio >= 0.65) or (near_center and inside_ratio >= 0.50))
    )
    return {
        "near_rim": bool(
            not above_arc_zone
            and small_enough
            and (center_inside or near_center)
            and inside_ratio >= 0.25
        ),
        "remove_now": bool(remove_now),
        "distance_to_hoop_center": distance,
    }


# static_rim_false_ball_indices는 공 좌표나 공 후보를 따로 정리할 때 사용.
def static_rim_false_ball_indices(
    near_rim_rows: Sequence[Tuple[int, int, float, float, float]],
    scale: float,
    fps: float,
) -> Set[int]:

    if not near_rim_rows:
        return set()

    max_gap = max(2, int(round((float(fps) if fps and fps > 0 else 30.0) * 0.18)))
    min_rows = 3
    max_spread = 14.0 * scale
    remove_indices: Set[int] = set()
    current: List[Tuple[int, int, float, float, float]] = []

    for item in sorted(near_rim_rows, key=lambda data_row: data_row[1]):
        if current and item[1] - current[-1][1] > max_gap:
            remove_indices.update(static_rim_run_indices(current, min_rows, max_spread))
            current = []
        current.append(item)

    remove_indices.update(static_rim_run_indices(current, min_rows, max_spread))
    return remove_indices


# 슛으로 볼 만한 구간이나 성공 여부를 판단하는 부분.
# 슛 후보 구간이나 성공/실패 판단에 쓰는 함수.
def non_shot_rim_ball_indices(
    result_data: pd.DataFrame,
    near_rim_rows: Sequence[Tuple[int, int, float, float, float]],
    hoop_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
    scale: float,
    fps: float,
) -> Set[int]:

    if not near_rim_rows:
        return set()

    ball_table = result_data[result_data["class"].eq("ball")].copy()
    if ball_table.empty:
        return set()

    max_gap = max(3, int(round((float(fps) if fps and fps > 0 else 30.0) * 0.45)))
    remove_indices: Set[int] = set()
    for cluster in cluster_indexed_frame_rows(near_rim_rows, max_gap=max_gap):
        if shot_like_rim_cluster(
            cluster=cluster,
            ball_table=ball_table,
            hoop_by_frame=hoop_by_frame,
            frame_size=frame_size,
            scale=scale,
            fps=fps,
        ):
            continue

        static_false = rim_cluster_is_static_enough(cluster, scale, fps)
        sudden_false = sudden_rim_jump_without_high_arc(
            cluster=cluster,
            ball_table=ball_table,
            hoop_by_frame=hoop_by_frame,
            scale=scale,
            fps=fps,
        )
        if not static_false and not sudden_false:
            continue

        cluster_indices = {data_row[0] for data_row in cluster}
        if sudden_false and not static_false:
            remove_indices.update(cluster_indices)
            continue

        segment_indices = matching_ball_segment_indices(result_data, cluster_indices)
        remove_indices.update(segment_indices or cluster_indices)

    return remove_indices


# rim_cluster_is_static_enough는 림/골대 위치를 안정적으로 잡기 위한 보조 처리.
def rim_cluster_is_static_enough(
    cluster: Sequence[Tuple[int, int, float, float, float]],
    scale: float,
    fps: float,
) -> bool:

    if len(cluster) < 3:
        return False

    fps_value = float(fps) if fps and fps > 0 else 30.0
    xs = [data_row[2] for data_row in cluster]
    ys = [data_row[3] for data_row in cluster]
    spread = math.hypot(max(xs) - min(xs), max(ys) - min(ys))
    duration_frames = max(data_row[1] for data_row in cluster) - min(data_row[1] for data_row in cluster) + 1
    short_static_limit = 20.0 * scale
    long_static_limit = 38.0 * scale
    if duration_frames >= max(4, int(round(fps_value * 0.16))) and spread <= short_static_limit:
        return True
    if duration_frames >= max(8, int(round(fps_value * 0.42))) and spread <= long_static_limit:
        return True
    return False


# cluster_indexed_frame_rows 처리용 보조 함수.
# data_rows, max_gap 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def cluster_indexed_frame_rows(
    data_rows: Sequence[Tuple[int, int, float, float, float]],
    max_gap: int,
) -> List[List[Tuple[int, int, float, float, float]]]:
    clusters: List[List[Tuple[int, int, float, float, float]]] = []
    current: List[Tuple[int, int, float, float, float]] = []
    for data_row in sorted(data_rows, key=lambda item: item[1]):
        if current and data_row[1] - current[-1][1] > max_gap:
            clusters.append(current)
            current = []
        current.append(data_row)
    if current:
        clusters.append(current)
    return clusters


# 먼저 후보 프레임 구간을 만들고, 조건을 통과한 경우에만 이벤트 row로 추가함.
def shot_like_rim_cluster(
    cluster: Sequence[Tuple[int, int, float, float, float]],
    ball_table: pd.DataFrame,
    hoop_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
    scale: float,
    fps: float,
) -> bool:

    if not cluster:
        return False

    fps_value = float(fps) if fps and fps > 0 else 30.0
    start_frame = min(data_row[1] for data_row in cluster)
    end_frame = max(data_row[1] for data_row in cluster)
    min_rim_distance = min(data_row[4] for data_row in cluster)
    lookback = max(18, int(round(fps_value * 2.2)))

    prior_df = ball_table[
        (ball_table["frame"] < start_frame)
        & (ball_table["frame"] >= start_frame - lookback)
    ].sort_values("frame")
    if len(prior_df) < 3:
        return False

    prior_distances: List[Tuple[int, float, float, float]] = []
    for _, prior_row in prior_df.iterrows():
        detection_record = prior_row.to_dict()
        try:
            frame = int(detection_record["frame"])
        except (TypeError, ValueError):
            continue
        hoop = nearest_hoop_record(frame, detection_record, hoop_by_frame, search_window=5)
        if hoop is None:
            continue
        distance = math.hypot(
            float(detection_record["x_center"]) - float(hoop["x_center"]),
            float(detection_record["y_center"]) - float(hoop["y_center"]),
        )
        prior_distances.append(
            (
                frame,
                distance,
                float(detection_record["x_center"]),
                float(detection_record["y_center"]),
            )
        )

    if len(prior_distances) < 3:
        return False

    if high_arc_shot_approach(prior_distances, cluster, scale, fps):
        return True

    far_prior = [item for item in prior_distances if item[1] >= min_rim_distance + 55.0 * scale]
    if not far_prior:
        return False

    first_far = far_prior[0]
    last_prior = prior_distances[-1]
    approach_drop = first_far[1] - min_rim_distance
    last_drop = last_prior[1] - min_rim_distance
    frame_gap = max(1, start_frame - first_far[0])
    approach_speed = approach_drop / float(frame_gap)
    path_motion = math.hypot(cluster[0][2] - first_far[2], cluster[0][3] - first_far[3])
    cluster_duration = max(1, end_frame - start_frame + 1)


    if approach_drop < 65.0 * scale:
        return False
    if last_drop < 24.0 * scale:
        return False
    if approach_speed < 1.35 * scale and path_motion < 85.0 * scale:
        return False
    if cluster_duration > int(round(fps_value * 1.4)) and approach_speed < 2.0 * scale:
        return False
    return True


# sudden_rim_jump_without_high_arc는 림/골대 위치를 안정적으로 잡기 위한 보조 처리.
def sudden_rim_jump_without_high_arc(
    cluster: Sequence[Tuple[int, int, float, float, float]],
    ball_table: pd.DataFrame,
    hoop_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    scale: float,
    fps: float,
) -> bool:

    if not cluster or ball_table.empty:
        return False

    fps_value = float(fps) if fps and fps > 0 else 30.0
    start_frame = min(data_row[1] for data_row in cluster)
    lookback = max(8, int(round(fps_value * 0.85)))
    prior_df = ball_table[
        (ball_table["frame"] < start_frame)
        & (ball_table["frame"] >= start_frame - lookback)
    ].sort_values("frame")
    if prior_df.empty:
        return False

    prior_distances = rim_prior_distances(prior_df, hoop_by_frame)
    if len(prior_distances) >= 3 and high_arc_shot_approach(prior_distances, cluster, scale, fps):
        return False

    last_prior = prior_df.iloc[-1].to_dict()
    last_frame = _as_finite_float(last_prior.get("frame"))
    last_x = _as_finite_float(last_prior.get("x_center"))
    last_y = _as_finite_float(last_prior.get("y_center"))
    if last_frame is None or last_x is None or last_y is None:
        return False

    frame_gap = max(1, start_frame - int(last_frame))
    if frame_gap > max(8, int(round(fps_value * 0.28))):
        return False

    cluster_x = sum(data_row[2] for data_row in cluster) / float(len(cluster))
    cluster_y = sum(data_row[3] for data_row in cluster) / float(len(cluster))
    jump_distance = math.hypot(cluster_x - last_x, cluster_y - last_y)
    if jump_distance < max(95.0 * scale, 34.0 * scale * frame_gap):
        return False

    if cluster_y - last_y >= 25.0 * scale:
        return False

    min_cluster_rim_distance = min(data_row[4] for data_row in cluster)
    if prior_distances:
        last_rim_distance = prior_distances[-1][1]
        if last_rim_distance <= min_cluster_rim_distance + 55.0 * scale:
            return False

    return True


# rim_prior_distances는 두 위치가 얼마나 가까운지 판단할 때 사용.
def rim_prior_distances(
    prior_df: pd.DataFrame,
    hoop_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
) -> List[Tuple[int, float, float, float]]:
    distances: List[Tuple[int, float, float, float]] = []
    for _, prior_row in prior_df.iterrows():
        detection_record = prior_row.to_dict()
        try:
            frame = int(detection_record["frame"])
        except (TypeError, ValueError):
            continue
        hoop = nearest_hoop_record(frame, detection_record, hoop_by_frame, search_window=5)
        if hoop is None:
            continue
        distance = math.hypot(
            float(detection_record["x_center"]) - float(hoop["x_center"]),
            float(detection_record["y_center"]) - float(hoop["y_center"]),
        )
        distances.append(
            (
                frame,
                distance,
                float(detection_record["x_center"]),
                float(detection_record["y_center"]),
            )
        )
    return distances


# high_arc_shot_approach에서는 슛으로 볼 만한 프레임 구간을 판단.
def high_arc_shot_approach(
    prior_distances: Sequence[Tuple[int, float, float, float]],
    cluster: Sequence[Tuple[int, int, float, float, float]],
    scale: float,
    fps: float,
) -> bool:

    if len(prior_distances) < 3 or not cluster:
        return False

    fps_value = float(fps) if fps and fps > 0 else 30.0
    start_frame = min(data_row[1] for data_row in cluster)
    cluster_x = sum(data_row[2] for data_row in cluster) / float(len(cluster))
    cluster_y = sum(data_row[3] for data_row in cluster) / float(len(cluster))

    high_prior = min(prior_distances, key=lambda item: item[3])
    high_frame, _, high_x, high_y = high_prior
    frame_gap = max(1, start_frame - int(high_frame))
    if frame_gap > int(round(fps_value * 1.6)):
        return False

    vertical_drop = cluster_y - high_y
    path_motion = math.hypot(cluster_x - high_x, cluster_y - high_y)
    vertical_speed = vertical_drop / float(frame_gap)

    if vertical_drop < 30.0 * scale:
        return False
    if path_motion < 45.0 * scale and vertical_speed < 0.55 * scale:
        return False
    return True


# matching_ball_segment_indices는 공 좌표나 공 후보를 따로 정리할 때 사용.
def matching_ball_segment_indices(
    result_data: pd.DataFrame,
    cluster_indices: Set[int],
) -> Set[int]:

    if "ball_segment" not in result_data.columns or not cluster_indices:
        return set()

    segments: Set[int] = set()
    for index in cluster_indices:
        if index not in result_data.index:
            continue
        value = result_data.at[index, "ball_segment"]
        number = _as_finite_float(value)
        if number is not None:
            segments.add(int(number))
    if not segments:
        return set()

    segment_values = pd.to_numeric(result_data["ball_segment"], errors="coerce")
    mask = result_data["class"].eq("ball") & segment_values.isin(segments)
    return {int(index) for index in result_data.index[mask]}


# static_rim_run_indices는 림/골대 위치를 안정적으로 잡기 위한 보조 처리.
def static_rim_run_indices(
    data_rows: Sequence[Tuple[int, int, float, float, float]],
    min_rows: int,
    max_spread: float,
) -> Set[int]:
    if len(data_rows) < min_rows:
        return set()

    xs = [data_row[2] for data_row in data_rows]
    ys = [data_row[3] for data_row in data_rows]
    spread = math.hypot(max(xs) - min(xs), max(ys) - min(ys))
    median_distance = sorted(data_row[4] for data_row in data_rows)[len(data_rows) // 2]
    if spread <= max_spread and median_distance <= max_spread * 1.8:
        return {data_row[0] for data_row in data_rows}
    return set()


# edge_static_ball_candidate에서는 공 후보를 비교해서 남길 것만 고름.
def edge_static_ball_candidate(
    ball_record: Mapping[str, Any],
    frame_size: Tuple[int, int],
    scale: float,
) -> bool:

    width, height = frame_size
    ball_x = _as_finite_float(ball_record.get("x_center"))
    ball_y = _as_finite_float(ball_record.get("y_center"))
    if ball_x is None or ball_y is None:
        return False
    margin = max(36.0 * scale, _ball_candidate_size(ball_record) * 1.20)
    return (
        ball_x <= margin
        or ball_x >= float(width) - margin
        or ball_y <= margin
        or ball_y >= float(height) - margin
    )


# static_edge_false_ball_indices는 공 좌표나 공 후보를 따로 정리할 때 사용.
def static_edge_false_ball_indices(
    near_edge_rows: Sequence[Tuple[int, int, float, float, float]],
    scale: float,
    fps: float,
) -> Set[int]:

    if not near_edge_rows:
        return set()

    max_gap = max(2, int(round((float(fps) if fps and fps > 0 else 30.0) * 0.20)))
    min_rows = 4
    max_spread = 18.0 * scale
    remove_indices: Set[int] = set()
    current: List[Tuple[int, int, float, float, float]] = []

    for item in sorted(near_edge_rows, key=lambda data_row: data_row[1]):
        if current and item[1] - current[-1][1] > max_gap:
            remove_indices.update(static_edge_run_indices(current, min_rows, max_spread))
            current = []
        current.append(item)

    remove_indices.update(static_edge_run_indices(current, min_rows, max_spread))
    return remove_indices


# static_edge_run_indices 처리용 보조 함수.
# data_rows, min_rows, max_spread 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def static_edge_run_indices(
    data_rows: Sequence[Tuple[int, int, float, float, float]],
    min_rows: int,
    max_spread: float,
) -> Set[int]:
    if len(data_rows) < min_rows:
        return set()
    xs = [data_row[2] for data_row in data_rows]
    ys = [data_row[3] for data_row in data_rows]
    confidences = [data_row[4] for data_row in data_rows if math.isfinite(data_row[4])]
    x_spread = max(xs) - min(xs)
    spread = math.hypot(x_spread, max(ys) - min(ys))
    if not confidences:
        return set()
    avg_confidence = sum(confidences) / len(confidences)
    median_confidence = sorted(confidences)[len(confidences) // 2]
    if spread <= max_spread and (avg_confidence <= 0.24 or median_confidence <= 0.08):
        return {data_row[0] for data_row in data_rows}
    if len(data_rows) >= 6 and x_spread <= max_spread and (avg_confidence <= 0.20 or median_confidence <= 0.06):
        return {data_row[0] for data_row in data_rows}
    y_spread = max(ys) - min(ys)
    if len(data_rows) >= 5 and y_spread <= max_spread * 1.65 and median_confidence <= 0.10:
        return {data_row[0] for data_row in data_rows}
    return set()


# static_contextless_ball_indices는 공 좌표나 공 후보를 따로 정리할 때 사용.
def static_contextless_ball_indices(
    result_data: pd.DataFrame,
    player_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
    scale: float,
    fps: float,
) -> Set[int]:

    ball_table = result_data[result_data["class"].eq("ball")].copy()
    if ball_table.empty:
        return set()

    fps_value = float(fps) if fps and fps > 0 else 30.0
    max_gap = max(2, int(round(fps_value * 0.20)))
    min_rows = max(7, int(round(fps_value * 0.28)))
    max_spread = 24.0 * scale

    data_rows: List[Tuple[int, int, float, float, float]] = []
    for index, data_row in ball_table.dropna(subset=["frame", "x_center", "y_center"]).iterrows():
        data_rows.append(
            (
                int(index),
                int(data_row["frame"]),
                float(data_row["x_center"]),
                float(data_row["y_center"]),
                float(data_row.get("confidence", 0.0) or 0.0),
            )
        )

    remove_indices: Set[int] = set()
    current: List[Tuple[int, int, float, float, float]] = []
    for data_row in sorted(data_rows, key=lambda item: item[1]):
        if current and data_row[1] - current[-1][1] > max_gap:
            remove_indices.update(
                static_contextless_run_indices(
                    result_data,
                    current,
                    player_by_frame,
                    frame_size,
                    min_rows,
                    max_spread,
                    scale,
                )
            )
            current = []
        current.append(data_row)

    remove_indices.update(
        static_contextless_run_indices(
            result_data,
            current,
            player_by_frame,
            frame_size,
            min_rows,
            max_spread,
            scale,
        )
    )
    return remove_indices


# static_contextless_run_indices 처리용 보조 함수.
# result_data, data_rows, player_by_frame 값을 받아 필요한 형태로 바꾸고 다음 단계에 넘김.
def static_contextless_run_indices(
    result_data: pd.DataFrame,
    data_rows: Sequence[Tuple[int, int, float, float, float]],
    player_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
    min_rows: int,
    max_spread: float,
    scale: float,
) -> Set[int]:
    if len(data_rows) < min_rows:
        return set()

    xs = [data_row[2] for data_row in data_rows]
    ys = [data_row[3] for data_row in data_rows]
    x_spread = max(xs) - min(xs)
    y_spread = max(ys) - min(ys)
    spread = math.hypot(x_spread, y_spread)
    if spread > max_spread:
        return set()

    near_player_count = 0
    for _, frame, x, y, _ in data_rows:
        if ball_point_near_player((x, y), frame, player_by_frame, frame_size, scale):
            near_player_count += 1
    near_player_ratio = near_player_count / float(len(data_rows))
    if near_player_ratio >= 0.30:
        return set()

    cluster_indices = {data_row[0] for data_row in data_rows}
    segment_indices = matching_ball_segment_indices(result_data, cluster_indices)
    return segment_indices or cluster_indices


# ball_point_near_player는 두 위치가 얼마나 가까운지 판단할 때 사용.
def ball_point_near_player(
    point: Tuple[float, float],
    frame: int,
    player_by_frame: Mapping[int, Sequence[Mapping[str, Any]]],
    frame_size: Tuple[int, int],
    scale: float,
) -> bool:
    del frame_size
    margin = 95.0 * scale
    for current_frame in range(frame - 2, frame + 3):
        for player in player_by_frame.get(current_frame, []):
            if not _record_has_bbox(player):
                continue
            if _point_inside_expanded_bbox(point[0], point[1], player, margin):
                return True
    return False


# 림 bbox가 잠깐 사라질 때 앞뒤 프레임으로 보간.
def stabilize_hoop_detections(
    data_frame: pd.DataFrame,
    total_frames: Optional[int],
    frame_size: Tuple[int, int],
    fps: float,
) -> pd.DataFrame:

    if data_frame.empty or "class" not in data_frame.columns:
        return data_frame

    hoop_table = data_frame[data_frame["class"].eq("hoop")].copy()
    if hoop_table.empty:
        return data_frame

    non_hoop_df = data_frame[~data_frame["class"].eq("hoop")].copy()
    selected = select_primary_hoop_observations(hoop_table, frame_size)
    if selected.empty:
        return non_hoop_df.reset_index(drop=True)

    scale = max(0.85, min(1.45, max(frame_size) / 1280.0))
    fps_value = float(fps) if fps and fps > 0 else 30.0
    max_gap = 28 if fps_value >= 45.0 else 18
    max_gap_distance = 130.0 * scale

    selected_rows = selected.sort_values("frame").to_dict("records")
    stabilized_rows: List[Dict[str, Any]] = []
    for index, left in enumerate(selected_rows):
        stabilized_rows.append(dict(left))
        if index >= len(selected_rows) - 1:
            continue

        right = selected_rows[index + 1]
        frame_gap = int(right["frame"]) - int(left["frame"])
        if frame_gap <= 1 or frame_gap > max_gap:
            continue

        distance = math.hypot(
            float(right["x_center"]) - float(left["x_center"]),
            float(right["y_center"]) - float(left["y_center"]),
        )
        if distance > max(max_gap_distance, 9.0 * scale * frame_gap):
            continue

        for frame in range(int(left["frame"]) + 1, int(right["frame"])):
            ratio = (frame - int(left["frame"])) / float(frame_gap)
            stabilized_rows.append(make_interpolated_hoop_row(left, right, frame, ratio))

    stabilized_hoop_df = pd.DataFrame(stabilized_rows)
    if total_frames is not None and not stabilized_hoop_df.empty:
        frames = pd.to_numeric(stabilized_hoop_df["frame"], errors="coerce")
        stabilized_hoop_df = stabilized_hoop_df[frames.between(0, int(total_frames) - 1)]

    result_data = pd.concat([non_hoop_df, stabilized_hoop_df], ignore_index=True, sort=False)
    return result_data.sort_values(["frame", "class", "track_id"]).reset_index(drop=True)


# select_primary_hoop_observations는 림/골대 위치를 안정적으로 잡기 위한 보조 처리.
def select_primary_hoop_observations(
    hoop_table: pd.DataFrame,
    frame_size: Tuple[int, int],
) -> pd.DataFrame:

    if hoop_table.empty:
        return hoop_table

    numeric_columns = ["frame", "confidence", "x_center", "y_center", "x1", "y1", "x2", "y2"]
    result_data = hoop_table.copy()
    for column in numeric_columns:
        if column in result_data.columns:
            result_data[column] = pd.to_numeric(result_data[column], errors="coerce")
    result_data = result_data.dropna(subset=["frame", "x_center", "y_center", "x1", "y1", "x2", "y2"])
    if result_data.empty:
        return result_data

    selected_rows: List[Dict[str, Any]] = []
    for _, frame_df in result_data.groupby("frame", sort=True):
        candidates = filter_hoop_candidates(frame_df.to_dict("records"), frame_size)
        if candidates:
            data_row = dict(candidates[0])
            data_row["track_id"] = 0
            data_row["hoop_status"] = data_row.get("hoop_status", "Detected")
            selected_rows.append(data_row)

    if not selected_rows:
        return result_data.iloc[0:0].copy()
    return pd.DataFrame(selected_rows).sort_values("frame").reset_index(drop=True)


# make_interpolated_hoop_row는 림/골대 위치를 안정적으로 잡기 위한 보조 처리.
def make_interpolated_hoop_row(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    frame: int,
    ratio: float,
) -> Dict[str, Any]:

    data_row = dict(left)
    data_row["frame"] = int(frame)
    data_row["class"] = "hoop"
    try:
        left_track_id = int(float(left.get("track_id", -1)))
    except (TypeError, ValueError):
        left_track_id = 0
    data_row["track_id"] = left_track_id if left_track_id >= 0 else 0
    for column in ["x_center", "y_center", "x1", "y1", "x2", "y2"]:
        data_row[column] = round(
            float(left[column]) + (float(right[column]) - float(left[column])) * float(ratio),
            2,
        )
    data_row["confidence"] = round(
        max(0.03, min(float(left.get("confidence", 0.0) or 0.0), float(right.get("confidence", 0.0) or 0.0)) * 0.85),
        3,
    )
    data_row["hoop_status"] = "Interpolated"
    return data_row


# team_id 표기를 뒤쪽 집계에서 비교하기 쉽게 맞춤.
def normalize_team_id(value: Any) -> str:

    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "unknown"}:
        return ""
    if text.endswith(".0"):
        text = text[:-2]
    return text


# _validate_input_file는 위쪽에서 만든 값을 다음 단계에 넘기기 쉽게 정리.
def _validate_input_file(path: str, label: str) -> None:

    if not Path(path).exists():
        raise FileNotFoundError(f"{label} file not found: {path}")


__all__ = ["run_yolo_tracking_pipeline"]


## Cell 1. 라이브러리

CSV 읽고 표 확인할 때 필요한 것들만 먼저 import.


In [7]:
# 아래 셀에서 쓰는 기본 도구만 먼저 가져옴.
# 파일 경로는 Path, CSV 확인은 pandas, 숫자 계산은 numpy로 처리.

from pathlib import Path

import numpy as np

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("노트북 실행 준비 완료")

노트북 실행 준비 완료


## Cell 2. 경로 설정

영상 파일, 모델 폴더, CSV 저장 위치는 여기서 바꾸면 됨.


In [8]:
# 입력 영상, 모델 경로, 저장될 CSV 경로를 먼저 고정.
# start_sec는 시작 시간이고, run_sec가 0이면 전체 영상을 끝까지 돌림.
# 모델 파일이 없으면 오래 돌기 전에 여기서 바로 멈추게 함.

video_path = Path("1_.mp4")

model_dir = Path("models")

save_dir = Path("runs/verify")
save_dir.mkdir(parents=True, exist_ok=True)

start_sec = 0

run_sec = 0

# 객체 탐지 결과가 저장될 CSV.
track_csv = save_dir / "improved_tracking_results.csv"

# 실시간 이벤트와 30초 누적 스탯이 함께 저장될 CSV.
event_csv = save_dir / "event.csv"

# 전체/팀별 이벤트 요약 결과가 저장될 CSV.
summary_csv = save_dir / "event_summary.csv"


# 필요한 파일이 있는지 먼저 확인.
def check_file(file_path, file_label):
    if not file_path.exists():
        raise FileNotFoundError(f"{file_label} 파일을 찾을 수 없습니다: {file_path}")


# 모델 파일명을 받아 실제 경로로 바꿈.
def find_model(model_name):
    model_path = model_dir / model_name
    check_file(model_path, model_name)
    return model_path


check_file(video_path, "입력 영상")

player_model = find_model("player_detector.pt")
ball_model = find_model("best.pt")
ball_model_2 = find_model("ball_detector_model.pt")
court_model = find_model("court_keypoint_detector.pt")

print("입력 영상:", video_path)
print("모델 폴더:", model_dir)
print("결과 저장 폴더:", save_dir)
print("생성될 CSV: improved_tracking_results.csv, event.csv, event_summary.csv")

입력 영상: 1_.mp4
모델 폴더: models
결과 저장 폴더: runs\verify
생성될 CSV: improved_tracking_results.csv, event.csv, event_summary.csv


## Cell 3. 영상 돌리는 셀

제일 오래 걸리는 부분. 전체 영상이면 GPU를 오래 쓴다.


In [9]:
# 이전 CSV를 지운 뒤 같은 이름으로 새 결과를 저장.
# run_sec가 0이면 max_duration_seconds를 None으로 넘겨 전체 영상 처리.
# 실제 무거운 처리는 run_yolo_tracking_pipeline() 안에서 진행.

# 새로 돌릴 때 이전 CSV와 섞이지 않게 정리.
def clear_old_outputs():
    old_files = [
        track_csv,
        event_csv,
        summary_csv,
    ]

    for file_path in old_files:
        if file_path.exists():
            file_path.unlink()


# 위에서 맞춘 경로와 설정으로 실제 분석 실행.
def run_analysis():
    clear_old_outputs()

    # 분석 길이가 0이면 전체 영상을 분석함.
    if run_sec == 0:
        limit_sec = None
    else:
        limit_sec = float(run_sec)

    paths = run_yolo_tracking_pipeline(
        video_path=str(video_path),
        model_path=str(player_model),
        ball_model_path=str(ball_model),
        secondary_ball_model_path=str(ball_model_2),
        court_model_path=str(court_model),
        csv_output_path=str(track_csv),
        event_csv_output_path=str(event_csv),
        event_summary_output_path=str(summary_csv),
        start_time_seconds=float(start_sec),
        max_duration_seconds=limit_sec,
    )

    return paths


paths = run_analysis()

print("영상 기반 객체 탐지 및 이벤트 스탯 생성 완료")
print(paths)

YOLO processing frame: 100
YOLO processing frame: 200
YOLO processing frame: 300
YOLO processing frame: 400
YOLO processing frame: 500
YOLO processing frame: 600
YOLO processing frame: 700
YOLO processing frame: 800
YOLO processing frame: 900
YOLO processing frame: 1000
YOLO processing frame: 1100
YOLO processing frame: 1200
YOLO processing frame: 1300
YOLO processing frame: 1400
YOLO processing frame: 1500
YOLO processing frame: 1600
YOLO processing frame: 1700
YOLO processing frame: 1800
YOLO processing frame: 1900
YOLO processing frame: 2000
YOLO processing frame: 2100
YOLO processing frame: 2200
YOLO processing frame: 2300
YOLO processing frame: 2400
YOLO processing frame: 2500
YOLO processing frame: 2600
YOLO processing frame: 2700
YOLO processing frame: 2800
YOLO processing frame: 2900
YOLO processing frame: 3000
YOLO processing frame: 3100
YOLO processing frame: 3200
YOLO processing frame: 3300
YOLO processing frame: 3400
YOLO processing frame: 3500
YOLO processing frame: 3600
Y

## Cell 4. CSV 다시 읽기

분석 끝나고 나온 CSV를 읽어서 숫자 컬럼이랑 팀 ID 정도만 정리.


In [10]:
# 생성된 CSV 3개를 다시 읽고 컬럼 이름을 먼저 정리.
# event.csv에서 30초 누적 row와 실제 이벤트 row를 따로 분리.
# 숫자 계산에 쓰는 컬럼만 numeric으로 바꾼 뒤 표로 확인.
# 화면 표시용으로만 NaN을 빈칸처럼 보이게 처리.

# 저장된 CSV를 pandas로 읽어옴.
def read_csv(csv_path):
    # CSV를 읽어 온다.
    check_file(csv_path, "생성된 CSV")
    return pd.read_csv(csv_path, encoding="utf-8-sig")


# 컬럼 이름 앞뒤 공백 정리.
def fix_columns(dataframe):
    out = dataframe.copy()
    out.columns = [str(column).strip() for column in out.columns]
    return out


# team_id가 1.0처럼 읽힌 경우 보기 좋게 정리.
def clean_team_id(team_id):
    # 팀 ID를 비교하기 쉬운 문자열로 통일함.
    if pd.isna(team_id):
        return ""

    team_text = str(team_id).strip()

    if team_text.endswith(".0"):
        team_text = team_text[:-2]

    if team_text.lower() in {"nan", "none", "unknown"}:
        return ""

    return team_text


# 계산에 필요한 컬럼만 숫자형으로 바꿈.
def to_number_columns(dataframe, column_names):
    out = dataframe.copy()

    for column_name in column_names:
        if column_name in out.columns:
            out[column_name] = pd.to_numeric(out[column_name], errors="coerce")

    return out


# 화면 표시용으로 NaN만 빈칸처럼 보이게 함.
def show_table(dataframe, row_count=None):
    # 화면에 보여줄 때만 NaN을 빈칸으로 변환.
    view_df = dataframe.head(row_count).copy() if row_count is not None else dataframe.copy()
    display(view_df.fillna(""))


# 영상 분석으로 새로 만들어진 CSV를 읽어 온다.
tracks_df = read_csv(track_csv)
events_df = read_csv(event_csv)
summary_df = read_csv(summary_csv)

tracks_df = fix_columns(tracks_df)
events_df = fix_columns(events_df)
summary_df = fix_columns(summary_df)

# 이벤트 CSV에서 30초별 누적 스탯 행만 따로 분리함.
stats_30s = events_df[events_df["row_type"].eq("cumulative_30s_team")].copy()

event_rows = events_df[events_df["row_type"].eq("event")].copy()

track_num_cols = [
    "frame", "track_id", "confidence", "x_center", "y_center", "x1", "y1", "x2", "y2",
    "team_id", "team_confidence",
]

stat_num_cols = [
    "count", "frame", "time_sec", "shot_attempt_count", "shot_made_count", "shot_missed_count",
    "pass_count", "steal_or_block_count", "foul_count", "rebound_count", "shot_success_rate_pct",
    "possession_frame_count", "possession_time_sec", "possession_pct", "player_track_id",
    "secondary_track_id", "confidence",
]

tracks_df = to_number_columns(tracks_df, track_num_cols)
events_df = to_number_columns(events_df, stat_num_cols)
summary_df = to_number_columns(summary_df, stat_num_cols)
stats_30s = to_number_columns(stats_30s, stat_num_cols)
event_rows = to_number_columns(event_rows, stat_num_cols)

dfs = [tracks_df, events_df, summary_df, stats_30s, event_rows]

# 여러 항목을 돌면서 같은 기준을 적용하는 부분.
for data in dfs:
    if "team_id" in data.columns:
        data["team_id"] = data["team_id"].apply(clean_team_id)

need_cols = ["frame", "class", "confidence"]
need_cols_existing = [column for column in need_cols if column in tracks_df.columns]
tracks_df = tracks_df.dropna(subset=need_cols_existing)

print("객체 탐지 데이터 크기:", tracks_df.shape)
print("30초별 누적 스탯 데이터 크기:", stats_30s.shape)
print("실시간 이벤트 감지 데이터 크기:", event_rows.shape)
print("최종 이벤트 요약 데이터 크기:", summary_df.shape)

show_table(tracks_df, 5)
show_table(stats_30s, 5)
show_table(event_rows, 5)

객체 탐지 데이터 크기: (235224, 33)
30초별 누적 스탯 데이터 크기: (104, 24)
실시간 이벤트 감지 데이터 크기: (1642, 24)
최종 이벤트 요약 데이터 크기: (1681, 24)


,frame,track_id,class,confidence,x_center,y_center,x1,y1,x2,y2,ball_status,cleaning_note,raw_x_center,raw_y_center,ball_segment,is_reliable,reliability_reason,jump_distance,context_distance,player_body_overlap,player_body_track_id,player_body_region,player_body_rel_x,player_body_rel_y,player_body_torso,motion_distance,frame_gap_from_previous_ball,bbox_size,player_body_core,team_id,team_name,team_confidence,team_color_hex
0,0,3,player,0.458,1926.40,836.80,1673.60,398.40,2179.20,1275.20,,,,,,,,,,,,,,,,,,,,1,unknown,0.654,#8c6158
1,0,5,player,0.318,2211.20,806.40,2003.20,333.60,2419.20,1279.20,,,,,,,,,,,,,,,,,,,,1,unknown,0.884,#8c6158
2,0,7,player,0.303,1763.20,558.30,1686.40,322.20,1840.00,794.40,,,,,,,,,,,,,,,,,,,,1,unknown,0.762,#8c6158
3,1,3,player,0.376,1937.88,812.40,1668.86,343.85,2206.91,1280.94,,,,,,,,,,,,,,,,,,,,1,unknown,0.654,#8c6158
4,1,5,player,0.372,2205.46,807.12,1998.00,335.75,2412.91,1278.48,,,,,,,,,,,,,,,,,,,,1,unknown,0.884,#8c6158


,row_type,event_id,event_type,event_label_ko,count,frame,time_sec,team_id,team_name,shot_attempt_count,shot_made_count,shot_missed_count,pass_count,steal_or_block_count,foul_count,rebound_count,shot_success_rate_pct,possession_frame_count,possession_time_sec,possession_pct,player_track_id,secondary_track_id,confidence,details
1642,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,28.0,899,30.0,1,unknown,3.0,2.0,1.0,11.0,8.0,3.0,0.0,66.67,221.0,7.367,65.58,,,,cumulative_from_start_interval_30s_snapshot_1
1643,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,17.0,899,30.0,2,unknown,1.0,1.0,0.0,3.0,10.0,1.0,1.0,100.00,116.0,3.867,34.42,,,,cumulative_from_start_interval_30s_snapshot_1
1644,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,60.0,1799,60.0,1,unknown,5.0,4.0,1.0,27.0,17.0,5.0,1.0,80.00,517.0,17.233,66.45,,,,cumulative_from_start_interval_30s_snapshot_2
1645,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,35.0,1799,60.0,2,unknown,3.0,2.0,1.0,8.0,18.0,2.0,1.0,66.67,261.0,8.700,33.55,,,,cumulative_from_start_interval_30s_snapshot_2
1646,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,93.0,2699,90.0,1,unknown,7.0,5.0,2.0,42.0,27.0,8.0,2.0,71.43,740.0,24.667,66.19,,,,cumulative_from_start_interval_30s_snapshot_3


,row_type,event_id,event_type,event_label_ko,count,frame,time_sec,team_id,team_name,shot_attempt_count,shot_made_count,shot_missed_count,pass_count,steal_or_block_count,foul_count,rebound_count,shot_success_rate_pct,possession_frame_count,possession_time_sec,possession_pct,player_track_id,secondary_track_id,confidence,details
0,event,E00001,shot_attempt,슛시도,,129,4.300,1,unknown,,,,,,,,,,,,156.0,,0.760,near_rim_distance_px=48.4
1,event,E00002,shot_made,슛성공,,129,4.300,1,unknown,,,,,,,,,,,,156.0,,0.640,ball_crossed_rim_zone
2,event,E00003,shot_attempt,슛시도,,234,7.800,1,unknown,,,,,,,,,,,,156.0,,0.727,near_rim_distance_px=57.0
3,event,E00004,shot_made,슛성공,,234,7.800,1,unknown,,,,,,,,,,,,156.0,,0.607,ball_crossed_rim_zone
4,event,E00005,foul,파울,,280,9.333,1,unknown,,,,,,,,,,,,149.0,187.0,0.280,heuristic_contact_near_ball_during_opponent_po...


## Cell 5. 마지막 결과 표

팀별로 보기 편하게 묶어서 마지막 표만 확인.


In [11]:
# 선수 탐지 결과를 팀별로 묶고 이벤트 요약 CSV와 합침.
# 2점/3점 구분은 없어서 성공 슛당 2.3점으로 참고 점수를 계산.
# 마지막에는 점수 추정, 성공률, 점유율 기준으로 정렬해서 표를 봄.

players_df = tracks_df[tracks_df["class"].astype(str).str.lower().eq("player")].copy()

# 팀별로 탐지된 선수 박스 수, 선수 track 수, 평균 confidence를 계산.
team_track_summary = (
    players_df[players_df["team_id"].ne("")]
    .groupby("team_id", as_index=False)
    .agg(
        detected_player_boxes=("class", "count"),
        unique_player_tracks=("track_id", "nunique"),
        average_detection_confidence=("confidence", "mean"),
        average_team_confidence=("team_confidence", "mean"),
    )
)

# 최종 요약 CSV에서 팀별 요약 행만 가져온다.
team_stats = summary_df[summary_df["row_type"].eq("team_summary")].copy()

# 팀별 이벤트 스탯과 객체 탐지 요약을 합친다.
team_table = team_stats.merge(team_track_summary, on="team_id", how="left")

event_cols = [
    "shot_attempt_count", "shot_made_count", "shot_missed_count", "pass_count",
    "steal_or_block_count", "foul_count", "rebound_count", "shot_success_rate_pct",
    "possession_frame_count", "possession_time_sec", "possession_pct",
]

# 슛 시도 횟수 컬럼이 없으면 성공 슛과 실패 슛을 더해서 계산.
if "shot_attempt_count" not in team_table.columns:
    team_table["shot_attempt_count"] = team_table.get("shot_made_count", 0).fillna(0) + team_table.get("shot_missed_count", 0).fillna(0)

# 필요한 컬럼들을 한 번에 정리하는 반복문.
for column in event_cols:
    if column not in team_table.columns:
        team_table[column] = 0
    team_table[column] = team_table[column].fillna(0)

# 2점/3점 구분은 따로 하지 않아서, 성공 슛당 평균 2.3점으로 참고 점수를 계산.
team_table["estimated_points_from_made_shots"] = team_table["shot_made_count"] * 2.3

event_type_table = (
    event_rows.groupby("event_type", as_index=False)
    .agg(event_count=("event_type", "count"), average_confidence=("confidence", "mean"))
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

last_30s = (
    stats_30s.sort_values("time_sec")
    .groupby("team_id", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

final_cols = [
    "team_id", "team_name", "shot_attempt_count", "shot_made_count", "shot_missed_count",
    "estimated_points_from_made_shots", "shot_success_rate_pct", "pass_count",
    "steal_or_block_count", "foul_count", "rebound_count", "possession_time_sec",
    "possession_pct", "detected_player_boxes", "unique_player_tracks",
    "average_detection_confidence", "average_team_confidence",
]

final_cols = [column for column in final_cols if column in team_table.columns]
final_table = team_table[final_cols].copy()

final_table = final_table.sort_values(
    ["estimated_points_from_made_shots", "shot_success_rate_pct", "possession_pct"],
    ascending=False,
).reset_index(drop=True)

print("최종 팀별 이벤트 스탯 결과")
show_table(final_table)

print("이벤트 타입별 감지 횟수")
show_table(event_type_table)

print("팀별 마지막 30초 누적 스탯")
show_table(last_30s)

최종 팀별 이벤트 스탯 결과


,team_id,team_name,shot_attempt_count,shot_made_count,shot_missed_count,estimated_points_from_made_shots,shot_success_rate_pct,pass_count,steal_or_block_count,foul_count,rebound_count,possession_time_sec,possession_pct,detected_player_boxes,unique_player_tracks,average_detection_confidence,average_team_confidence
0,1,unknown,72.0,48.0,24.0,110.4,66.67,322.0,252.0,73.0,20.0,246.5,49.05,75841,2601,0.800478,0.765321
1,2,unknown,80.0,46.0,34.0,105.8,57.50,327.0,249.0,46.0,33.0,256.0,50.95,79733,2856,0.785362,0.766745


이벤트 타입별 감지 횟수


,event_type,event_count,average_confidence
0,pass,649,0.580000
1,steal_or_block,501,0.520000
2,shot_attempt,160,0.669831
3,foul,119,0.280000
4,shot_made,99,0.601455
5,shot_missed,61,0.466049
6,rebound,53,0.500000


팀별 마지막 30초 누적 스탯


,row_type,event_id,event_type,event_label_ko,count,frame,time_sec,team_id,team_name,shot_attempt_count,shot_made_count,shot_missed_count,pass_count,steal_or_block_count,foul_count,rebound_count,shot_success_rate_pct,possession_frame_count,possession_time_sec,possession_pct,player_track_id,secondary_track_id,confidence,details
0,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,811.0,46799,1560.0,1,unknown,72.0,48.0,24.0,322.0,252.0,73.0,20.0,66.67,7395.0,246.5,49.05,,,,cumulative_from_start_interval_30s_snapshot_52
1,cumulative_30s_team,,team_cumulative_totals,team_cumulative_totals,815.0,46799,1560.0,2,unknown,80.0,46.0,34.0,327.0,249.0,46.0,33.0,57.50,7680.0,256.0,50.95,,,,cumulative_from_start_interval_30s_snapshot_52
